<a href="https://colab.research.google.com/github/yanchenliu-cxk/ESM/blob/main/Boltz-High-confidence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================================================
# Cell 8A：安装 Boltz 环境
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi

# 安装 Boltz
!pip install -q -U "boltz[cuda]" biopython pandas openpyxl pyyaml tqdm

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

# 测试 boltz 命令
!boltz predict --help | head -n 40

Mounted at /content/drive
Thu May 21 11:39:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------

In [2]:
# =========================================================
# Cell B0：Boltz Notebook 初始化
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi || true

import sys
import subprocess
from pathlib import Path

print("Python:", sys.version)

# 检查 boltz 是否已经安装
!which boltz || true
!boltz predict --help | head -n 40

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Thu May 21 11:43:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             45W /  400W |       6MiB /  40960MiB |      0%      Default |
|          

In [3]:
# =========================================================
# Cell B1：检查前序结果文件
# =========================================================

from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

mapping_xlsx = BASE / "cofactor_mapping/EC_to_Cofactor_MultiEvidence_Mapping.xlsx"
manifest_xlsx = BASE / "cofactor_modeling_manifest/01_All500_Modeling_Manifest.xlsx"
fasta_path = BASE / "Top500_Candidate_Enzymes_Cleaned.fasta"

files = {
    "cofactor_mapping": mapping_xlsx,
    "modeling_manifest": manifest_xlsx,
    "fasta": fasta_path,
}

for name, path in files.items():
    print(name, "=>", path, "exists:", path.exists())

missing = [name for name, path in files.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"缺少文件: {missing}")

df_tasks = pd.read_excel(manifest_xlsx, sheet_name="all_modeling_tasks")

print("任务总数:", len(df_tasks))
print("\nModeling_Queue 统计:")
print(df_tasks["Modeling_Queue"].value_counts(dropna=False))

print("\nCofactor_Type 统计:")
print(df_tasks["Cofactor_Type"].value_counts(dropna=False))

display(df_tasks.head())

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [4]:
# =========================================================
# Cell B0-repair：修复 Boltz notebook 中 numpy / pandas 二进制不兼容
# 运行完后必须 Runtime -> Restart session
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

!python --version
!nvidia-smi || true

# 1. 升级安装工具
!pip install -q --upgrade pip setuptools wheel

# 2. 先卸载容易冲突的科学栈
!pip uninstall -y numpy pandas scipy scikit-learn

# 3. 安装 Boltz 需要的主程序
!pip install -q -U "boltz[cuda]"

# 4. 重新安装与 Colab / pandas 兼容的科学栈
# 重点：pandas 必须在 numpy 固定后重新安装
!pip install --no-cache-dir --force-reinstall \
    "numpy==1.26.4" \
    "pandas==2.2.2" \
    "scipy==1.14.1" \
    "scikit-learn==1.5.2" \
    "requests==2.32.4" \
    "openpyxl" \
    "pyyaml" \
    "tqdm" \
    "biopython"

print("✅ 修复安装完成。现在请点击 Runtime -> Restart session，然后从 Cell B0-verify 继续。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Python 3.12.13
Thu May 21 11:45:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             45W /  400W |       6MiB /  40960MiB |      0%      Defaul

✅ 修复安装完成。现在请点击 Runtime -> Restart session，然后从 Cell B0-verify 继续。


In [1]:
# =========================================================
# Cell B0-verify：重启后验证 Boltz notebook 环境
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import scipy
import sklearn
import torch
import requests

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
print("torch:", torch.__version__)
print("requests:", requests.__version__)
print("cuda available:", torch.cuda.is_available())

print("\nBoltz command:")
!which boltz
!boltz predict --help | head -n 30

print("\n✅ 如果 pandas 能 import，boltz predict --help 能显示，就可以继续 Cell B1。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
numpy: 1.26.4
pandas: 2.2.2
scipy: 1.14.1
sklearn: 1.5.2
torch: 2.10.0+cu128
requests: 2.32.4
cuda available: True

Boltz command:
/usr/local/bin/boltz
Usage: boltz predict [OPTIONS] DATA

  Run predictions with Boltz.

Options:
  --out_dir PATH                  The path where to save the predictions.
  --cache PATH                    The directory where to download the data and
                                  model. Default is ~/.boltz, or $BOLTZ_CACHE
                                  if set.
  --checkpoint PATH               An optional checkpoint, will use the
                                  provided Boltz-1 model by default.
  --devices INTEGER               The number of devices to use for prediction.
                                  Default is 1.
  --accelerator [gpu|cpu|tpu]     The accelerator to use for prediction.
                             

In [2]:
# =========================================================
# Cell B1：检查前序结果文件
# =========================================================

from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

mapping_xlsx = BASE / "cofactor_mapping/EC_to_Cofactor_MultiEvidence_Mapping.xlsx"
manifest_xlsx = BASE / "cofactor_modeling_manifest/01_All500_Modeling_Manifest.xlsx"
fasta_path = BASE / "Top500_Candidate_Enzymes_Cleaned.fasta"

files = {
    "cofactor_mapping": mapping_xlsx,
    "modeling_manifest": manifest_xlsx,
    "fasta": fasta_path,
}

for name, path in files.items():
    print(name, "=>", path, "exists:", path.exists())

missing = [name for name, path in files.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"缺少文件: {missing}")

df_tasks = pd.read_excel(manifest_xlsx, sheet_name="all_modeling_tasks")

print("任务总数:", len(df_tasks))

print("\nModeling_Queue 统计:")
print(df_tasks["Modeling_Queue"].value_counts(dropna=False))

print("\nCofactor_Type 统计:")
print(df_tasks["Cofactor_Type"].value_counts(dropna=False))

display(df_tasks.head())

cofactor_mapping => /content/drive/MyDrive/Horizyn_Checkpoints/cofactor_mapping/EC_to_Cofactor_MultiEvidence_Mapping.xlsx exists: True
modeling_manifest => /content/drive/MyDrive/Horizyn_Checkpoints/cofactor_modeling_manifest/01_All500_Modeling_Manifest.xlsx exists: True
fasta => /content/drive/MyDrive/Horizyn_Checkpoints/Top500_Candidate_Enzymes_Cleaned.fasta exists: True
任务总数: 1053

Modeling_Queue 统计:
Modeling_Queue
Primary_holo                573
Secondary_holo              335
Apo_only_control            138
Secondary_review_no_holo      7
Name: count, dtype: int64

Cofactor_Type 统计:
Cofactor_Type
metal_ion                 460
organic_cofactor          373
NaN                       145
heme                       38
metal_cluster              36
metal_complex_cofactor      1
Name: count, dtype: int64


,Task_ID,Enzyme_ID,Original_ID,CleanContact_Entry,Predicted_EC_4digit,Top1_EC,CLEANContact_Distance,Sequence_length,Cofactor_Category,Modeling_Queue,...,Cofactor_Score,Cofactor_Confidence,Model_Token_or_CCD,Cofactor_Type,Modeling_Note,Evidence_Sources,All_Cofactor_Candidates,Input_PDB_Path,PDB_Found,Recommended_Next_Tool
0,NODE_1_length_436095_cov_65.793628_39__MetalMg...,NODE_1_length_436095_cov_65.793628_39,NODE_1_length_436095_cov_65.793628_39,NODE_1_length_436095_cov_65.793628_39,2.7.1.113;2.7.1.76,2.7.1.113,10.3859;10.4172,404,Probable cofactor-dependent,Primary_holo,...,0.4,Probable,MG,metal_ion,Mg2+，后续需验证 Asp/Glu 配位几何,M-CSA,Metal:Mg:0.40,/content/drive/MyDrive/Enzyme_Mining/best_conf...,True,metal-site predictor + coordination check
1,NODE_1_length_436095_cov_65.793628_41__MetalMg...,NODE_1_length_436095_cov_65.793628_41,NODE_1_length_436095_cov_65.793628_41,NODE_1_length_436095_cov_65.793628_41,2.7.1.113,2.7.1.113,8.8571,362,Probable cofactor-dependent,Primary_holo,...,0.4,Probable,MG,metal_ion,Mg2+，后续需验证 Asp/Glu 配位几何,M-CSA,Metal:Mg:0.40,/content/drive/MyDrive/Enzyme_Mining/best_conf...,True,metal-site predictor + coordination check
2,NODE_1_length_436095_cov_65.793628_87__NADP+__...,NODE_1_length_436095_cov_65.793628_87,NODE_1_length_436095_cov_65.793628_87,NODE_1_length_436095_cov_65.793628_87,1.1.1.355,1.1.1.355,9.0461,238,Probable cofactor-dependent,Primary_holo,...,0.5,Probable,NAP,organic_cofactor,NADP+/NADPH 状态需根据反应方向确认；PDB CCD 常用 NAP,EC-class-prior;KEGG;UniProt,NADP+:0.50;NAD+:0.20;FAD:0.05;Fe-S cluster:0.0...,/content/drive/MyDrive/Enzyme_Mining/best_conf...,True,DISCODE + Rossmann-toolbox
3,NODE_1_length_436095_cov_65.793628_87__NAD+__r...,NODE_1_length_436095_cov_65.793628_87,NODE_1_length_436095_cov_65.793628_87,NODE_1_length_436095_cov_65.793628_87,1.1.1.355,1.1.1.355,9.0461,238,Probable cofactor-dependent,Primary_holo,...,0.2,Probable,NAD,organic_cofactor,NAD+/NADH 状态需根据反应方向确认；可先用 NAD 作为 holo 建模 ligand,EC-class-prior;KEGG;UniProt,NADP+:0.50;NAD+:0.20;FAD:0.05;Fe-S cluster:0.0...,/content/drive/MyDrive/Enzyme_Mining/best_conf...,True,DISCODE + Rossmann-toolbox
4,NODE_1_length_436095_cov_65.793628_95__CoA__rank1,NODE_1_length_436095_cov_65.793628_95,NODE_1_length_436095_cov_65.793628_95,NODE_1_length_436095_cov_65.793628_95,2.3.1.30,2.3.1.30,7.6437,200,Probable cofactor-dependent,Primary_holo,...,0.5,Probable,COA,organic_cofactor,如果反应涉及酰基转移，后续应考虑具体 acyl-CoA,EC-class-prior;KEGG;UniProt,CoA:0.50,/content/drive/MyDrive/Enzyme_Mining/best_conf...,True,AlphaFill/BioLiP2 + holo cofolding


In [3]:
# =========================================================
# Cell B2：从 Primary_holo manifest 生成 Boltz YAML
# =========================================================

import re
import yaml
import pandas as pd
from pathlib import Path
from Bio import SeqIO
from tqdm.auto import tqdm

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

manifest_xlsx = BASE / "cofactor_modeling_manifest/01_All500_Modeling_Manifest.xlsx"
fasta_path = BASE / "Top500_Candidate_Enzymes_Cleaned.fasta"

yaml_dir = BASE / "boltz_inputs/primary_organic_yaml"
yaml_dir.mkdir(parents=True, exist_ok=True)

summary_out = BASE / "boltz_inputs/primary_organic_yaml_manifest.xlsx"

df_tasks = pd.read_excel(manifest_xlsx, sheet_name="all_modeling_tasks")

def safe_id(x):
    x = str(x).strip()
    x = re.sub(r"[^A-Za-z0-9_.-]+", "_", x)
    return x[:220]

# 建 FASTA 索引
seq_by_id = {}

for rec in SeqIO.parse(str(fasta_path), "fasta"):
    seq = str(rec.seq).upper().replace("*", "").replace(" ", "")
    possible_ids = {
        rec.id,
        rec.name,
        rec.description.split()[0],
        safe_id(rec.id),
        safe_id(rec.description.split()[0]),
    }
    for pid in possible_ids:
        if pid:
            seq_by_id[str(pid)] = seq

print("FASTA 序列索引数量:", len(seq_by_id))

def get_sequence(row):
    candidates = []
    for c in ["Enzyme_ID", "Original_ID", "CleanContact_Entry"]:
        if c in row and pd.notna(row[c]):
            candidates.append(str(row[c]))
            candidates.append(safe_id(str(row[c])))

    for c in candidates:
        if c in seq_by_id:
            return seq_by_id[c]

    return ""

def clean_task_name(s):
    s = str(s)
    s = re.sub(r"[^A-Za-z0-9_.-]+", "_", s)
    return s[:180]

# 先只跑有机辅因子 + heme
allowed_types = {"organic_cofactor", "heme"}

df_primary = df_tasks[
    (df_tasks["Modeling_Queue"] == "Primary_holo") &
    (df_tasks["Cofactor_Type"].isin(allowed_types)) &
    (df_tasks["Model_Token_or_CCD"].astype(str).str.len() > 0)
].copy()

df_primary["Cofactor_Score"] = pd.to_numeric(df_primary["Cofactor_Score"], errors="coerce").fillna(0)
df_primary = df_primary.sort_values("Cofactor_Score", ascending=False)

print("Primary organic/heme Boltz 候选任务数:", len(df_primary))

# 清理旧 YAML，避免重复
for old in yaml_dir.glob("*.yaml"):
    old.unlink()

rows = []

for _, row in tqdm(df_primary.iterrows(), total=len(df_primary), desc="Writing Boltz YAML"):
    seq = get_sequence(row)

    if not seq:
        rows.append({
            "Task_ID": row["Task_ID"],
            "Enzyme_ID": row.get("Enzyme_ID", ""),
            "Cofactor": row.get("Cofactor", ""),
            "YAML_Path": "",
            "Status": "missing_sequence",
            "Sequence_Length": 0
        })
        continue

    task_name = clean_task_name(row["Task_ID"])
    yaml_path = yaml_dir / f"{task_name}.yaml"

    ccd = str(row["Model_Token_or_CCD"]).strip()

    boltz_input = {
        "sequences": [
            {
                "protein": {
                    "id": "A",
                    "sequence": seq
                }
            },
            {
                "ligand": {
                    "id": "B",
                    "ccd": ccd
                }
            }
        ]
    }

    with open(yaml_path, "w") as f:
        yaml.safe_dump(boltz_input, f, sort_keys=False)

    rows.append({
        "Task_ID": row["Task_ID"],
        "Enzyme_ID": row.get("Enzyme_ID", ""),
        "Cofactor": row.get("Cofactor", ""),
        "Cofactor_Score": row.get("Cofactor_Score", ""),
        "Cofactor_Type": row.get("Cofactor_Type", ""),
        "CCD": ccd,
        "YAML_Path": str(yaml_path),
        "Status": "ok",
        "Sequence_Length": len(seq),
        "Predicted_EC_4digit": row.get("Predicted_EC_4digit", ""),
        "Evidence_Sources": row.get("Evidence_Sources", ""),
    })

df_yaml = pd.DataFrame(rows)
df_yaml.to_excel(summary_out, index=False)

print("\n✅ Boltz YAML 生成完成")
print("YAML 目录:", yaml_dir)
print("YAML manifest:", summary_out)
print("\n状态统计:")
print(df_yaml["Status"].value_counts(dropna=False))

display(df_yaml.head(20))

FASTA 序列索引数量: 500
Primary organic/heme Boltz 候选任务数: 371


Writing Boltz YAML:   0%|          | 0/371 [00:00<?, ?it/s]


✅ Boltz YAML 生成完成
YAML 目录: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_inputs/primary_organic_yaml
YAML manifest: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_inputs/primary_organic_yaml_manifest.xlsx

状态统计:
Status
ok    371
Name: count, dtype: int64


,Task_ID,Enzyme_ID,Cofactor,Cofactor_Score,Cofactor_Type,CCD,YAML_Path,Status,Sequence_Length,Predicted_EC_4digit,Evidence_Sources
0,NODE_39_length_35792_cov_65.109898_24__CoA__rank1,NODE_39_length_35792_cov_65.109898_24,CoA,0.9,organic_cofactor,COA,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,ok,399,2.3.1.16,EC-class-prior;KEGG;M-CSA;UniProt
1,NODE_3_length_362446_cov_66.707646_306__Heme__...,NODE_3_length_362446_cov_66.707646_306,Heme,0.9,heme,HEM,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,ok,98,1.14.14.18,EC-class-prior;KEGG;M-CSA;UniProt
2,NODE_3_length_362446_cov_66.707646_252__FMN__r...,NODE_3_length_362446_cov_66.707646_252,FMN,0.9,organic_cofactor,FMN,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,ok,236,1.4.3.5,EC-class-prior;KEGG;M-CSA;UniProt
3,NODE_7_length_201326_cov_66.280429_78__FAD__rank1,NODE_7_length_201326_cov_66.280429_78,FAD,0.9,organic_cofactor,FAD,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,ok,338,1.18.1.2,EC-class-prior;KEGG;M-CSA;UniProt
4,NODE_7_length_201326_cov_66.280429_78__NADP+__...,NODE_7_length_201326_cov_66.280429_78,NADP+,0.9,organic_cofactor,NAP,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,ok,338,1.18.1.2,EC-class-prior;KEGG;M-CSA;UniProt
5,NODE_1_length_436095_cov_65.793628_272__Heme__...,NODE_1_length_436095_cov_65.793628_272,Heme,0.9,heme,HEM,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,ok,182,1.14.14.18,EC-class-prior;KEGG;M-CSA;UniProt
6,NODE_56_length_6717_cov_64.332380_3__NAD+__rank1,NODE_56_length_6717_cov_64.332380_3,NAD+,0.9,organic_cofactor,NAD,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,ok,145,1.6.5.2,EC-class-prior;KEGG;M-CSA;UniProt
7,NODE_6_length_231008_cov_66.606731_97__FAD__rank1,NODE_6_length_231008_cov_66.606731_97,FAD,0.9,organic_cofactor,FAD,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,ok,85,1.4.3.2,EC-class-prior;KEGG;M-CSA;UniProt
8,NODE_6_length_231008_cov_66.606731_103__FAD__r...,NODE_6_length_231008_cov_66.606731_103,FAD,0.9,organic_cofactor,FAD,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,ok,107,1.4.3.2,EC-class-prior;KEGG;M-CSA;UniProt
9,NODE_27_length_69195_cov_65.603663_62__NADP+__...,NODE_27_length_69195_cov_65.603663_62,NADP+,0.9,organic_cofactor,NAP,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,ok,344,1.17.1.8,EC-class-prior;KEGG;M-CSA;UniProt


In [4]:
# =========================================================
# Cell B3：生成 pilot 10 YAML 子集
# =========================================================

import shutil
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

summary_out = BASE / "boltz_inputs/primary_organic_yaml_manifest.xlsx"
pilot_dir = BASE / "boltz_inputs/pilot_10_yaml"
pilot_dir.mkdir(parents=True, exist_ok=True)

df_yaml = pd.read_excel(summary_out)

df_ok = df_yaml[df_yaml["Status"] == "ok"].copy()
df_ok["Cofactor_Score"] = pd.to_numeric(df_ok["Cofactor_Score"], errors="coerce").fillna(0)
df_ok = df_ok.sort_values("Cofactor_Score", ascending=False)

df_pilot = df_ok.head(10).copy()

# 清空旧 pilot YAML
for f in pilot_dir.glob("*.yaml"):
    f.unlink()

for _, row in df_pilot.iterrows():
    src = Path(row["YAML_Path"])
    dst = pilot_dir / src.name
    shutil.copy2(src, dst)

pilot_manifest = pilot_dir / "pilot_10_manifest.xlsx"
df_pilot.to_excel(pilot_manifest, index=False)

print("✅ Pilot YAML 已生成")
print("Pilot 数量:", len(df_pilot))
print("Pilot 目录:", pilot_dir)
print("Pilot manifest:", pilot_manifest)

display(df_pilot[["Task_ID", "Enzyme_ID", "Cofactor", "Cofactor_Score", "CCD", "YAML_Path"]])

✅ Pilot YAML 已生成
Pilot 数量: 10
Pilot 目录: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_inputs/pilot_10_yaml
Pilot manifest: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_inputs/pilot_10_yaml/pilot_10_manifest.xlsx


,Task_ID,Enzyme_ID,Cofactor,Cofactor_Score,CCD,YAML_Path
23,NODE_26_length_70757_cov_67.122722_55__CoA__rank1,NODE_26_length_70757_cov_67.122722_55,CoA,0.9,COA,/content/drive/MyDrive/Horizyn_Checkpoints/bol...
22,NODE_25_length_76722_cov_66.604332_6__NAD+__rank1,NODE_25_length_76722_cov_66.604332_6,NAD+,0.9,NAD,/content/drive/MyDrive/Horizyn_Checkpoints/bol...
32,NODE_1_length_436095_cov_65.793628_305__FMN__r...,NODE_1_length_436095_cov_65.793628_305,FMN,0.9,FMN,/content/drive/MyDrive/Horizyn_Checkpoints/bol...
33,NODE_6_length_231008_cov_66.606731_16__FAD__rank1,NODE_6_length_231008_cov_66.606731_16,FAD,0.9,FAD,/content/drive/MyDrive/Horizyn_Checkpoints/bol...
21,NODE_25_length_76722_cov_66.604332_89__FAD__rank1,NODE_25_length_76722_cov_66.604332_89,FAD,0.9,FAD,/content/drive/MyDrive/Horizyn_Checkpoints/bol...
20,NODE_11_length_154862_cov_66.368492_122__Heme_...,NODE_11_length_154862_cov_66.368492_122,Heme,0.9,HEM,/content/drive/MyDrive/Horizyn_Checkpoints/bol...
19,NODE_25_length_76722_cov_66.604332_89__NADP+__...,NODE_25_length_76722_cov_66.604332_89,NADP+,0.9,NAP,/content/drive/MyDrive/Horizyn_Checkpoints/bol...
18,NODE_12_length_136290_cov_66.430554_166__Heme_...,NODE_12_length_136290_cov_66.430554_166,Heme,0.9,HEM,/content/drive/MyDrive/Horizyn_Checkpoints/bol...
8,NODE_6_length_231008_cov_66.606731_103__FAD__r...,NODE_6_length_231008_cov_66.606731_103,FAD,0.9,FAD,/content/drive/MyDrive/Horizyn_Checkpoints/bol...
16,NODE_22_length_94753_cov_66.514481_84__FAD__rank1,NODE_22_length_94753_cov_66.514481_84,FAD,0.9,FAD,/content/drive/MyDrive/Horizyn_Checkpoints/bol...


In [5]:
# =========================================================
# Cell B4：运行 Boltz pilot 10
# =========================================================

from pathlib import Path

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

pilot_dir = BASE / "boltz_inputs/pilot_10_yaml"
out_dir = BASE / "boltz_outputs/pilot_10"
cache_dir = BASE / "boltz_cache"

out_dir.mkdir(parents=True, exist_ok=True)
cache_dir.mkdir(parents=True, exist_ok=True)

print("输入目录:", pilot_dir)
print("输出目录:", out_dir)

# 首选运行方式
!BOLTZ_CACHE="$cache_dir" boltz predict "$pilot_dir" \
    --out_dir "$out_dir" \
    --use_msa_server \
    --output_format pdb \
    --override

# 如果上面因为 kernel / cuequivariance / triton 报错，
# 改用下面这个更稳但较慢的版本：
# !BOLTZ_CACHE="$cache_dir" boltz predict "$pilot_dir" \
#     --out_dir "$out_dir" \
#     --use_msa_server \
#     --output_format pdb \
#     --override \
#     --no_kernels

输入目录: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_inputs/pilot_10_yaml
输出目录: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/pilot_10
MSA server enabled: https://api.colabfold.com
MSA server authentication: no credentials provided
Extracting the CCD data to /content/drive/MyDrive/Horizyn_Checkpoints/boltz_cache/mols. This may take a bit of time. You may change the cache directory with the --cache flag.
Checking input data.
Traceback (most recent call last):
  File "/usr/local/bin/boltz", line 8, in <module>
    sys.exit(cli())
             ^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1157, in __call__
    return self.main(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1078, in main
    rv = self.invoke(ctx)
         ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1688, in invoke
    return _process_result(sub_ctx.command.invoke(s

In [6]:
# =========================================================
# Cell B5：汇总 Boltz pilot confidence
# =========================================================

import json
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

out_dir = BASE / "boltz_outputs/pilot_10"
pilot_manifest = BASE / "boltz_inputs/pilot_10_yaml/pilot_10_manifest.xlsx"

df_pilot = pd.read_excel(pilot_manifest)

rows = []

pred_root = out_dir / "predictions"
print("Prediction root exists:", pred_root.exists(), pred_root)

for pred_dir in pred_root.glob("*"):
    if not pred_dir.is_dir():
        continue

    input_name = pred_dir.name

    conf_files = sorted(pred_dir.glob("confidence_*_model_0.json"))
    pdb_files = sorted(pred_dir.glob("*_model_0.pdb"))

    # 有些版本输出层级不同，兜底递归找
    if not conf_files:
        conf_files = sorted(pred_dir.rglob("confidence*.json"))
    if not pdb_files:
        pdb_files = sorted(pred_dir.rglob("*model_0.pdb"))

    conf = {}
    if conf_files:
        with open(conf_files[0]) as f:
            conf = json.load(f)

    rows.append({
        "Input_Name": input_name,
        "Prediction_Dir": str(pred_dir),
        "Best_Model_PDB": str(pdb_files[0]) if pdb_files else "",
        "Confidence_JSON": str(conf_files[0]) if conf_files else "",
        "confidence_score": conf.get("confidence_score", None),
        "ptm": conf.get("ptm", None),
        "iptm": conf.get("iptm", None),
        "ligand_iptm": conf.get("ligand_iptm", None),
        "complex_plddt": conf.get("complex_plddt", None),
        "complex_iplddt": conf.get("complex_iplddt", None),
        "complex_pde": conf.get("complex_pde", None),
        "complex_ipde": conf.get("complex_ipde", None),
    })

df_conf = pd.DataFrame(rows)

summary_path = BASE / "boltz_outputs/pilot_10_confidence_summary.xlsx"
df_conf.to_excel(summary_path, index=False)

print("✅ Boltz pilot confidence 汇总完成")
print(summary_path)

if len(df_conf) == 0:
    print("⚠️ 没有找到 predictions 输出。请检查 Cell B4 日志。")
else:
    display(df_conf.sort_values("confidence_score", ascending=False, na_position="last").head(20))

Prediction root exists: False /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/pilot_10/predictions
✅ Boltz pilot confidence 汇总完成
/content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/pilot_10_confidence_summary.xlsx
⚠️ 没有找到 predictions 输出。请检查 Cell B4 日志。


In [7]:
# =========================================================
# Cell B5-debug：诊断 Boltz pilot 是否真的有输出
# =========================================================

from pathlib import Path

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

pilot_dir = BASE / "boltz_inputs/pilot_10_yaml"
out_dir = BASE / "boltz_outputs/pilot_10"

print("pilot_dir exists:", pilot_dir.exists(), pilot_dir)
print("out_dir exists:", out_dir.exists(), out_dir)

print("\nPilot directory files:")
if pilot_dir.exists():
    for p in sorted(pilot_dir.iterdir()):
        print(p.name)

print("\nOutput directory tree, first 80 files:")
if out_dir.exists():
    count = 0
    for p in out_dir.rglob("*"):
        print(p)
        count += 1
        if count >= 80:
            break
else:
    print("out_dir 不存在")

print("\nSearching for prediction files:")
for pattern in ["*.pdb", "*.cif", "*.json", "confidence*.json", "*model_0*"]:
    hits = list(out_dir.rglob(pattern)) if out_dir.exists() else []
    print(pattern, "=>", len(hits))
    for h in hits[:10]:
        print(" ", h)

pilot_dir exists: True /content/drive/MyDrive/Horizyn_Checkpoints/boltz_inputs/pilot_10_yaml
out_dir exists: True /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/pilot_10

Pilot directory files:
NODE_11_length_154862_cov_66.368492_122__Heme__rank1.yaml
NODE_12_length_136290_cov_66.430554_166__Heme__rank1.yaml
NODE_1_length_436095_cov_65.793628_305__FMN__rank1.yaml
NODE_22_length_94753_cov_66.514481_84__FAD__rank1.yaml
NODE_25_length_76722_cov_66.604332_6__NAD___rank1.yaml
NODE_25_length_76722_cov_66.604332_89__FAD__rank1.yaml
NODE_25_length_76722_cov_66.604332_89__NADP___rank2.yaml
NODE_26_length_70757_cov_67.122722_55__CoA__rank1.yaml
NODE_6_length_231008_cov_66.606731_103__FAD__rank1.yaml
NODE_6_length_231008_cov_66.606731_16__FAD__rank1.yaml
pilot_10_manifest.xlsx

Output directory tree, first 80 files:
/content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/pilot_10/boltz_results_pilot_10_yaml

Searching for prediction files:
*.pdb => 0
*.cif => 0
*.json => 0
confidence*.

In [8]:
# =========================================================
# Cell B4-single-debug：单个 YAML 最小化 Boltz 测试
# =========================================================

from pathlib import Path
import shutil
import os

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

# 你的 pilot YAML 目录
pilot_dir = BASE / "boltz_inputs/pilot_10_yaml"

# 本地临时目录，避免直接在 Google Drive 里跑
single_input_dir = Path("/content/boltz_single_input")
single_out_dir = Path("/content/boltz_single_out")
single_cache_dir = Path("/content/boltz_cache")
log_path = Path("/content/boltz_single_run.log")

# 清理旧目录
for d in [single_input_dir, single_out_dir]:
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

single_cache_dir.mkdir(parents=True, exist_ok=True)

# 只选择 .yaml，不要 xlsx
yaml_files = sorted(pilot_dir.glob("*.yaml"))

print("找到 YAML 数量:", len(yaml_files))
for y in yaml_files[:10]:
    print(" -", y.name)

if len(yaml_files) == 0:
    raise FileNotFoundError(f"{pilot_dir} 下没有 .yaml 文件")

# 优先选择 FAD / FMN / NAD / SAM / CoA，先避开 Heme
preferred = []
for y in yaml_files:
    name = y.name.upper()
    if any(x in name for x in ["FAD", "FMN", "NAD", "NAP", "SAM", "COA"]):
        preferred.append(y)

test_yaml = preferred[0] if preferred else yaml_files[0]
dst_yaml = single_input_dir / test_yaml.name
shutil.copy2(test_yaml, dst_yaml)

print("\n本次测试 YAML:")
print(dst_yaml)

print("\nYAML 内容预览:")
print(dst_yaml.read_text()[:1000])

print("\n本地输入目录:")
!ls -lh /content/boltz_single_input

print("\nBoltz 版本/帮助:")
!which boltz
!boltz predict --help | head -n 60

找到 YAML 数量: 10
 - NODE_11_length_154862_cov_66.368492_122__Heme__rank1.yaml
 - NODE_12_length_136290_cov_66.430554_166__Heme__rank1.yaml
 - NODE_1_length_436095_cov_65.793628_305__FMN__rank1.yaml
 - NODE_22_length_94753_cov_66.514481_84__FAD__rank1.yaml
 - NODE_25_length_76722_cov_66.604332_6__NAD___rank1.yaml
 - NODE_25_length_76722_cov_66.604332_89__FAD__rank1.yaml
 - NODE_25_length_76722_cov_66.604332_89__NADP___rank2.yaml
 - NODE_26_length_70757_cov_67.122722_55__CoA__rank1.yaml
 - NODE_6_length_231008_cov_66.606731_103__FAD__rank1.yaml
 - NODE_6_length_231008_cov_66.606731_16__FAD__rank1.yaml

本次测试 YAML:
/content/boltz_single_input/NODE_1_length_436095_cov_65.793628_305__FMN__rank1.yaml

YAML 内容预览:
sequences:
- protein:
    id: A
    sequence: MTLLEGLLADLKLEGDQLWNAVAGLDADGWATPTPAAGWTVATQIAHLLWTDEVAVISATDKQAWDELVLVAIQDPTGYVDQQAIEVARLAPEALLARWGKAREALPAALRAVPKGQKMPWFGPPMSPTSMATARFMETWAHALDVYDALGIEPERSDRVRHVAHLGVRTRDFAFSVHELPAPTEEFRIDLVSPSGDQWSWGPEDAAQTVTGSAWDFCLLVTQRVHRGDTDLVASGTDAEH

In [9]:
# =========================================================
# Cell B4-single-run：运行单个 Boltz 测试并保存日志
# =========================================================

from pathlib import Path

single_input_dir = Path("/content/boltz_single_input")
single_out_dir = Path("/content/boltz_single_out")
single_cache_dir = Path("/content/boltz_cache")
log_path = Path("/content/boltz_single_run.log")

print("输入目录:", single_input_dir)
print("输出目录:", single_out_dir)
print("缓存目录:", single_cache_dir)
print("日志文件:", log_path)

!BOLTZ_CACHE="/content/boltz_cache" boltz predict "/content/boltz_single_input" \
    --out_dir "/content/boltz_single_out" \
    --use_msa_server \
    --output_format pdb \
    --override \
    --no_kernels \
    2>&1 | tee "/content/boltz_single_run.log"

输入目录: /content/boltz_single_input
输出目录: /content/boltz_single_out
缓存目录: /content/boltz_cache
日志文件: /content/boltz_single_run.log
MSA server enabled: https://api.colabfold.com
MSA server authentication: no credentials provided
Extracting the CCD data to /content/boltz_cache/mols. This may take a bit of time. You may change the cache directory with the --cache flag.
Checking input data.
Processing 1 inputs with 1 threads.
  0%|          | 0/1 [00:00<?, ?it/s]Generating MSA for /content/boltz_single_input/NODE_1_length_436095_cov_65.793628_305__FMN__rank1.yaml with 1 protein entities.
Calling MSA server for target NODE_1_length_436095_cov_65.793628_305__FMN__rank1 with 1 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server

PENDING:   0%|          | 0/150 [elapsed: 00:00 remaining: ?]Sleeping for 6s. Reason: PENDING

RUNNING:   4%|▍         | 6/150 [elapsed: 00:06 remaining: 02:37]Sleeping for 8s. Reason: RUNNING

RUNNI

In [10]:
# =========================================================
# Cell B4-single-check：检查单个 Boltz 输出
# =========================================================

from pathlib import Path

single_out_dir = Path("/content/boltz_single_out")
log_path = Path("/content/boltz_single_run.log")

print("输出目录存在:", single_out_dir.exists())
print("\n输出目录树，前 100 个文件:")
if single_out_dir.exists():
    count = 0
    for p in single_out_dir.rglob("*"):
        print(p)
        count += 1
        if count >= 100:
            break

print("\n搜索输出文件:")
for pattern in ["*.pdb", "*.cif", "*.json", "confidence*.json", "*model_0*"]:
    hits = list(single_out_dir.rglob(pattern)) if single_out_dir.exists() else []
    print(pattern, "=>", len(hits))
    for h in hits[:10]:
        print(" ", h)

print("\n日志最后 80 行:")
if log_path.exists():
    lines = log_path.read_text(errors="ignore").splitlines()
    for line in lines[-80:]:
        print(line)
else:
    print("没有找到日志文件")

输出目录存在: True

输出目录树，前 100 个文件:
/content/boltz_single_out/boltz_results_boltz_single_input
/content/boltz_single_out/boltz_results_boltz_single_input/predictions
/content/boltz_single_out/boltz_results_boltz_single_input/processed
/content/boltz_single_out/boltz_results_boltz_single_input/lightning_logs
/content/boltz_single_out/boltz_results_boltz_single_input/msa
/content/boltz_single_out/boltz_results_boltz_single_input/predictions/NODE_1_length_436095_cov_65.793628_305__FMN__rank1
/content/boltz_single_out/boltz_results_boltz_single_input/processed/manifest.json
/content/boltz_single_out/boltz_results_boltz_single_input/processed/constraints
/content/boltz_single_out/boltz_results_boltz_single_input/processed/records
/content/boltz_single_out/boltz_results_boltz_single_input/processed/msa
/content/boltz_single_out/boltz_results_boltz_single_input/processed/structures
/content/boltz_single_out/boltz_results_boltz_single_input/processed/templates
/content/boltz_single_out/boltz_result

In [11]:
# =========================================================
# Cell B4-batch-local：本地跑 pilot 10，完成后复制回云盘
# =========================================================

from pathlib import Path
import shutil

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

drive_pilot_dir = BASE / "boltz_inputs/pilot_10_yaml"
local_input_dir = Path("/content/boltz_pilot10_input")
local_out_dir = Path("/content/boltz_pilot10_out")
local_cache_dir = Path("/content/boltz_cache")
drive_out_dir = BASE / "boltz_outputs/pilot_10_localrun"

# 清理本地目录
for d in [local_input_dir, local_out_dir]:
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

local_cache_dir.mkdir(parents=True, exist_ok=True)

# 只复制 YAML，不复制 xlsx
yaml_files = sorted(drive_pilot_dir.glob("*.yaml"))
for y in yaml_files:
    shutil.copy2(y, local_input_dir / y.name)

print("本地 YAML 数量:", len(list(local_input_dir.glob("*.yaml"))))
print("本地输入目录:", local_input_dir)
print("本地输出目录:", local_out_dir)

!BOLTZ_CACHE="/content/boltz_cache" boltz predict "/content/boltz_pilot10_input" \
    --out_dir "/content/boltz_pilot10_out" \
    --use_msa_server \
    --output_format pdb \
    --override \
    --no_kernels \
    2>&1 | tee "/content/boltz_pilot10_run.log"

# 复制结果回云盘
if drive_out_dir.exists():
    shutil.rmtree(drive_out_dir)
shutil.copytree(local_out_dir, drive_out_dir)

shutil.copy2("/content/boltz_pilot10_run.log", BASE / "boltz_outputs/pilot_10_localrun.log")

print("✅ 已复制输出到云盘:")
print(drive_out_dir)

本地 YAML 数量: 10
本地输入目录: /content/boltz_pilot10_input
本地输出目录: /content/boltz_pilot10_out
MSA server enabled: https://api.colabfold.com
MSA server authentication: no credentials provided
Checking input data.
Processing 10 inputs with 10 threads.
  0%|          | 0/10 [00:00<?, ?it/s]Generating MSA for /content/boltz_pilot10_input/NODE_6_length_231008_cov_66.606731_16__FAD__rank1.yaml with 1 protein entities.
Calling MSA server for target NODE_6_length_231008_cov_66.606731_16__FAD__rank1 with 1 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server
SUBMIT:   0%|          | 0/150 [elapsed: 00:00 remaining: ?]Generating MSA for /content/boltz_pilot10_input/NODE_6_length_231008_cov_66.606731_103__FAD__rank1.yaml with 1 protein entities.
Calling MSA server for target NODE_6_length_231008_cov_66.606731_103__FAD__rank1 with 1 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provi

In [12]:
# =========================================================
# Cell B5-localrun-summary：汇总 localrun pilot 输出
# =========================================================

import json
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")
out_dir = BASE / "boltz_outputs/pilot_10_localrun"

rows = []

all_conf = sorted(out_dir.rglob("confidence*.json"))
all_pdb = sorted(out_dir.rglob("*.pdb"))
all_cif = sorted(out_dir.rglob("*.cif"))

print("confidence json:", len(all_conf))
print("pdb files:", len(all_pdb))
print("cif files:", len(all_cif))

candidate_dirs = sorted(set([p.parent for p in all_conf + all_pdb + all_cif]))

for pred_dir in candidate_dirs:
    conf_files = sorted(pred_dir.glob("confidence*.json"))
    pdb_files = sorted(pred_dir.glob("*.pdb"))
    cif_files = sorted(pred_dir.glob("*.cif"))

    conf = {}
    if conf_files:
        with open(conf_files[0]) as f:
            conf = json.load(f)

    rows.append({
        "Prediction_Dir": str(pred_dir),
        "Best_Model_PDB": str(pdb_files[0]) if pdb_files else "",
        "Best_Model_CIF": str(cif_files[0]) if cif_files else "",
        "Confidence_JSON": str(conf_files[0]) if conf_files else "",
        "confidence_score": conf.get("confidence_score", None),
        "ptm": conf.get("ptm", None),
        "iptm": conf.get("iptm", None),
        "ligand_iptm": conf.get("ligand_iptm", None),
        "complex_plddt": conf.get("complex_plddt", None),
        "complex_iplddt": conf.get("complex_iplddt", None),
        "complex_pde": conf.get("complex_pde", None),
        "complex_ipde": conf.get("complex_ipde", None),
    })

df_conf = pd.DataFrame(rows)

summary_path = BASE / "boltz_outputs/pilot_10_localrun_confidence_summary.xlsx"
df_conf.to_excel(summary_path, index=False)

print("✅ 汇总完成:")
print(summary_path)

if len(df_conf) > 0:
    display(df_conf.sort_values("confidence_score", ascending=False, na_position="last").head(20))
else:
    print("仍然没有找到输出，请查看 /content/boltz_pilot10_run.log 或云盘里的 pilot_10_localrun.log")

confidence json: 10
pdb files: 10
cif files: 0
✅ 汇总完成:
/content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/pilot_10_localrun_confidence_summary.xlsx


,Prediction_Dir,Best_Model_PDB,Best_Model_CIF,Confidence_JSON,confidence_score,ptm,iptm,ligand_iptm,complex_plddt,complex_iplddt,complex_pde,complex_ipde
7,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,0.926218,0.966857,0.893962,0.893962,0.934282,0.836107,0.495222,2.388013
2,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,0.908389,0.955979,0.901206,0.901206,0.910185,0.699018,0.447678,1.180011
4,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,0.896801,0.947516,0.841174,0.841174,0.910707,0.767104,0.485976,1.449611
0,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,0.895598,0.945249,0.987283,0.987283,0.872677,0.899861,0.408127,0.532319
1,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,0.863788,0.919531,0.943623,0.943623,0.843830,0.815628,0.550257,1.084017
6,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,0.845307,0.933437,0.869421,0.869421,0.839278,0.651314,0.510938,1.554703
5,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,0.809167,0.924498,0.838805,0.838805,0.801757,0.563614,0.534512,1.862607
9,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,0.707132,0.797750,0.737373,0.737373,0.699572,0.532582,0.569544,3.419591
8,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,0.619576,0.887284,0.828206,0.828206,0.567419,0.346676,0.568074,1.270014
3,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,0.609209,0.825085,0.793454,0.793454,0.563147,0.404872,0.846265,2.340542


In [13]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")
out_dir = BASE / "boltz_outputs"

print("boltz_outputs exists:", out_dir.exists())
print("boltz_outputs 内容：")

for p in sorted(out_dir.iterdir()):
    print(p.name, "DIR" if p.is_dir() else "FILE", p.stat().st_size if p.is_file() else "")

boltz_outputs exists: True
boltz_outputs 内容：
pilot_10 DIR 
pilot_10_confidence_summary.xlsx FILE 4783
pilot_10_localrun DIR 
pilot_10_localrun.log FILE 16799
pilot_10_localrun_confidence_summary.xlsx FILE 6682


In [14]:
from pathlib import Path
import shutil

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

src = BASE / "boltz_outputs/pilot_10_localrun_confidence_summary.xlsx"
dst_dir = BASE / "boltz_outputs/pilot_10"
dst_dir.mkdir(parents=True, exist_ok=True)

dst = dst_dir / "pilot_10_confidence_summary.xlsx"

print("src exists:", src.exists(), src)

if not src.exists():
    # 兜底搜索
    candidates = list(BASE.rglob("*confidence_summary*.xlsx"))
    print("搜索到候选文件：")
    for c in candidates:
        print(c)
    if candidates:
        src = candidates[0]
        print("使用候选文件:", src)
    else:
        raise FileNotFoundError("没有找到 confidence summary xlsx")

shutil.copy2(src, dst)

print("✅ 已复制到：")
print(dst)

# 再列出 pilot_10 文件夹
print("\npilot_10 文件夹内容：")
for p in sorted(dst_dir.iterdir()):
    print(p.name, "DIR" if p.is_dir() else "FILE", p.stat().st_size if p.is_file() else "")

src exists: True /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/pilot_10_localrun_confidence_summary.xlsx
✅ 已复制到：
/content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/pilot_10/pilot_10_confidence_summary.xlsx

pilot_10 文件夹内容：
boltz_results_pilot_10_yaml DIR 
pilot_10_confidence_summary.xlsx FILE 6682


In [15]:
from pathlib import Path
import shutil

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

# 你当前 Boltz 输出根目录，按你截图结果是 pilot_10 或 localrun 中某一个
search_root = BASE / "boltz_outputs"

pdb_out = BASE / "boltz_outputs/pilot_10/pilot_10_pdb_models"
pdb_out.mkdir(parents=True, exist_ok=True)

pdb_files = sorted(search_root.rglob("*.pdb"))

print("找到 PDB 数量:", len(pdb_files))

for pdb in pdb_files:
    # 避免重复复制同名文件
    dst = pdb_out / pdb.name
    if pdb.resolve() != dst.resolve():
        shutil.copy2(pdb, dst)

print("✅ 已整理 PDB 到：")
print(pdb_out)

print("\nPDB 文件：")
for p in sorted(pdb_out.glob("*.pdb")):
    print(p.name)

找到 PDB 数量: 10
✅ 已整理 PDB 到：
/content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/pilot_10/pilot_10_pdb_models

PDB 文件：
NODE_11_length_154862_cov_66.368492_122__Heme__rank1_model_0.pdb
NODE_12_length_136290_cov_66.430554_166__Heme__rank1_model_0.pdb
NODE_1_length_436095_cov_65.793628_305__FMN__rank1_model_0.pdb
NODE_22_length_94753_cov_66.514481_84__FAD__rank1_model_0.pdb
NODE_25_length_76722_cov_66.604332_6__NAD___rank1_model_0.pdb
NODE_25_length_76722_cov_66.604332_89__FAD__rank1_model_0.pdb
NODE_25_length_76722_cov_66.604332_89__NADP___rank2_model_0.pdb
NODE_26_length_70757_cov_67.122722_55__CoA__rank1_model_0.pdb
NODE_6_length_231008_cov_66.606731_103__FAD__rank1_model_0.pdb
NODE_6_length_231008_cov_66.606731_16__FAD__rank1_model_0.pdb


In [16]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")
pdb_root = BASE / "boltz_outputs/pilot_10/pilot_10_pdb_models"

rows = []

for pdb in sorted(pdb_root.glob("*.pdb")):
    hetatm_lines = []
    ligand_resnames = set()

    with open(pdb, "r", errors="ignore") as f:
        for line in f:
            if line.startswith("HETATM"):
                hetatm_lines.append(line)
                if len(line) >= 20:
                    ligand_resnames.add(line[17:20].strip())

    rows.append({
        "PDB": str(pdb),
        "File": pdb.name,
        "HETATM_count": len(hetatm_lines),
        "Ligand_resnames": ";".join(sorted(ligand_resnames)),
        "Has_ligand": len(hetatm_lines) > 0
    })

df_lig = pd.DataFrame(rows)

out = BASE / "boltz_outputs/pilot_10/pilot_10_ligand_check.xlsx"
df_lig.to_excel(out, index=False)

print("✅ ligand 检查完成：")
print(out)

display(df_lig)

✅ ligand 检查完成：
/content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/pilot_10/pilot_10_ligand_check.xlsx


,PDB,File,HETATM_count,Ligand_resnames,Has_ligand
0,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,NODE_11_length_154862_cov_66.368492_122__Heme_...,43,LIG,True
1,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,NODE_12_length_136290_cov_66.430554_166__Heme_...,43,LIG,True
2,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,NODE_1_length_436095_cov_65.793628_305__FMN__r...,31,LIG,True
3,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,NODE_22_length_94753_cov_66.514481_84__FAD__ra...,53,LIG,True
4,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,NODE_25_length_76722_cov_66.604332_6__NAD___ra...,44,LIG,True
5,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,NODE_25_length_76722_cov_66.604332_89__FAD__ra...,53,LIG,True
6,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,NODE_25_length_76722_cov_66.604332_89__NADP___...,48,LIG,True
7,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,NODE_26_length_70757_cov_67.122722_55__CoA__ra...,48,LIG,True
8,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,NODE_6_length_231008_cov_66.606731_103__FAD__r...,53,LIG,True
9,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,NODE_6_length_231008_cov_66.606731_16__FAD__ra...,53,LIG,True


In [17]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

pdb_dir = BASE / "boltz_outputs/pilot_10/pilot_10_pdb_models"
boltz_conf = BASE / "boltz_outputs/pilot_10_localrun_confidence_summary.xlsx"

print("PDB dir exists:", pdb_dir.exists(), pdb_dir)
print("Boltz confidence exists:", boltz_conf.exists(), boltz_conf)

pdb_files = sorted(pdb_dir.glob("*.pdb"))
print("PDB 数量:", len(pdb_files))

for p in pdb_files:
    print(p.name)

assert len(pdb_files) > 0, "没有找到 Boltz holo PDB"

PDB dir exists: True /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/pilot_10/pilot_10_pdb_models
Boltz confidence exists: True /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/pilot_10_localrun_confidence_summary.xlsx
PDB 数量: 10
NODE_11_length_154862_cov_66.368492_122__Heme__rank1_model_0.pdb
NODE_12_length_136290_cov_66.430554_166__Heme__rank1_model_0.pdb
NODE_1_length_436095_cov_65.793628_305__FMN__rank1_model_0.pdb
NODE_22_length_94753_cov_66.514481_84__FAD__rank1_model_0.pdb
NODE_25_length_76722_cov_66.604332_6__NAD___rank1_model_0.pdb
NODE_25_length_76722_cov_66.604332_89__FAD__rank1_model_0.pdb
NODE_25_length_76722_cov_66.604332_89__NADP___rank2_model_0.pdb
NODE_26_length_70757_cov_67.122722_55__CoA__rank1_model_0.pdb
NODE_6_length_231008_cov_66.606731_103__FAD__rank1_model_0.pdb
NODE_6_length_231008_cov_66.606731_16__FAD__rank1_model_0.pdb


In [19]:
from pathlib import Path
import numpy as np
import pandas as pd
import math

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")
pdb_dir = BASE / "boltz_outputs/pilot_10/pilot_10_pdb_models"

out_dir = BASE / "consensus_holo_pocket/pilot_10"
out_dir.mkdir(parents=True, exist_ok=True)

def parse_pdb_atoms(pdb_path):
    protein_atoms = []
    ligand_atoms = []

    with open(pdb_path, "r", errors="ignore") as f:
        for line in f:
            if not (line.startswith("ATOM") or line.startswith("HETATM")):
                continue

            try:
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])
            except:
                continue

            atom = {
                "record": line[0:6].strip(),
                "atom_name": line[12:16].strip(),
                "resname": line[17:20].strip(),
                "chain": line[21].strip(),
                "resseq": line[22:26].strip(),
                "icode": line[26].strip(),
                "coord": np.array([x, y, z], dtype=float),
                "line": line.rstrip()
            }

            if line.startswith("HETATM"):
                ligand_atoms.append(atom)
            elif line.startswith("ATOM"):
                protein_atoms.append(atom)

    return protein_atoms, ligand_atoms

def residue_key(atom):
    return f"{atom['chain']}:{atom['resname']}{atom['resseq']}{atom['icode']}"

rows = []

for pdb in sorted(pdb_dir.glob("*.pdb")):
    protein_atoms, ligand_atoms = parse_pdb_atoms(pdb)

    if len(ligand_atoms) == 0:
        rows.append({
            "PDB_File": pdb.name,
            "Has_Ligand": False,
            "Ligand_Atom_Count": 0,
            "Ligand_Center_X": None,
            "Ligand_Center_Y": None,
            "Ligand_Center_Z": None,
            "Pocket_Residues_5A": "",
            "Pocket_Residues_8A": "",
            "N_Residues_5A": 0,
            "N_Residues_8A": 0,
        })
        continue

    lig_coords = np.array([a["coord"] for a in ligand_atoms])
    lig_center = lig_coords.mean(axis=0)

    res_5 = set()
    res_8 = set()

    for atom in protein_atoms:
        # 到 ligand 任一原子的最小距离，比到中心更合理
        min_dist = np.min(np.linalg.norm(lig_coords - atom["coord"], axis=1))

        if min_dist <= 5.0:
            res_5.add(residue_key(atom))
        if min_dist <= 8.0:
            res_8.add(residue_key(atom))

    rows.append({
        "PDB_File": pdb.name,
        "PDB_Path": str(pdb),
        "Has_Ligand": True,
        "Ligand_Atom_Count": len(ligand_atoms),
        "Ligand_Resnames": ";".join(sorted(set(a["resname"] for a in ligand_atoms))),
        "Ligand_Center_X": lig_center[0],
        "Ligand_Center_Y": lig_center[1],
        "Ligand_Center_Z": lig_center[2],
        "Pocket_Residues_5A": ";".join(sorted(res_5)),
        "Pocket_Residues_8A": ";".join(sorted(res_8)),
        "N_Residues_5A": len(res_5),
        "N_Residues_8A": len(res_8),
    })

df_lig_pocket = pd.DataFrame(rows)

out_xlsx = out_dir / "pilot_10_ligand_centered_pocket_residues.xlsx"
df_lig_pocket.to_excel(out_xlsx, index=False)

print("✅ ligand-centered pocket residues 已生成")
print(out_xlsx)

display(df_lig_pocket[[
    "PDB_File",
    "Has_Ligand",
    "Ligand_Atom_Count",
    "Ligand_Resnames",
    "N_Residues_5A",
    "N_Residues_8A"
]])

✅ ligand-centered pocket residues 已生成
/content/drive/MyDrive/Horizyn_Checkpoints/consensus_holo_pocket/pilot_10/pilot_10_ligand_centered_pocket_residues.xlsx


,PDB_File,Has_Ligand,Ligand_Atom_Count,Ligand_Resnames,N_Residues_5A,N_Residues_8A
0,NODE_11_length_154862_cov_66.368492_122__Heme_...,True,43,LIG,36,73
1,NODE_12_length_136290_cov_66.430554_166__Heme_...,True,43,LIG,30,71
2,NODE_1_length_436095_cov_65.793628_305__FMN__r...,True,31,LIG,17,39
3,NODE_22_length_94753_cov_66.514481_84__FAD__ra...,True,53,LIG,22,45
4,NODE_25_length_76722_cov_66.604332_6__NAD___ra...,True,44,LIG,22,45
5,NODE_25_length_76722_cov_66.604332_89__FAD__ra...,True,53,LIG,12,37
6,NODE_25_length_76722_cov_66.604332_89__NADP___...,True,48,LIG,14,37
7,NODE_26_length_70757_cov_67.122722_55__CoA__ra...,True,48,LIG,19,53
8,NODE_6_length_231008_cov_66.606731_103__FAD__r...,True,53,LIG,17,29
9,NODE_6_length_231008_cov_66.606731_16__FAD__ra...,True,53,LIG,6,17


In [20]:
from pathlib import Path
import requests
import tarfile
import shutil

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")
pdb_dir = BASE / "boltz_outputs/pilot_10/pilot_10_pdb_models"

# Java 17
!apt-get update -qq
!apt-get install -y openjdk-17-jre-headless > /dev/null
!java -version

# 下载 P2Rank
P2RANK_ROOT = Path("/content/p2rank")
if P2RANK_ROOT.exists():
    shutil.rmtree(P2RANK_ROOT)
P2RANK_ROOT.mkdir(parents=True, exist_ok=True)

url = "https://github.com/rdk/p2rank/releases/download/2.5/p2rank_2.5.tar.gz"
tar_path = Path("/content/p2rank_2.5.tar.gz")

if not tar_path.exists():
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    tar_path.write_bytes(r.content)

with tarfile.open(tar_path, "r:gz") as tar:
    tar.extractall(P2RANK_ROOT)

prank = list(P2RANK_ROOT.rglob("prank"))[0]
prank.chmod(0o755)

print("PRANK:", prank)

# 准备 dataset
p2rank_input_dir = BASE / "p2rank_holo_inputs/pilot_10_holo_pdb"
p2rank_input_dir.mkdir(parents=True, exist_ok=True)

for old in p2rank_input_dir.glob("*.pdb"):
    old.unlink()

for p in sorted(pdb_dir.glob("*.pdb")):
    shutil.copy2(p, p2rank_input_dir / p.name)

ds_path = BASE / "p2rank_holo_inputs/pilot_10_holo.ds"
with open(ds_path, "w") as f:
    for p in sorted(p2rank_input_dir.glob("*.pdb")):
        f.write(str(p) + "\n")

p2rank_out = BASE / "p2rank_holo_outputs/pilot_10_holo"
p2rank_out.mkdir(parents=True, exist_ok=True)

print("Dataset:", ds_path)
print("Output:", p2rank_out)

!{prank} predict \
    -f "{ds_path}" \
    -o "{p2rank_out}" \
    -c alphafold \
    -threads 2

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


/tmp/ipykernel_4094/2320881670.py:29: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(P2RANK_ROOT)


PRANK: /content/p2rank/p2rank_2.5/prank
Dataset: /content/drive/MyDrive/Horizyn_Checkpoints/p2rank_holo_inputs/pilot_10_holo.ds
Output: /content/drive/MyDrive/Horizyn_Checkpoints/p2rank_holo_outputs/pilot_10_holo
P2Rank 2.5
[INFO] Console - P2Rank 2.5

[INFO] Console - 
[INFO] Main - loading default config from [/content/p2rank/p2rank_2.5/config/default.groovy]
[INFO] Main - Looking for config in /content/alphafold
[INFO] Main - Looking for config in /content/alphafold.groovy
[INFO] Main - Looking for config in /content/p2rank/p2rank_2.5/config/alphafold
[INFO] Main - Looking for config in /content/p2rank/p2rank_2.5/config/alphafold.groovy
[INFO] Main - overriding default config with [/content/p2rank/p2rank_2.5/config/alphafold.groovy]
DIR: /content/p2rank/p2rank_2.5/config/../test_data
[INFO] Console - DIR: /content/p2rank/p2rank_2.5/config/../test_data
DIR2: /content/p2rank/p2rank_2.5/config/../test_data
[INFO] Console - DIR2: /content/p2rank/p2rank_2.5/config/../test_data
DIR: /cont

In [21]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")
p2rank_out = BASE / "p2rank_holo_outputs/pilot_10_holo"
pdb_dir = BASE / "p2rank_holo_inputs/pilot_10_holo_pdb"

prediction_files = sorted(p2rank_out.rglob("*_predictions.csv"))
print("P2Rank prediction CSV 数量:", len(prediction_files))

def ligand_center_from_pdb(pdb_path):
    coords = []
    with open(pdb_path, "r", errors="ignore") as f:
        for line in f:
            if line.startswith("HETATM"):
                try:
                    coords.append([
                        float(line[30:38]),
                        float(line[38:46]),
                        float(line[46:54])
                    ])
                except:
                    pass
    if not coords:
        return None
    return np.array(coords, dtype=float).mean(axis=0)

def find_center_columns(df):
    cols = list(df.columns)
    low = {c.lower(): c for c in cols}

    for trio in [
        ("center_x", "center_y", "center_z"),
        ("x", "y", "z")
    ]:
        if all(t in low for t in trio):
            return low[trio[0]], low[trio[1]], low[trio[2]]

    # P2Rank 常见列：center_x center_y center_z
    xcol = ycol = zcol = None
    for c in cols:
        cl = c.lower()
        if "center" in cl and "x" in cl:
            xcol = c
        if "center" in cl and "y" in cl:
            ycol = c
        if "center" in cl and "z" in cl:
            zcol = c

    return xcol, ycol, zcol

all_rows = []

for csv_file in prediction_files:
    try:
        df_p = pd.read_csv(csv_file)
    except:
        df_p = pd.read_csv(csv_file, sep="\t")

    stem = csv_file.name.replace("_predictions.csv", "")

    # 找对应 PDB
    matched_pdb = None
    for pdb in pdb_dir.glob("*.pdb"):
        if pdb.stem in stem or stem in pdb.stem:
            matched_pdb = pdb
            break

    if matched_pdb is None:
        continue

    lig_center = ligand_center_from_pdb(matched_pdb)

    xcol, ycol, zcol = find_center_columns(df_p)

    if lig_center is None or xcol is None:
        all_rows.append({
            "PDB_File": matched_pdb.name,
            "Num_P2Rank_Pockets": len(df_p),
            "Nearest_P2Rank_Distance_A": None,
            "P2Rank_Hit_Class": "Cannot_compute"
        })
        continue

    dists = []
    for _, r in df_p.iterrows():
        pc = np.array([float(r[xcol]), float(r[ycol]), float(r[zcol])])
        dists.append(float(np.linalg.norm(pc - lig_center)))

    df_p["Ligand_to_Pocket_Center_A"] = dists
    nearest = df_p.sort_values("Ligand_to_Pocket_Center_A").iloc[0]
    dist = float(nearest["Ligand_to_Pocket_Center_A"])

    if dist <= 6:
        hit = "Strong_<=6A"
    elif dist <= 10:
        hit = "Moderate_6_10A"
    elif dist <= 15:
        hit = "Weak_10_15A"
    else:
        hit = "Miss_>15A"

    score_col = None
    for c in ["score", "probability", "pocket_score"]:
        if c in df_p.columns:
            score_col = c
            break

    all_rows.append({
        "PDB_File": matched_pdb.name,
        "PDB_Path": str(matched_pdb),
        "Num_P2Rank_Pockets": len(df_p),
        "Nearest_P2Rank_Distance_A": round(dist, 3),
        "P2Rank_Hit_Class": hit,
        "Nearest_P2Rank_Score": nearest[score_col] if score_col else None,
        "P2Rank_CSV": str(csv_file)
    })

df_p2rank_hit = pd.DataFrame(all_rows)

out_xlsx = BASE / "p2rank_holo_outputs/pilot_10_p2rank_ligand_distance_summary.xlsx"
df_p2rank_hit.to_excel(out_xlsx, index=False)

print("✅ P2Rank ligand-distance summary 已生成")
print(out_xlsx)

print(df_p2rank_hit["P2Rank_Hit_Class"].value_counts(dropna=False))
display(df_p2rank_hit)

P2Rank prediction CSV 数量: 1
✅ P2Rank ligand-distance summary 已生成
/content/drive/MyDrive/Horizyn_Checkpoints/p2rank_holo_outputs/pilot_10_p2rank_ligand_distance_summary.xlsx


KeyError: 'P2Rank_Hit_Class'

In [22]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")
p2rank_out = BASE / "p2rank_holo_outputs/pilot_10_holo"

print("p2rank_out exists:", p2rank_out.exists(), p2rank_out)

print("\nP2Rank 输出目录前 100 个文件：")
count = 0
for p in p2rank_out.rglob("*"):
    print(p)
    count += 1
    if count >= 100:
        break

print("\nCSV 文件：")
csvs = sorted(p2rank_out.rglob("*.csv"))
for c in csvs:
    print(c)
    try:
        df_tmp = pd.read_csv(c)
    except:
        df_tmp = pd.read_csv(c, sep="\t")
    print("  shape:", df_tmp.shape)
    print("  columns:", df_tmp.columns.tolist())
    display(df_tmp.head())

p2rank_out exists: True /content/drive/MyDrive/Horizyn_Checkpoints/p2rank_holo_outputs/pilot_10_holo

P2Rank 输出目录前 100 个文件：
/content/drive/MyDrive/Horizyn_Checkpoints/p2rank_holo_outputs/pilot_10_holo/run.log
/content/drive/MyDrive/Horizyn_Checkpoints/p2rank_holo_outputs/pilot_10_holo/params.txt
/content/drive/MyDrive/Horizyn_Checkpoints/p2rank_holo_outputs/pilot_10_holo/visualizations
/content/drive/MyDrive/Horizyn_Checkpoints/p2rank_holo_outputs/pilot_10_holo/pilot_10_holo.ds_predictions.csv
/content/drive/MyDrive/Horizyn_Checkpoints/p2rank_holo_outputs/pilot_10_holo/pilot_10_holo.ds_residues.csv
/content/drive/MyDrive/Horizyn_Checkpoints/p2rank_holo_outputs/pilot_10_holo/visualizations/data
/content/drive/MyDrive/Horizyn_Checkpoints/p2rank_holo_outputs/pilot_10_holo/visualizations/pilot_10_holo.ds_pymol.pml
/content/drive/MyDrive/Horizyn_Checkpoints/p2rank_holo_outputs/pilot_10_holo/visualizations/pilot_10_holo.ds_chimerax.cxc
/content/drive/MyDrive/Horizyn_Checkpoints/p2rank_holo_o

,name,rank,score,probability,sas_points,surf_atoms,center_x,center_y,center_z,residue_ids,surf_atom_ids


/content/drive/MyDrive/Horizyn_Checkpoints/p2rank_holo_outputs/pilot_10_holo/pilot_10_holo.ds_residues.csv
  shape: (0, 7)
  columns: ['chain', ' residue_label', ' residue_name', ' score', ' zscore', ' probability', ' pocket']


,chain,residue_label,residue_name,score,zscore,probability,pocket


In [23]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")
log_path = BASE / "p2rank_holo_outputs/pilot_10_holo/run.log"

print("log exists:", log_path.exists(), log_path)

if log_path.exists():
    lines = log_path.read_text(errors="ignore").splitlines()
    print("\n最后 120 行日志：")
    for line in lines[-120:]:
        print(line)

log exists: True /content/drive/MyDrive/Horizyn_Checkpoints/p2rank_holo_outputs/pilot_10_holo/run.log

最后 120 行日志：
[INFO] Console - predicting pockets for proteins from dataset [pilot_10_holo.ds]
[INFO] PredictPocketsRoutine - outdir: /content/drive/MyDrive/Horizyn_Checkpoints/p2rank_holo_outputs/pilot_10_holo
[INFO] Model - Loading model from directory (v3 format): /content/p2rank/p2rank_2.5/models/alphafold
[INFO] FeatureSetup - enabledFeatures: [chem, volsite, protrusion, atom_table]
[INFO] Dataset - processing dataset [pilot_10_holo.ds] using 0 threads
[INFO] Dataset - 
------------------------------------------------------------------------------------------------------------------------
processing [pilot_10_holo.ds] (1/1)
------------------------------------------------------------------------------------------------------------------------

[INFO] Console - processing [pilot_10_holo.ds] (1/1)
[INFO] Protein - loading protein [/content/drive/MyDrive/Horizyn_Checkpoints/p2rank_hol

In [24]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")
pdb_dir = BASE / "boltz_outputs/pilot_10/pilot_10_pdb_models"

rows = []

for pdb in sorted(pdb_dir.glob("*.pdb")):
    atom_count = 0
    hetatm_count = 0
    chains = set()
    resnames = set()

    with open(pdb, "r", errors="ignore") as f:
        for line in f:
            if line.startswith("ATOM"):
                atom_count += 1
                chains.add(line[21].strip())
            elif line.startswith("HETATM"):
                hetatm_count += 1
                resnames.add(line[17:20].strip())

    rows.append({
        "PDB_File": pdb.name,
        "ATOM_count": atom_count,
        "HETATM_count": hetatm_count,
        "Protein_Chains": ";".join(sorted(chains)),
        "Ligand_Resnames": ";".join(sorted(resnames)),
        "PDB_Path": str(pdb)
    })

df_atom_check = pd.DataFrame(rows)

out = BASE / "p2rank_holo_outputs/pilot_10_pdb_atom_check.xlsx"
df_atom_check.to_excel(out, index=False)

print("✅ PDB 原子检查完成")
print(out)

display(df_atom_check)

✅ PDB 原子检查完成
/content/drive/MyDrive/Horizyn_Checkpoints/p2rank_holo_outputs/pilot_10_pdb_atom_check.xlsx


,PDB_File,ATOM_count,HETATM_count,Protein_Chains,Ligand_Resnames,PDB_Path
0,NODE_11_length_154862_cov_66.368492_122__Heme_...,3076,43,A,LIG,/content/drive/MyDrive/Horizyn_Checkpoints/bol...
1,NODE_12_length_136290_cov_66.430554_166__Heme_...,2424,43,A,LIG,/content/drive/MyDrive/Horizyn_Checkpoints/bol...
2,NODE_1_length_436095_cov_65.793628_305__FMN__r...,1997,31,A,LIG,/content/drive/MyDrive/Horizyn_Checkpoints/bol...
3,NODE_22_length_94753_cov_66.514481_84__FAD__ra...,762,53,A,LIG,/content/drive/MyDrive/Horizyn_Checkpoints/bol...
4,NODE_25_length_76722_cov_66.604332_6__NAD___ra...,1722,44,A,LIG,/content/drive/MyDrive/Horizyn_Checkpoints/bol...
5,NODE_25_length_76722_cov_66.604332_89__FAD__ra...,1078,53,A,LIG,/content/drive/MyDrive/Horizyn_Checkpoints/bol...
6,NODE_25_length_76722_cov_66.604332_89__NADP___...,1078,48,A,LIG,/content/drive/MyDrive/Horizyn_Checkpoints/bol...
7,NODE_26_length_70757_cov_67.122722_55__CoA__ra...,2781,48,A,LIG,/content/drive/MyDrive/Horizyn_Checkpoints/bol...
8,NODE_6_length_231008_cov_66.606731_103__FAD__r...,824,53,A,LIG,/content/drive/MyDrive/Horizyn_Checkpoints/bol...
9,NODE_6_length_231008_cov_66.606731_16__FAD__ra...,753,53,A,LIG,/content/drive/MyDrive/Horizyn_Checkpoints/bol...


In [25]:
from pathlib import Path
import shutil
import subprocess

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

pdb_dir = BASE / "boltz_outputs/pilot_10/pilot_10_pdb_models"
p2rank_single_out = BASE / "p2rank_holo_outputs/pilot_10_holo_single"

# 找 prank 可执行文件
prank_candidates = list(Path("/content/p2rank").rglob("prank"))
if not prank_candidates:
    raise FileNotFoundError("没有找到 prank，请先运行 P2Rank 安装 Cell。")

prank = prank_candidates[0]
prank.chmod(0o755)

# 清理旧结果
if p2rank_single_out.exists():
    shutil.rmtree(p2rank_single_out)
p2rank_single_out.mkdir(parents=True, exist_ok=True)

pdb_files = sorted(pdb_dir.glob("*.pdb"))

print("PDB 数量:", len(pdb_files))
print("PRANK:", prank)

for pdb in pdb_files:
    sample_out = p2rank_single_out / pdb.stem
    sample_out.mkdir(parents=True, exist_ok=True)

    print("\nRunning P2Rank:", pdb.name)

    cmd = [
        str(prank),
        "predict",
        "-f", str(pdb),
        "-o", str(sample_out),
        "-c", "alphafold",
        "-threads", "2"
    ]

    res = subprocess.run(cmd, capture_output=True, text=True)

    log_file = sample_out / "p2rank_run.log"
    log_file.write_text(res.stdout + "\n\nSTDERR:\n" + res.stderr)

    if res.returncode != 0:
        print("  ⚠️ failed:", pdb.name)
        print(res.stderr[-800:])
    else:
        print("  ✅ done")

print("\n完成。检查输出 CSV：")
csvs = sorted(p2rank_single_out.rglob("*.csv"))
print("CSV 数量:", len(csvs))

for c in csvs[:30]:
    print(c)

PDB 数量: 10
PRANK: /content/p2rank/p2rank_2.5/prank

Running P2Rank: NODE_11_length_154862_cov_66.368492_122__Heme__rank1_model_0.pdb
  ✅ done

Running P2Rank: NODE_12_length_136290_cov_66.430554_166__Heme__rank1_model_0.pdb
  ✅ done

Running P2Rank: NODE_1_length_436095_cov_65.793628_305__FMN__rank1_model_0.pdb
  ✅ done

Running P2Rank: NODE_22_length_94753_cov_66.514481_84__FAD__rank1_model_0.pdb
  ✅ done

Running P2Rank: NODE_25_length_76722_cov_66.604332_6__NAD___rank1_model_0.pdb
  ✅ done

Running P2Rank: NODE_25_length_76722_cov_66.604332_89__FAD__rank1_model_0.pdb
  ✅ done

Running P2Rank: NODE_25_length_76722_cov_66.604332_89__NADP___rank2_model_0.pdb
  ✅ done

Running P2Rank: NODE_26_length_70757_cov_67.122722_55__CoA__rank1_model_0.pdb
  ✅ done

Running P2Rank: NODE_6_length_231008_cov_66.606731_103__FAD__rank1_model_0.pdb
  ✅ done

Running P2Rank: NODE_6_length_231008_cov_66.606731_16__FAD__rank1_model_0.pdb
  ✅ done

完成。检查输出 CSV：
CSV 数量: 20
/content/drive/MyDrive/Horizyn_Che

In [26]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")
p2rank_single_out = BASE / "p2rank_holo_outputs/pilot_10_holo_single"

csvs = sorted(p2rank_single_out.rglob("*.csv"))

summary = []

for c in csvs:
    try:
        df_tmp = pd.read_csv(c)
    except:
        df_tmp = pd.read_csv(c, sep="\t")

    summary.append({
        "CSV": str(c),
        "File": c.name,
        "Rows": len(df_tmp),
        "Columns": ";".join(df_tmp.columns.tolist())
    })

df_csv_check = pd.DataFrame(summary)

out = BASE / "p2rank_holo_outputs/pilot_10_holo_single_csv_check.xlsx"
df_csv_check.to_excel(out, index=False)

print("✅ P2Rank CSV 检查完成")
print(out)

display(df_csv_check)

✅ P2Rank CSV 检查完成
/content/drive/MyDrive/Horizyn_Checkpoints/p2rank_holo_outputs/pilot_10_holo_single_csv_check.xlsx


,CSV,File,Rows,Columns
0,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...,NODE_11_length_154862_cov_66.368492_122__Heme_...,6,name ; rank; score; probability; sas_po...
1,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...,NODE_11_length_154862_cov_66.368492_122__Heme_...,393,chain; residue_label; residue_name; score; zsc...
2,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...,NODE_12_length_136290_cov_66.430554_166__Heme_...,8,name ; rank; score; probability; sas_po...
3,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...,NODE_12_length_136290_cov_66.430554_166__Heme_...,300,chain; residue_label; residue_name; score; zsc...
4,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...,NODE_1_length_436095_cov_65.793628_305__FMN__r...,2,name ; rank; score; probability; sas_po...
5,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...,NODE_1_length_436095_cov_65.793628_305__FMN__r...,260,chain; residue_label; residue_name; score; zsc...
6,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...,NODE_22_length_94753_cov_66.514481_84__FAD__ra...,0,name ; rank; score; probability; sas_po...
7,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...,NODE_22_length_94753_cov_66.514481_84__FAD__ra...,95,chain; residue_label; residue_name; score; zsc...
8,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...,NODE_25_length_76722_cov_66.604332_6__NAD___ra...,4,name ; rank; score; probability; sas_po...
9,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...,NODE_25_length_76722_cov_66.604332_6__NAD___ra...,226,chain; residue_label; residue_name; score; zsc...


In [27]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

p2rank_single_out = BASE / "p2rank_holo_outputs/pilot_10_holo_single"
pdb_dir = BASE / "boltz_outputs/pilot_10/pilot_10_pdb_models"

out_xlsx = BASE / "p2rank_holo_outputs/pilot_10_p2rank_ligand_distance_summary_single.xlsx"

def ligand_center_from_pdb(pdb_path):
    coords = []
    with open(pdb_path, "r", errors="ignore") as f:
        for line in f:
            if line.startswith("HETATM"):
                try:
                    coords.append([
                        float(line[30:38]),
                        float(line[38:46]),
                        float(line[46:54])
                    ])
                except:
                    pass

    if not coords:
        return None

    return np.array(coords, dtype=float).mean(axis=0)

def find_center_columns(df):
    cols = list(df.columns)
    low = {c.lower().strip(): c for c in cols}

    for trio in [
        ("center_x", "center_y", "center_z"),
        ("x", "y", "z"),
        ("cen_x", "cen_y", "cen_z")
    ]:
        if all(t in low for t in trio):
            return low[trio[0]], low[trio[1]], low[trio[2]]

    return None, None, None

rows = []

for pdb in sorted(pdb_dir.glob("*.pdb")):
    sample_dir = p2rank_single_out / pdb.stem
    lig_center = ligand_center_from_pdb(pdb)

    pred_csvs = sorted(sample_dir.rglob("*_predictions.csv"))
    if not pred_csvs:
        pred_csvs = sorted(sample_dir.rglob("*predictions*.csv"))

    if not pred_csvs:
        rows.append({
            "PDB_File": pdb.name,
            "PDB_Path": str(pdb),
            "Has_Ligand": lig_center is not None,
            "Num_P2Rank_Pockets": 0,
            "Nearest_P2Rank_Distance_A": None,
            "P2Rank_Hit_Class": "No_P2Rank_CSV"
        })
        continue

    try:
        df_p = pd.read_csv(pred_csvs[0])
    except:
        df_p = pd.read_csv(pred_csvs[0], sep="\t")

    if df_p.empty:
        rows.append({
            "PDB_File": pdb.name,
            "PDB_Path": str(pdb),
            "Has_Ligand": lig_center is not None,
            "Num_P2Rank_Pockets": 0,
            "Nearest_P2Rank_Distance_A": None,
            "P2Rank_Hit_Class": "No_P2Rank_pocket_detected",
            "P2Rank_CSV": str(pred_csvs[0])
        })
        continue

    xcol, ycol, zcol = find_center_columns(df_p)

    if lig_center is None or xcol is None:
        rows.append({
            "PDB_File": pdb.name,
            "PDB_Path": str(pdb),
            "Has_Ligand": lig_center is not None,
            "Num_P2Rank_Pockets": len(df_p),
            "Nearest_P2Rank_Distance_A": None,
            "P2Rank_Hit_Class": "Cannot_compute_distance",
            "P2Rank_CSV": str(pred_csvs[0])
        })
        continue

    dists = []

    for _, r in df_p.iterrows():
        pc = np.array([
            float(r[xcol]),
            float(r[ycol]),
            float(r[zcol])
        ])
        dists.append(float(np.linalg.norm(pc - lig_center)))

    df_p["Ligand_to_Pocket_Center_A"] = dists
    nearest = df_p.sort_values("Ligand_to_Pocket_Center_A").iloc[0]

    dist = float(nearest["Ligand_to_Pocket_Center_A"])

    if dist <= 6:
        hit = "Strong_<=6A"
    elif dist <= 10:
        hit = "Moderate_6_10A"
    elif dist <= 15:
        hit = "Weak_10_15A"
    else:
        hit = "Miss_>15A"

    score_col = None
    for c in ["score", "probability", "pocket_score", "prob"]:
        if c in df_p.columns:
            score_col = c
            break

    rank_col = None
    for c in ["rank", "Rank"]:
        if c in df_p.columns:
            rank_col = c
            break

    rows.append({
        "PDB_File": pdb.name,
        "PDB_Path": str(pdb),
        "Has_Ligand": True,
        "Num_P2Rank_Pockets": len(df_p),
        "Nearest_P2Rank_Distance_A": round(dist, 3),
        "P2Rank_Hit_Class": hit,
        "Nearest_P2Rank_Score": nearest[score_col] if score_col else None,
        "Nearest_P2Rank_Rank": nearest[rank_col] if rank_col else None,
        "P2Rank_CSV": str(pred_csvs[0])
    })

df_p2rank_hit = pd.DataFrame(rows)
df_p2rank_hit.to_excel(out_xlsx, index=False)

print("✅ P2Rank single-run ligand distance summary 已生成")
print(out_xlsx)

print("\nP2Rank_Hit_Class 统计:")
print(df_p2rank_hit["P2Rank_Hit_Class"].value_counts(dropna=False))

display(df_p2rank_hit)

✅ P2Rank single-run ligand distance summary 已生成
/content/drive/MyDrive/Horizyn_Checkpoints/p2rank_holo_outputs/pilot_10_p2rank_ligand_distance_summary_single.xlsx

P2Rank_Hit_Class 统计:
P2Rank_Hit_Class
Strong_<=6A                  5
Weak_10_15A                  2
Moderate_6_10A               2
No_P2Rank_pocket_detected    1
Name: count, dtype: int64


,PDB_File,PDB_Path,Has_Ligand,Num_P2Rank_Pockets,Nearest_P2Rank_Distance_A,P2Rank_Hit_Class,Nearest_P2Rank_Score,Nearest_P2Rank_Rank,P2Rank_CSV
0,NODE_11_length_154862_cov_66.368492_122__Heme_...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,True,6,3.819,Strong_<=6A,NaN,NaN,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...
1,NODE_12_length_136290_cov_66.430554_166__Heme_...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,True,8,1.831,Strong_<=6A,NaN,NaN,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...
2,NODE_1_length_436095_cov_65.793628_305__FMN__r...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,True,2,12.371,Weak_10_15A,NaN,NaN,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...
3,NODE_22_length_94753_cov_66.514481_84__FAD__ra...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,True,0,NaN,No_P2Rank_pocket_detected,NaN,NaN,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...
4,NODE_25_length_76722_cov_66.604332_6__NAD___ra...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,True,4,4.570,Strong_<=6A,NaN,NaN,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...
5,NODE_25_length_76722_cov_66.604332_89__FAD__ra...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,True,5,6.160,Moderate_6_10A,NaN,NaN,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...
6,NODE_25_length_76722_cov_66.604332_89__NADP___...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,True,5,5.553,Strong_<=6A,NaN,NaN,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...
7,NODE_26_length_70757_cov_67.122722_55__CoA__ra...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,True,3,8.900,Moderate_6_10A,NaN,NaN,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...
8,NODE_6_length_231008_cov_66.606731_103__FAD__r...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,True,2,3.809,Strong_<=6A,NaN,NaN,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...
9,NODE_6_length_231008_cov_66.606731_16__FAD__ra...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,True,2,12.698,Weak_10_15A,NaN,NaN,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...


In [28]:
# =========================================================
# Cell C4：运行 fpocket on Boltz holo PDB
# =========================================================

from pathlib import Path
import shutil

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")
pdb_dir = BASE / "boltz_outputs/pilot_10/pilot_10_pdb_models"

# 安装 fpocket
!apt-get update -qq
!apt-get install -y fpocket > /dev/null

!fpocket -h | head -n 20

# 输出目录
fpocket_base = BASE / "fpocket_holo_outputs/pilot_10_holo"
fpocket_base.mkdir(parents=True, exist_ok=True)

# 本地运行，避免 Google Drive 慢
local_fpocket = Path("/content/fpocket_pilot_10")
if local_fpocket.exists():
    shutil.rmtree(local_fpocket)
local_fpocket.mkdir(parents=True, exist_ok=True)

# 复制 PDB 到本地
for pdb in sorted(pdb_dir.glob("*.pdb")):
    shutil.copy2(pdb, local_fpocket / pdb.name)

# 逐个运行 fpocket
for pdb in sorted(local_fpocket.glob("*.pdb")):
    print("Running fpocket:", pdb.name)
    !cd "{local_fpocket}" && fpocket -f "{pdb.name}" > /dev/null

# 复制结果回云盘
if fpocket_base.exists():
    shutil.rmtree(fpocket_base)
shutil.copytree(local_fpocket, fpocket_base)

print("✅ fpocket 完成")
print(fpocket_base)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
E: Unable to locate package fpocket
/bin/bash: line 1: fpocket: command not found
Running fpocket: NODE_11_length_154862_cov_66.368492_122__Heme__rank1_model_0.pdb
/bin/bash: line 1: fpocket: command not found
Running fpocket: NODE_12_length_136290_cov_66.430554_166__Heme__rank1_model_0.pdb
/bin/bash: line 1: fpocket: command not found
Running fpocket: NODE_1_length_436095_cov_65.793628_305__FMN__rank1_model_0.pdb
/bin/bash: line 1: fpocket: command not found
Running fpocket: NODE_22_length_94753_cov_66.514481_84__FAD__rank1_model_0.pdb
/bin/bash: line 1: fpocket: command not found
Running fpocket: NODE_25_length_76722_cov_66.604332_6__NAD___rank1_model_0.pdb
/bin/bash: line 1: fpocket: command not found
Running fpocket: NODE_25_length_76722_cov_66.604332_89__FAD__rank1_model_0.pdb
/bin/bash: line 1:

In [29]:
# =========================================================
# Cell C5：计算 ligand–fpocket pocket center 距离
# =========================================================

from pathlib import Path
import numpy as np
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")
pdb_dir = BASE / "boltz_outputs/pilot_10/pilot_10_pdb_models"
fpocket_base = BASE / "fpocket_holo_outputs/pilot_10_holo"

def ligand_center_from_pdb(pdb_path):
    coords = []
    with open(pdb_path, "r", errors="ignore") as f:
        for line in f:
            if line.startswith("HETATM"):
                try:
                    coords.append([
                        float(line[30:38]),
                        float(line[38:46]),
                        float(line[46:54])
                    ])
                except:
                    pass
    if not coords:
        return None
    return np.array(coords, dtype=float).mean(axis=0)

def pocket_center_from_pocket_pdb(pocket_pdb):
    coords = []
    with open(pocket_pdb, "r", errors="ignore") as f:
        for line in f:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                try:
                    coords.append([
                        float(line[30:38]),
                        float(line[38:46]),
                        float(line[46:54])
                    ])
                except:
                    pass
    if not coords:
        return None
    return np.array(coords, dtype=float).mean(axis=0)

rows = []

for pdb in sorted(pdb_dir.glob("*.pdb")):
    lig_center = ligand_center_from_pdb(pdb)

    out_folder = fpocket_base / f"{pdb.stem}_out"
    pockets_dir = out_folder / "pockets"

    pocket_files = sorted(pockets_dir.glob("pocket*_atm.pdb")) if pockets_dir.exists() else []

    if lig_center is None:
        rows.append({
            "PDB_File": pdb.name,
            "Num_fpocket_Pockets": len(pocket_files),
            "Nearest_fpocket_Distance_A": None,
            "fpocket_Hit_Class": "No_ligand"
        })
        continue

    if not pocket_files:
        rows.append({
            "PDB_File": pdb.name,
            "Num_fpocket_Pockets": 0,
            "Nearest_fpocket_Distance_A": None,
            "fpocket_Hit_Class": "No_fpocket_pocket"
        })
        continue

    dist_rows = []

    for pocket in pocket_files:
        pc = pocket_center_from_pocket_pdb(pocket)
        if pc is None:
            continue

        dist = float(np.linalg.norm(pc - lig_center))
        dist_rows.append((pocket, dist))

    if not dist_rows:
        rows.append({
            "PDB_File": pdb.name,
            "Num_fpocket_Pockets": len(pocket_files),
            "Nearest_fpocket_Distance_A": None,
            "fpocket_Hit_Class": "Cannot_compute"
        })
        continue

    nearest_pocket, nearest_dist = sorted(dist_rows, key=lambda x: x[1])[0]

    if nearest_dist <= 6:
        hit = "Strong_<=6A"
    elif nearest_dist <= 10:
        hit = "Moderate_6_10A"
    elif nearest_dist <= 15:
        hit = "Weak_10_15A"
    else:
        hit = "Miss_>15A"

    rows.append({
        "PDB_File": pdb.name,
        "Num_fpocket_Pockets": len(pocket_files),
        "Nearest_fpocket_Distance_A": round(nearest_dist, 3),
        "Nearest_fpocket_Pocket": str(nearest_pocket),
        "fpocket_Hit_Class": hit
    })

df_fpocket_hit = pd.DataFrame(rows)

out_xlsx = BASE / "fpocket_holo_outputs/pilot_10_fpocket_ligand_distance_summary.xlsx"
df_fpocket_hit.to_excel(out_xlsx, index=False)

print("✅ fpocket ligand-distance summary 已生成")
print(out_xlsx)

print("\nfpocket_Hit_Class 统计:")
print(df_fpocket_hit["fpocket_Hit_Class"].value_counts(dropna=False))

display(df_fpocket_hit)

✅ fpocket ligand-distance summary 已生成
/content/drive/MyDrive/Horizyn_Checkpoints/fpocket_holo_outputs/pilot_10_fpocket_ligand_distance_summary.xlsx

fpocket_Hit_Class 统计:
fpocket_Hit_Class
No_fpocket_pocket    10
Name: count, dtype: int64


,PDB_File,Num_fpocket_Pockets,Nearest_fpocket_Distance_A,fpocket_Hit_Class
0,NODE_11_length_154862_cov_66.368492_122__Heme_...,0,None,No_fpocket_pocket
1,NODE_12_length_136290_cov_66.430554_166__Heme_...,0,None,No_fpocket_pocket
2,NODE_1_length_436095_cov_65.793628_305__FMN__r...,0,None,No_fpocket_pocket
3,NODE_22_length_94753_cov_66.514481_84__FAD__ra...,0,None,No_fpocket_pocket
4,NODE_25_length_76722_cov_66.604332_6__NAD___ra...,0,None,No_fpocket_pocket
5,NODE_25_length_76722_cov_66.604332_89__FAD__ra...,0,None,No_fpocket_pocket
6,NODE_25_length_76722_cov_66.604332_89__NADP___...,0,None,No_fpocket_pocket
7,NODE_26_length_70757_cov_67.122722_55__CoA__ra...,0,None,No_fpocket_pocket
8,NODE_6_length_231008_cov_66.606731_103__FAD__r...,0,None,No_fpocket_pocket
9,NODE_6_length_231008_cov_66.606731_16__FAD__ra...,0,None,No_fpocket_pocket


In [30]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")
fpocket_base = BASE / "fpocket_holo_outputs/pilot_10_holo"

print("fpocket_base exists:", fpocket_base.exists(), fpocket_base)

print("\nfpocket 输出目录前 150 个文件：")
count = 0
for p in fpocket_base.rglob("*"):
    print(p)
    count += 1
    if count >= 150:
        break

print("\n搜索 pocket 文件：")
for pattern in ["*_out", "pockets", "pocket*_atm.pdb", "*info.txt", "*.log"]:
    hits = list(fpocket_base.rglob(pattern))
    print(pattern, "=>", len(hits))
    for h in hits[:10]:
        print(" ", h)

fpocket_base exists: True /content/drive/MyDrive/Horizyn_Checkpoints/fpocket_holo_outputs/pilot_10_holo

fpocket 输出目录前 150 个文件：
/content/drive/MyDrive/Horizyn_Checkpoints/fpocket_holo_outputs/pilot_10_holo/NODE_25_length_76722_cov_66.604332_89__NADP___rank2_model_0.pdb
/content/drive/MyDrive/Horizyn_Checkpoints/fpocket_holo_outputs/pilot_10_holo/NODE_11_length_154862_cov_66.368492_122__Heme__rank1_model_0.pdb
/content/drive/MyDrive/Horizyn_Checkpoints/fpocket_holo_outputs/pilot_10_holo/NODE_12_length_136290_cov_66.430554_166__Heme__rank1_model_0.pdb
/content/drive/MyDrive/Horizyn_Checkpoints/fpocket_holo_outputs/pilot_10_holo/NODE_22_length_94753_cov_66.514481_84__FAD__rank1_model_0.pdb
/content/drive/MyDrive/Horizyn_Checkpoints/fpocket_holo_outputs/pilot_10_holo/NODE_1_length_436095_cov_65.793628_305__FMN__rank1_model_0.pdb
/content/drive/MyDrive/Horizyn_Checkpoints/fpocket_holo_outputs/pilot_10_holo/NODE_6_length_231008_cov_66.606731_16__FAD__rank1_model_0.pdb
/content/drive/MyDrive/

In [31]:
from pathlib import Path
import shutil
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

holo_pdb_dir = BASE / "boltz_outputs/pilot_10/pilot_10_pdb_models"
protein_only_dir = BASE / "fpocket_holo_inputs/pilot_10_protein_only_pdb"

protein_only_dir.mkdir(parents=True, exist_ok=True)

# 清空旧文件
for f in protein_only_dir.glob("*.pdb"):
    f.unlink()

rows = []

for pdb in sorted(holo_pdb_dir.glob("*.pdb")):
    out_pdb = protein_only_dir / pdb.name

    atom_count = 0
    hetatm_count = 0

    with open(pdb, "r", errors="ignore") as fin, open(out_pdb, "w") as fout:
        for line in fin:
            if line.startswith("ATOM"):
                fout.write(line)
                atom_count += 1
            elif line.startswith("HETATM"):
                hetatm_count += 1
            elif line.startswith("TER") or line.startswith("END"):
                fout.write(line)

    rows.append({
        "Original_PDB": str(pdb),
        "Protein_Only_PDB": str(out_pdb),
        "ATOM_count": atom_count,
        "Removed_HETATM_count": hetatm_count,
    })

df_protein_only = pd.DataFrame(rows)

out_xlsx = BASE / "fpocket_holo_inputs/pilot_10_protein_only_check.xlsx"
df_protein_only.to_excel(out_xlsx, index=False)

print("✅ protein-only PDB 已生成")
print(out_xlsx)

display(df_protein_only)

✅ protein-only PDB 已生成
/content/drive/MyDrive/Horizyn_Checkpoints/fpocket_holo_inputs/pilot_10_protein_only_check.xlsx


,Original_PDB,Protein_Only_PDB,ATOM_count,Removed_HETATM_count
0,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,3076,43
1,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,2424,43
2,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,1997,31
3,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,762,53
4,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,1722,44
5,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,1078,53
6,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,1078,48
7,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,2781,48
8,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,824,53
9,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,753,53


In [32]:
from pathlib import Path
import shutil
import subprocess

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

protein_only_dir = BASE / "fpocket_holo_inputs/pilot_10_protein_only_pdb"
fpocket_out_drive = BASE / "fpocket_holo_outputs/pilot_10_holo_protein_only"

local_run_dir = Path("/content/fpocket_pilot_10_protein_only")

# 安装 fpocket
!apt-get update -qq
!apt-get install -y fpocket > /dev/null

!which fpocket
!fpocket -h | head -n 20

# 清理本地目录
if local_run_dir.exists():
    shutil.rmtree(local_run_dir)
local_run_dir.mkdir(parents=True, exist_ok=True)

# 复制 protein-only PDB 到本地
for pdb in sorted(protein_only_dir.glob("*.pdb")):
    shutil.copy2(pdb, local_run_dir / pdb.name)

# 逐个运行 fpocket，保存每个日志
pdb_files = sorted(local_run_dir.glob("*.pdb"))

print("准备运行 fpocket 的 PDB 数量:", len(pdb_files))

for pdb in pdb_files:
    print("\nRunning fpocket:", pdb.name)

    cmd = ["fpocket", "-f", pdb.name]

    res = subprocess.run(
        cmd,
        cwd=str(local_run_dir),
        capture_output=True,
        text=True
    )

    log_path = local_run_dir / f"{pdb.stem}_fpocket.log"
    log_path.write_text(
        "STDOUT:\n" + res.stdout + "\n\nSTDERR:\n" + res.stderr,
        encoding="utf-8"
    )

    if res.returncode != 0:
        print("  ⚠️ fpocket failed")
        print(res.stderr[-800:])
    else:
        print("  ✅ done")

# 复制结果回云盘
if fpocket_out_drive.exists():
    shutil.rmtree(fpocket_out_drive)
shutil.copytree(local_run_dir, fpocket_out_drive)

print("\n✅ fpocket protein-only 运行完成")
print(fpocket_out_drive)

print("\n检查 pocket 文件数量：")
pocket_files = list(fpocket_out_drive.rglob("pocket*_atm.pdb"))
print("pocket*_atm.pdb:", len(pocket_files))
for p in pocket_files[:20]:
    print(p)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
E: Unable to locate package fpocket
/bin/bash: line 1: fpocket: command not found
准备运行 fpocket 的 PDB 数量: 10

Running fpocket: NODE_11_length_154862_cov_66.368492_122__Heme__rank1_model_0.pdb


FileNotFoundError: [Errno 2] No such file or directory: 'fpocket'

In [33]:
# =========================================================
# Cell fpocket-install：从源码安装 fpocket
# =========================================================

from pathlib import Path
import os
import subprocess
import shutil

# 1. 安装编译依赖
!apt-get update -qq
!apt-get install -y build-essential git libnetcdf-dev zlib1g-dev > /dev/null

# 2. 克隆并编译 fpocket
%cd /content

if Path("/content/fpocket").exists():
    !rm -rf /content/fpocket

!git clone https://github.com/Discngine/fpocket.git /content/fpocket

%cd /content/fpocket

# 有些版本 make test 可能慢，先只 make
!make clean || true
!make

# 3. 查找可执行文件
import glob
candidates = []
for pattern in [
    "/content/fpocket/bin/fpocket",
    "/content/fpocket/fpocket",
    "/content/fpocket/src/fpocket",
    "/content/fpocket/**/fpocket"
]:
    candidates.extend(glob.glob(pattern, recursive=True))

candidates = [Path(c) for c in candidates if Path(c).is_file()]

print("fpocket candidates:")
for c in candidates:
    print(c)

if not candidates:
    raise FileNotFoundError("源码编译后没有找到 fpocket 可执行文件，请查看 make 日志。")

fpocket_bin = candidates[0]
fpocket_bin.chmod(0o755)

# 4. 建立软链接到 /usr/local/bin
target = Path("/usr/local/bin/fpocket")
if target.exists() or target.is_symlink():
    target.unlink()

os.symlink(fpocket_bin, target)

print("✅ fpocket installed at:", fpocket_bin)
print("✅ symlink:", target)

!which fpocket
!fpocket -h | head -n 30

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
/content
Cloning into '/content/fpocket'...
remote: Enumerating objects: 11333, done.
remote: Counting objects: 100% (4284/4284), done.
remote: Compressing objects: 100% (981/981), done.
remote: Total 11333 (delta 3316), reused 4147 (delta 3294), pack-reused 7049 (from 1)
Receiving objects: 100% (11333/11333), 127.77 MiB | 13.32 MiB/s, done.
Resolving deltas: 100% (7543/7543), done.
Updating files: 100% (3895/3895), done.
/content/fpocket
rm -f src/qhull/src*.o
rm -f obj/*.o
rm -f bin/fpocket
rm -f bin/tpocket
rm -f bin/dpocket
rm -f bin/mdpocket
cd src/qhull && make clean && rm lib/libqhull*.a
make[1]: Entering directory '/content/fpocket/src/qhull'
rm -f src/*/*.o src/qhulltest/RoadTest.h.cpp build/*/*/*.o  build/*/*.o
rm -f src/*/*.obj build/*/*/*.obj build/*/*/*/*/*.obj build/*/*.obj 
rm -f bin/*

In [34]:
# =========================================================
# Cell fpocket-fix-2-retry：重跑 protein-only fpocket
# =========================================================

from pathlib import Path
import shutil
import subprocess

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

protein_only_dir = BASE / "fpocket_holo_inputs/pilot_10_protein_only_pdb"
fpocket_out_drive = BASE / "fpocket_holo_outputs/pilot_10_holo_protein_only"

local_run_dir = Path("/content/fpocket_pilot_10_protein_only")

FPOCKET_BIN = "/usr/local/bin/fpocket"

# 检查 fpocket 是否可用
if not Path(FPOCKET_BIN).exists():
    raise FileNotFoundError("没有找到 /usr/local/bin/fpocket，请先运行 fpocket-install。")

!{FPOCKET_BIN} -h | head -n 10

# 清理本地目录
if local_run_dir.exists():
    shutil.rmtree(local_run_dir)
local_run_dir.mkdir(parents=True, exist_ok=True)

# 复制 protein-only PDB 到本地
pdb_files = sorted(protein_only_dir.glob("*.pdb"))

print("protein-only PDB 数量:", len(pdb_files))

if len(pdb_files) == 0:
    raise FileNotFoundError(f"没有找到 protein-only PDB: {protein_only_dir}")

for pdb in pdb_files:
    shutil.copy2(pdb, local_run_dir / pdb.name)

# 逐个运行 fpocket，保存日志
local_pdbs = sorted(local_run_dir.glob("*.pdb"))

for pdb in local_pdbs:
    print("\nRunning fpocket:", pdb.name)

    cmd = [FPOCKET_BIN, "-f", pdb.name]

    res = subprocess.run(
        cmd,
        cwd=str(local_run_dir),
        capture_output=True,
        text=True
    )

    log_path = local_run_dir / f"{pdb.stem}_fpocket.log"
    log_path.write_text(
        "STDOUT:\n" + res.stdout + "\n\nSTDERR:\n" + res.stderr,
        encoding="utf-8"
    )

    if res.returncode != 0:
        print("  ⚠️ fpocket failed")
        print(res.stderr[-800:])
    else:
        print("  ✅ done")

# 复制结果回云盘
if fpocket_out_drive.exists():
    shutil.rmtree(fpocket_out_drive)

shutil.copytree(local_run_dir, fpocket_out_drive)

print("\n✅ fpocket protein-only 运行完成")
print(fpocket_out_drive)

# 检查 pocket 文件
pocket_files = list(fpocket_out_drive.rglob("pocket*_atm.pdb"))
info_files = list(fpocket_out_drive.rglob("*info.txt"))
out_dirs = list(fpocket_out_drive.rglob("*_out"))

print("\n输出检查：")
print("*_out dirs:", len(out_dirs))
print("pocket*_atm.pdb:", len(pocket_files))
print("*info.txt:", len(info_files))

for p in pocket_files[:20]:
    print(p)

***** POCKET HUNTING BEGINS ***** 
! Invalid pdb name given.

:||: fpocket 4.0 :||:
        
Mandatory parameters : 
	fpocket -f --file pdb or cif file                                      
	[ fpocket -F --fileList fileList ]                                  


protein-only PDB 数量: 10

Running fpocket: NODE_11_length_154862_cov_66.368492_122__Heme__rank1_model_0.pdb
  ✅ done

Running fpocket: NODE_12_length_136290_cov_66.430554_166__Heme__rank1_model_0.pdb
  ✅ done

Running fpocket: NODE_1_length_436095_cov_65.793628_305__FMN__rank1_model_0.pdb
  ✅ done

Running fpocket: NODE_22_length_94753_cov_66.514481_84__FAD__rank1_model_0.pdb
  ✅ done

Running fpocket: NODE_25_length_76722_cov_66.604332_6__NAD___rank1_model_0.pdb
  ✅ done

Running fpocket: NODE_25_length_76722_cov_66.604332_89__FAD__rank1_model_0.pdb
  ✅ done

Running fpocket: NODE_25_length_76722_cov_66.604332_89__NADP___rank2_model_0.pdb
  ✅ done

Running fpocket: NODE_26_length_70757_cov_67.122722_55__CoA__rank1_model_0.pdb
  

In [37]:
from pathlib import Path
import numpy as np
import pandas as pd
import re

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

holo_pdb_dir = BASE / "boltz_outputs/pilot_10/pilot_10_pdb_models"
fpocket_base = BASE / "fpocket_holo_outputs/pilot_10_holo_protein_only"

def ligand_center_from_holo_pdb(pdb_path):
    coords = []

    with open(pdb_path, "r", errors="ignore") as f:
        for line in f:
            if line.startswith("HETATM"):
                try:
                    coords.append([
                        float(line[30:38]),
                        float(line[38:46]),
                        float(line[46:54])
                    ])
                except:
                    pass

    if not coords:
        return None

    return np.array(coords, dtype=float).mean(axis=0)

def pocket_center_from_pocket_pdb(pocket_pdb):
    coords = []

    with open(pocket_pdb, "r", errors="ignore") as f:
        for line in f:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                try:
                    coords.append([
                        float(line[30:38]),
                        float(line[38:46]),
                        float(line[46:54])
                    ])
                except:
                    pass

    if not coords:
        return None

    return np.array(coords, dtype=float).mean(axis=0)

rows = []

for holo_pdb in sorted(holo_pdb_dir.glob("*.pdb")):
    lig_center = ligand_center_from_holo_pdb(holo_pdb)

    # fpocket 输出目录
    out_folder = fpocket_base / f"{holo_pdb.stem}_out"
    pockets_dir = out_folder / "pockets"

    # 兜底：如果标准路径不存在，就递归搜索名字包含该 stem 的 pocket 文件
    if pockets_dir.exists():
        pocket_files = sorted(pockets_dir.glob("pocket*_atm.pdb"))
    else:
        pocket_files = sorted(fpocket_base.rglob(f"{holo_pdb.stem}_out/pockets/pocket*_atm.pdb"))

    if lig_center is None:
        rows.append({
            "PDB_File": holo_pdb.name,
            "Num_fpocket_Pockets": len(pocket_files),
            "Nearest_fpocket_Distance_A": None,
            "fpocket_Hit_Class": "No_ligand"
        })
        continue

    if not pocket_files:
        rows.append({
            "PDB_File": holo_pdb.name,
            "Num_fpocket_Pockets": 0,
            "Nearest_fpocket_Distance_A": None,
            "fpocket_Hit_Class": "No_fpocket_pocket"
        })
        continue

    dist_rows = []

    for pocket in pocket_files:
        pc = pocket_center_from_pocket_pdb(pocket)
        if pc is None:
            continue

        dist = float(np.linalg.norm(pc - lig_center))
        dist_rows.append((pocket, dist))

    if not dist_rows:
        rows.append({
            "PDB_File": holo_pdb.name,
            "Num_fpocket_Pockets": len(pocket_files),
            "Nearest_fpocket_Distance_A": None,
            "fpocket_Hit_Class": "Cannot_compute"
        })
        continue

    nearest_pocket, nearest_dist = sorted(dist_rows, key=lambda x: x[1])[0]

    if nearest_dist <= 6:
        hit = "Strong_<=6A"
    elif nearest_dist <= 10:
        hit = "Moderate_6_10A"
    elif nearest_dist <= 15:
        hit = "Weak_10_15A"
    else:
        hit = "Miss_>15A"

    rows.append({
        "PDB_File": holo_pdb.name,
        "Num_fpocket_Pockets": len(pocket_files),
        "Nearest_fpocket_Distance_A": round(nearest_dist, 3),
        "Nearest_fpocket_Pocket": str(nearest_pocket),
        "fpocket_Hit_Class": hit
    })

df_fpocket_hit = pd.DataFrame(rows)

out_xlsx = BASE / "fpocket_holo_outputs/pilot_10_fpocket_ligand_distance_summary_protein_only.xlsx"
df_fpocket_hit.to_excel(out_xlsx, index=False)

print("✅ fpocket protein-only ligand-distance summary 已生成")
print(out_xlsx)

print("\nfpocket_Hit_Class 统计:")
print(df_fpocket_hit["fpocket_Hit_Class"].value_counts(dropna=False))

display(df_fpocket_hit)

✅ fpocket protein-only ligand-distance summary 已生成
/content/drive/MyDrive/Horizyn_Checkpoints/fpocket_holo_outputs/pilot_10_fpocket_ligand_distance_summary_protein_only.xlsx

fpocket_Hit_Class 统计:
fpocket_Hit_Class
Strong_<=6A       7
Moderate_6_10A    2
Miss_>15A         1
Name: count, dtype: int64


,PDB_File,Num_fpocket_Pockets,Nearest_fpocket_Distance_A,Nearest_fpocket_Pocket,fpocket_Hit_Class
0,NODE_11_length_154862_cov_66.368492_122__Heme_...,27,4.805,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,Strong_<=6A
1,NODE_12_length_136290_cov_66.430554_166__Heme_...,21,0.886,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,Strong_<=6A
2,NODE_1_length_436095_cov_65.793628_305__FMN__r...,14,3.180,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,Strong_<=6A
3,NODE_22_length_94753_cov_66.514481_84__FAD__ra...,6,3.420,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,Strong_<=6A
4,NODE_25_length_76722_cov_66.604332_6__NAD___ra...,11,3.903,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,Strong_<=6A
5,NODE_25_length_76722_cov_66.604332_89__FAD__ra...,6,6.011,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,Moderate_6_10A
6,NODE_25_length_76722_cov_66.604332_89__NADP___...,10,4.790,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,Strong_<=6A
7,NODE_26_length_70757_cov_67.122722_55__CoA__ra...,20,7.185,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,Moderate_6_10A
8,NODE_6_length_231008_cov_66.606731_103__FAD__r...,7,2.964,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,Strong_<=6A
9,NODE_6_length_231008_cov_66.606731_16__FAD__ra...,6,16.229,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,Miss_>15A


In [38]:
# =========================================================
# Cell C6-fixed：合并 ligand-centered + P2Rank + fpocket protein-only + Boltz confidence
# =========================================================

from pathlib import Path
import pandas as pd
import re
import numpy as np

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

# -----------------------------
# 1. 输入文件路径
# -----------------------------
lig_xlsx = BASE / "consensus_holo_pocket/pilot_10/pilot_10_ligand_centered_pocket_residues.xlsx"

p2rank_xlsx = BASE / "p2rank_holo_outputs/pilot_10_p2rank_ligand_distance_summary_single.xlsx"

# 注意：这里必须使用 protein-only fpocket 修复后的结果
fpocket_xlsx = BASE / "fpocket_holo_outputs/pilot_10_fpocket_ligand_distance_summary_protein_only.xlsx"

# Boltz confidence 可能有两个位置，自动选择存在的
boltz_conf_candidates = [
    BASE / "boltz_outputs/pilot_10_localrun_confidence_summary.xlsx",
    BASE / "boltz_outputs/pilot_10/pilot_10_confidence_summary.xlsx",
    BASE / "boltz_outputs/pilot_10_fixed_confidence_summary.xlsx",
]

boltz_conf = None
for p in boltz_conf_candidates:
    if p.exists():
        boltz_conf = p
        break

required = {
    "ligand-centered pocket": lig_xlsx,
    "P2Rank summary": p2rank_xlsx,
    "fpocket protein-only summary": fpocket_xlsx,
    "Boltz confidence": boltz_conf,
}

for name, path in required.items():
    print(name, "=>", path, "exists:", path.exists() if path is not None else False)

if boltz_conf is None:
    raise FileNotFoundError("没有找到 Boltz confidence summary 文件，请检查 boltz_outputs 目录。")

for name, path in required.items():
    if path is None or not path.exists():
        raise FileNotFoundError(f"缺少输入文件: {name} -> {path}")

# -----------------------------
# 2. 读取数据
# -----------------------------
df_lig = pd.read_excel(lig_xlsx)
df_p2 = pd.read_excel(p2rank_xlsx)
df_fp = pd.read_excel(fpocket_xlsx)
df_conf = pd.read_excel(boltz_conf)

print("\n输入表行数：")
print("ligand-centered:", len(df_lig))
print("P2Rank:", len(df_p2))
print("fpocket:", len(df_fp))
print("Boltz confidence:", len(df_conf))

# -----------------------------
# 3. 文件名标准化
# -----------------------------
def norm_file(x):
    x = str(x)
    x = x.split("/")[-1]
    x = x.replace(".pdb", "")
    x = x.replace(".cif", "")
    x = x.replace(".json", "")
    x = re.sub(r"[^A-Za-z0-9_.-]+", "_", x)
    return x

# ligand / P2Rank / fpocket 都有 PDB_File
df_lig["norm"] = df_lig["PDB_File"].apply(norm_file)
df_p2["norm"] = df_p2["PDB_File"].apply(norm_file)
df_fp["norm"] = df_fp["PDB_File"].apply(norm_file)

# Boltz confidence 里通常是 Best_Model_PDB
if "Best_Model_PDB" in df_conf.columns:
    df_conf["norm"] = df_conf["Best_Model_PDB"].apply(norm_file)
elif "Prediction_Dir" in df_conf.columns:
    df_conf["norm"] = df_conf["Prediction_Dir"].apply(norm_file)
else:
    raise ValueError("Boltz confidence 表里没有 Best_Model_PDB 或 Prediction_Dir 列。")

# -----------------------------
# 4. 合并表格
# -----------------------------
df = df_lig.merge(
    df_p2,
    on="norm",
    how="left",
    suffixes=("", "_p2rank")
)

df = df.merge(
    df_fp,
    on="norm",
    how="left",
    suffixes=("", "_fpocket")
)

df = df.merge(
    df_conf,
    on="norm",
    how="left",
    suffixes=("", "_boltz")
)

# -----------------------------
# 5. 检查是否有未匹配项
# -----------------------------
print("\n合并后行数:", len(df))

print("\n缺失匹配检查：")
print("Missing P2Rank_Hit_Class:", df["P2Rank_Hit_Class"].isna().sum() if "P2Rank_Hit_Class" in df.columns else "列不存在")
print("Missing fpocket_Hit_Class:", df["fpocket_Hit_Class"].isna().sum() if "fpocket_Hit_Class" in df.columns else "列不存在")
print("Missing confidence_score:", df["confidence_score"].isna().sum() if "confidence_score" in df.columns else "列不存在")

if "P2Rank_Hit_Class" not in df.columns:
    raise ValueError("合并后没有 P2Rank_Hit_Class，请检查 P2Rank summary 表。")

if "fpocket_Hit_Class" not in df.columns:
    raise ValueError("合并后没有 fpocket_Hit_Class，请检查 fpocket protein-only summary 表。")

if "confidence_score" not in df.columns:
    raise ValueError("合并后没有 confidence_score，请检查 Boltz confidence 表。")

# -----------------------------
# 6. 支持分数函数
# -----------------------------
def support_from_hit(hit):
    hit = str(hit)

    if hit.startswith("Strong"):
        return 1.0
    if hit.startswith("Moderate"):
        return 0.75
    if hit.startswith("Weak"):
        return 0.40
    if hit.startswith("Miss"):
        return 0.0
    if hit.startswith("No_"):
        return 0.0
    if hit == "nan":
        return 0.0

    return 0.0

def ligand_support(row):
    has_lig = row.get("Has_Ligand", False)

    if str(has_lig).lower() not in ["true", "1", "yes"]:
        return 0.0

    try:
        n8 = float(row.get("N_Residues_8A", 0))
    except:
        n8 = 0

    try:
        n5 = float(row.get("N_Residues_5A", 0))
    except:
        n5 = 0

    # 8 Å 内残基充足，说明 ligand 有稳定局部环境
    if n8 >= 10 and n5 >= 5:
        return 1.0
    if n8 >= 10:
        return 0.85
    if n8 >= 5:
        return 0.70
    if n8 > 0:
        return 0.40

    return 0.0

def boltz_support(row):
    try:
        c = float(row.get("confidence_score", 0))
    except:
        c = 0

    if c >= 0.85:
        return 1.0
    if c >= 0.70:
        return 0.80
    if c >= 0.60:
        return 0.60
    if c > 0:
        return 0.30

    return 0.0

# -----------------------------
# 7. 计算各证据支持分数
# -----------------------------
df["Ligand_Centered_Support"] = df.apply(ligand_support, axis=1)
df["P2Rank_Support"] = df["P2Rank_Hit_Class"].apply(support_from_hit)
df["fpocket_Support"] = df["fpocket_Hit_Class"].apply(support_from_hit)
df["Boltz_Confidence_Support"] = df.apply(boltz_support, axis=1)

# -----------------------------
# 8. Consensus holo-pocket score
# -----------------------------
# 权重解释：
# ligand-centered 是你最直接的 holo 证据；
# P2Rank 和 fpocket 是两个独立 pocket predictor；
# Boltz confidence 是结构整体质量证据。
df["Consensus_Holo_Pocket_Score"] = (
    0.35 * df["Ligand_Centered_Support"] +
    0.25 * df["P2Rank_Support"] +
    0.25 * df["fpocket_Support"] +
    0.15 * df["Boltz_Confidence_Support"]
).round(3)

def classify_consensus(score):
    if score >= 0.75:
        return "High_confidence_holo_pocket"
    if score >= 0.50:
        return "Probable_holo_pocket"
    if score >= 0.30:
        return "Uncertain_manual_check"
    return "Low_confidence_or_failed"

df["Consensus_Holo_Pocket_Class"] = df["Consensus_Holo_Pocket_Score"].apply(classify_consensus)

# -----------------------------
# 9. 决策列
# -----------------------------
def next_action(row):
    cls = row["Consensus_Holo_Pocket_Class"]
    p2 = str(row.get("P2Rank_Hit_Class", ""))
    fp = str(row.get("fpocket_Hit_Class", ""))

    if cls == "High_confidence_holo_pocket":
        return "Proceed_to_substrate_docking_and_EZSpecificity"
    if cls == "Probable_holo_pocket":
        return "Proceed_but_keep_manual_inspection"
    if cls == "Uncertain_manual_check":
        return "Manual_inspection_or_rerun_Boltz_multiseed"
    if p2.startswith("Miss") and fp.startswith("Miss"):
        return "Low_priority_or_alternative_cofactor_pose"
    return "Hold"

df["Recommended_Next_Action"] = df.apply(next_action, axis=1)

# -----------------------------
# 10. 输出结果
# -----------------------------
out_dir = BASE / "consensus_holo_pocket"
out_dir.mkdir(parents=True, exist_ok=True)

out_xlsx = out_dir / "pilot_10_consensus_holo_pocket_summary_fixed.xlsx"
out_csv = out_dir / "pilot_10_consensus_holo_pocket_summary_fixed.csv"

df.to_excel(out_xlsx, index=False)
df.to_csv(out_csv, index=False)

print("\n✅ consensus holo-pocket summary 已生成")
print(out_xlsx)
print(out_csv)

print("\nConsensus class 统计:")
print(df["Consensus_Holo_Pocket_Class"].value_counts(dropna=False))

print("\nRecommended next action 统计:")
print(df["Recommended_Next_Action"].value_counts(dropna=False))

display_cols = [
    "PDB_File",
    "confidence_score",
    "N_Residues_5A",
    "N_Residues_8A",
    "P2Rank_Hit_Class",
    "Nearest_P2Rank_Distance_A",
    "fpocket_Hit_Class",
    "Nearest_fpocket_Distance_A",
    "Ligand_Centered_Support",
    "P2Rank_Support",
    "fpocket_Support",
    "Boltz_Confidence_Support",
    "Consensus_Holo_Pocket_Score",
    "Consensus_Holo_Pocket_Class",
    "Recommended_Next_Action",
]

display_cols = [c for c in display_cols if c in df.columns]

display(
    df[display_cols].sort_values(
        "Consensus_Holo_Pocket_Score",
        ascending=False
    )
)

ligand-centered pocket => /content/drive/MyDrive/Horizyn_Checkpoints/consensus_holo_pocket/pilot_10/pilot_10_ligand_centered_pocket_residues.xlsx exists: True
P2Rank summary => /content/drive/MyDrive/Horizyn_Checkpoints/p2rank_holo_outputs/pilot_10_p2rank_ligand_distance_summary_single.xlsx exists: True
fpocket protein-only summary => /content/drive/MyDrive/Horizyn_Checkpoints/fpocket_holo_outputs/pilot_10_fpocket_ligand_distance_summary_protein_only.xlsx exists: True
Boltz confidence => /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/pilot_10_localrun_confidence_summary.xlsx exists: True

输入表行数：
ligand-centered: 10
P2Rank: 10
fpocket: 10
Boltz confidence: 10

合并后行数: 10

缺失匹配检查：
Missing P2Rank_Hit_Class: 0
Missing fpocket_Hit_Class: 0
Missing confidence_score: 0

✅ consensus holo-pocket summary 已生成
/content/drive/MyDrive/Horizyn_Checkpoints/consensus_holo_pocket/pilot_10_consensus_holo_pocket_summary_fixed.xlsx
/content/drive/MyDrive/Horizyn_Checkpoints/consensus_holo_pocket/p

,PDB_File,confidence_score,N_Residues_5A,N_Residues_8A,P2Rank_Hit_Class,Nearest_P2Rank_Distance_A,fpocket_Hit_Class,Nearest_fpocket_Distance_A,Ligand_Centered_Support,P2Rank_Support,fpocket_Support,Boltz_Confidence_Support,Consensus_Holo_Pocket_Score,Consensus_Holo_Pocket_Class,Recommended_Next_Action
0,NODE_11_length_154862_cov_66.368492_122__Heme_...,0.895598,36,73,Strong_<=6A,3.819,Strong_<=6A,4.805,1.0,1.00,1.00,1.0,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
1,NODE_12_length_136290_cov_66.430554_166__Heme_...,0.863788,30,71,Strong_<=6A,1.831,Strong_<=6A,0.886,1.0,1.00,1.00,1.0,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
4,NODE_25_length_76722_cov_66.604332_6__NAD___ra...,0.896801,22,45,Strong_<=6A,4.570,Strong_<=6A,3.903,1.0,1.00,1.00,1.0,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
6,NODE_25_length_76722_cov_66.604332_89__NADP___...,0.845307,14,37,Strong_<=6A,5.553,Strong_<=6A,4.790,1.0,1.00,1.00,0.8,0.970,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
8,NODE_6_length_231008_cov_66.606731_103__FAD__r...,0.619576,17,29,Strong_<=6A,3.809,Strong_<=6A,2.964,1.0,1.00,1.00,0.6,0.940,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
7,NODE_26_length_70757_cov_67.122722_55__CoA__ra...,0.926218,19,53,Moderate_6_10A,8.900,Moderate_6_10A,7.185,1.0,0.75,0.75,1.0,0.875,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
2,NODE_1_length_436095_cov_65.793628_305__FMN__r...,0.908389,17,39,Weak_10_15A,12.371,Strong_<=6A,3.180,1.0,0.40,1.00,1.0,0.850,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
5,NODE_25_length_76722_cov_66.604332_89__FAD__ra...,0.809167,12,37,Moderate_6_10A,6.160,Moderate_6_10A,6.011,1.0,0.75,0.75,0.8,0.845,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
3,NODE_22_length_94753_cov_66.514481_84__FAD__ra...,0.609209,22,45,No_P2Rank_pocket_detected,NaN,Strong_<=6A,3.420,1.0,0.00,1.00,0.6,0.690,Probable_holo_pocket,Proceed_but_keep_manual_inspection
9,NODE_6_length_231008_cov_66.606731_16__FAD__ra...,0.707132,6,17,Weak_10_15A,12.698,Miss_>15A,16.229,1.0,0.40,0.00,0.8,0.570,Probable_holo_pocket,Proceed_but_keep_manual_inspection


In [39]:
# =========================================================
# Step D1：把 Primary organic/heme YAML 分成批次
# =========================================================

from pathlib import Path
import pandas as pd
import shutil
import math

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

yaml_manifest = BASE / "boltz_inputs/primary_organic_yaml_manifest.xlsx"
batch_root = BASE / "boltz_inputs/primary_organic_batches"

batch_size = 20   # Colab T4/L4 建议 10–20；A100 可改 30–50

df_yaml = pd.read_excel(yaml_manifest)

df_ok = df_yaml[df_yaml["Status"] == "ok"].copy()
df_ok["Cofactor_Score"] = pd.to_numeric(df_ok["Cofactor_Score"], errors="coerce").fillna(0)
df_ok = df_ok.sort_values("Cofactor_Score", ascending=False).reset_index(drop=True)

print("可用于 Boltz 的 Primary organic/heme 任务数:", len(df_ok))

if batch_root.exists():
    shutil.rmtree(batch_root)
batch_root.mkdir(parents=True, exist_ok=True)

batch_rows = []

n_batches = math.ceil(len(df_ok) / batch_size)

for i in range(n_batches):
    batch_id = f"batch_{i+1:03d}"
    batch_dir = batch_root / batch_id
    batch_dir.mkdir(parents=True, exist_ok=True)

    sub = df_ok.iloc[i * batch_size : (i + 1) * batch_size].copy()

    for _, row in sub.iterrows():
        src = Path(row["YAML_Path"])
        dst = batch_dir / src.name
        shutil.copy2(src, dst)

        batch_rows.append({
            "Batch_ID": batch_id,
            "Batch_Dir": str(batch_dir),
            **row.to_dict()
        })

df_batches = pd.DataFrame(batch_rows)

batch_manifest = BASE / "boltz_inputs/primary_organic_batches_manifest.xlsx"
df_batches.to_excel(batch_manifest, index=False)

print("✅ batch YAML 已生成")
print("Batch 数量:", n_batches)
print("Batch manifest:", batch_manifest)

print("\n每个 batch 的任务数:")
print(df_batches["Batch_ID"].value_counts().sort_index())

display(df_batches.head(20))

可用于 Boltz 的 Primary organic/heme 任务数: 371
✅ batch YAML 已生成
Batch 数量: 19
Batch manifest: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_inputs/primary_organic_batches_manifest.xlsx

每个 batch 的任务数:
Batch_ID
batch_001    20
batch_002    20
batch_003    20
batch_004    20
batch_005    20
batch_006    20
batch_007    20
batch_008    20
batch_009    20
batch_010    20
batch_011    20
batch_012    20
batch_013    20
batch_014    20
batch_015    20
batch_016    20
batch_017    20
batch_018    20
batch_019    11
Name: count, dtype: int64


,Batch_ID,Batch_Dir,Task_ID,Enzyme_ID,Cofactor,Cofactor_Score,Cofactor_Type,CCD,YAML_Path,Status,Sequence_Length,Predicted_EC_4digit,Evidence_Sources
0,batch_001,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,NODE_26_length_70757_cov_67.122722_55__CoA__rank1,NODE_26_length_70757_cov_67.122722_55,CoA,0.9,organic_cofactor,COA,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,ok,382,2.3.1.16;3.7.1.9,EC-class-prior;KEGG;M-CSA;UniProt
1,batch_001,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,NODE_25_length_76722_cov_66.604332_6__NAD+__rank1,NODE_25_length_76722_cov_66.604332_6,NAD+,0.9,organic_cofactor,NAD,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,ok,226,1.5.1.34,EC-class-prior;KEGG;M-CSA;UniProt
2,batch_001,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,NODE_1_length_436095_cov_65.793628_305__FMN__r...,NODE_1_length_436095_cov_65.793628_305,FMN,0.9,organic_cofactor,FMN,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,ok,260,1.4.3.5,EC-class-prior;KEGG;M-CSA;UniProt
3,batch_001,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,NODE_6_length_231008_cov_66.606731_16__FAD__rank1,NODE_6_length_231008_cov_66.606731_16,FAD,0.9,organic_cofactor,FAD,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,ok,93,1.1.2.7;1.4.3.2,EC-class-prior;KEGG;M-CSA;UniProt
4,batch_001,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,NODE_25_length_76722_cov_66.604332_89__FAD__rank1,NODE_25_length_76722_cov_66.604332_89,FAD,0.9,organic_cofactor,FAD,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,ok,134,3.4.24.64;1.14.13.7,EC-class-prior;KEGG;M-CSA;UniProt
5,batch_001,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,NODE_11_length_154862_cov_66.368492_122__Heme_...,NODE_11_length_154862_cov_66.368492_122,Heme,0.9,heme,HEM,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,ok,393,1.14.15.35,EC-class-prior;KEGG;M-CSA;UniProt
6,batch_001,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,NODE_25_length_76722_cov_66.604332_89__NADP+__...,NODE_25_length_76722_cov_66.604332_89,NADP+,0.9,organic_cofactor,NAP,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,ok,134,3.4.24.64;1.14.13.7,EC-class-prior;KEGG;M-CSA;UniProt
7,batch_001,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,NODE_12_length_136290_cov_66.430554_166__Heme_...,NODE_12_length_136290_cov_66.430554_166,Heme,0.9,heme,HEM,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,ok,300,1.14.14.18,EC-class-prior;KEGG;M-CSA;UniProt
8,batch_001,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,NODE_6_length_231008_cov_66.606731_103__FAD__r...,NODE_6_length_231008_cov_66.606731_103,FAD,0.9,organic_cofactor,FAD,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,ok,107,1.4.3.2,EC-class-prior;KEGG;M-CSA;UniProt
9,batch_001,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,NODE_22_length_94753_cov_66.514481_84__FAD__rank1,NODE_22_length_94753_cov_66.514481_84,FAD,0.9,organic_cofactor,FAD,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,ok,95,1.4.3.2,EC-class-prior;KEGG;M-CSA;UniProt


In [40]:
# =========================================================
# Step D2：运行一个正式 batch 的 Boltz
# =========================================================

from pathlib import Path
import shutil

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

batch_id = "batch_001"

drive_batch_dir = BASE / f"boltz_inputs/primary_organic_batches/{batch_id}"
drive_out_dir = BASE / f"boltz_outputs/primary_organic/{batch_id}"

local_input = Path(f"/content/boltz_{batch_id}_input")
local_out = Path(f"/content/boltz_{batch_id}_out")
local_cache = Path("/content/boltz_cache")

for d in [local_input, local_out]:
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

local_cache.mkdir(parents=True, exist_ok=True)

# 只复制 YAML
yaml_files = sorted(drive_batch_dir.glob("*.yaml"))
print("YAML 数量:", len(yaml_files))

for y in yaml_files:
    shutil.copy2(y, local_input / y.name)

# 运行 Boltz
!BOLTZ_CACHE="/content/boltz_cache" boltz predict "{local_input}" \
    --out_dir "{local_out}" \
    --use_msa_server \
    --output_format pdb \
    --override \
    --no_kernels \
    2>&1 | tee "/content/boltz_batch_001_run.log"

# 复制结果回云盘
if drive_out_dir.exists():
    shutil.rmtree(drive_out_dir)
shutil.copytree(local_out, drive_out_dir)

log_dst = BASE / f"boltz_outputs/primary_organic/{batch_id}_run.log"
log_dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2("/content/boltz_batch_001_run.log", log_dst)

print("✅ batch 输出已复制到:")
print(drive_out_dir)
print(log_dst)

YAML 数量: 20
MSA server enabled: https://api.colabfold.com
MSA server authentication: no credentials provided
Checking input data.
Processing 20 inputs with 12 threads.
  0%|          | 0/20 [00:00<?, ?it/s]Generating MSA for /content/boltz_batch_001_input/NODE_35_length_45322_cov_66.706421_4__FAD__rank1.yaml with 1 protein entities.
Calling MSA server for target NODE_35_length_45322_cov_66.706421_4__FAD__rank1 with 1 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server
SUBMIT:   0%|          | 0/150 [elapsed: 00:00 remaining: ?]Generating MSA for /content/boltz_batch_001_input/NODE_6_length_231008_cov_66.606731_16__FAD__rank1.yaml with 1 protein entities.
Calling MSA server for target NODE_6_length_231008_cov_66.606731_16__FAD__rank1 with 1 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server
SUBMIT:   0%|          | 0/150 [elapsed: 00:00 remaining

In [41]:
# =========================================================
# Step D3：汇总 batch Boltz confidence
# =========================================================

from pathlib import Path
import json
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

batch_id = "batch_001"
out_dir = BASE / f"boltz_outputs/primary_organic/{batch_id}"

rows = []

all_conf = sorted(out_dir.rglob("confidence*.json"))
all_pdb = sorted(out_dir.rglob("*.pdb"))

print("confidence json:", len(all_conf))
print("PDB files:", len(all_pdb))

candidate_dirs = sorted(set([p.parent for p in all_conf + all_pdb]))

for pred_dir in candidate_dirs:
    conf_files = sorted(pred_dir.glob("confidence*.json"))
    pdb_files = sorted(pred_dir.glob("*.pdb"))

    conf = {}
    if conf_files:
        with open(conf_files[0]) as f:
            conf = json.load(f)

    rows.append({
        "Batch_ID": batch_id,
        "Prediction_Dir": str(pred_dir),
        "Best_Model_PDB": str(pdb_files[0]) if pdb_files else "",
        "Confidence_JSON": str(conf_files[0]) if conf_files else "",
        "confidence_score": conf.get("confidence_score", None),
        "ptm": conf.get("ptm", None),
        "iptm": conf.get("iptm", None),
        "ligand_iptm": conf.get("ligand_iptm", None),
        "complex_plddt": conf.get("complex_plddt", None),
        "complex_iplddt": conf.get("complex_iplddt", None),
    })

df_conf = pd.DataFrame(rows)

summary_path = BASE / f"boltz_outputs/primary_organic/{batch_id}_confidence_summary.xlsx"
df_conf.to_excel(summary_path, index=False)

print("✅ Boltz batch confidence summary:")
print(summary_path)

display(df_conf.sort_values("confidence_score", ascending=False, na_position="last").head(20))

confidence json: 20
PDB files: 20
✅ Boltz batch confidence summary:
/content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_001_confidence_summary.xlsx


,Batch_ID,Prediction_Dir,Best_Model_PDB,Confidence_JSON,confidence_score,ptm,iptm,ligand_iptm,complex_plddt,complex_iplddt
16,batch_001,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,0.960423,0.971282,0.978958,0.978958,0.955789,0.952924
3,batch_001,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,0.950495,0.954807,0.942268,0.942268,0.952551,0.894769
4,batch_001,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,0.948654,0.961198,0.977233,0.977233,0.941510,0.908611
15,batch_001,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,0.928467,0.971071,0.910810,0.910810,0.932881,0.844463
6,batch_001,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,0.899817,0.948141,0.851039,0.851039,0.912012,0.720130
14,batch_001,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,0.896448,0.940719,0.890297,0.890297,0.897986,0.799891
11,batch_001,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,0.893232,0.956049,0.870999,0.870999,0.898791,0.719056
0,batch_001,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,0.891467,0.930531,0.987269,0.987269,0.867517,0.904033
9,batch_001,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,0.886403,0.947000,0.902499,0.902499,0.882379,0.762778
8,batch_001,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,0.878825,0.926012,0.836649,0.836649,0.889369,0.742206


In [42]:
# =========================================================
# Step D4：整理 batch Boltz PDB + ligand 检查
# =========================================================

from pathlib import Path
import shutil
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")
batch_id = "batch_001"

boltz_out = BASE / f"boltz_outputs/primary_organic/{batch_id}"
pdb_collect_dir = BASE / f"boltz_outputs/primary_organic/{batch_id}_pdb_models"
pdb_collect_dir.mkdir(parents=True, exist_ok=True)

# 清空旧 PDB
for f in pdb_collect_dir.glob("*.pdb"):
    f.unlink()

pdb_files = sorted(boltz_out.rglob("*.pdb"))

print("找到 Boltz PDB 数量:", len(pdb_files))

for pdb in pdb_files:
    shutil.copy2(pdb, pdb_collect_dir / pdb.name)

rows = []

for pdb in sorted(pdb_collect_dir.glob("*.pdb")):
    atom_count = 0
    hetatm_count = 0
    ligand_resnames = set()

    with open(pdb, "r", errors="ignore") as f:
        for line in f:
            if line.startswith("ATOM"):
                atom_count += 1
            elif line.startswith("HETATM"):
                hetatm_count += 1
                ligand_resnames.add(line[17:20].strip())

    rows.append({
        "Batch_ID": batch_id,
        "PDB_File": pdb.name,
        "PDB_Path": str(pdb),
        "ATOM_count": atom_count,
        "HETATM_count": hetatm_count,
        "Ligand_Resnames": ";".join(sorted(ligand_resnames)),
        "Has_Ligand": hetatm_count > 0,
        "Has_Protein": atom_count > 0,
    })

df_lig_check = pd.DataFrame(rows)

out_xlsx = BASE / f"boltz_outputs/primary_organic/{batch_id}_ligand_check.xlsx"
df_lig_check.to_excel(out_xlsx, index=False)

print("✅ batch ligand check 完成")
print(out_xlsx)

print("\n统计：")
print(df_lig_check[["Has_Protein", "Has_Ligand"]].value_counts(dropna=False))

display(df_lig_check)

找到 Boltz PDB 数量: 20
✅ batch ligand check 完成
/content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_001_ligand_check.xlsx

统计：
Has_Protein  Has_Ligand
True         True          20
Name: count, dtype: int64


,Batch_ID,PDB_File,PDB_Path,ATOM_count,HETATM_count,Ligand_Resnames,Has_Ligand,Has_Protein
0,batch_001,NODE_11_length_154862_cov_66.368492_122__Heme_...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,3076,43,LIG,True,True
1,batch_001,NODE_12_length_136290_cov_66.430554_166__Heme_...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,2424,43,LIG,True,True
2,batch_001,NODE_13_length_131311_cov_66.505852_95__FMN__r...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,1570,31,LIG,True,True
3,batch_001,NODE_1_length_436095_cov_65.793628_206__CoA__r...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,1699,48,LIG,True,True
4,batch_001,NODE_1_length_436095_cov_65.793628_253__FMN__r...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,2470,31,LIG,True,True
5,batch_001,NODE_1_length_436095_cov_65.793628_272__Heme__...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,1324,43,LIG,True,True
6,batch_001,NODE_1_length_436095_cov_65.793628_305__FMN__r...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,1997,31,LIG,True,True
7,batch_001,NODE_1_length_436095_cov_65.793628_391__FAD__r...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,1040,53,LIG,True,True
8,batch_001,NODE_1_length_436095_cov_65.793628_391__NADP__...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,1040,48,LIG,True,True
9,batch_001,NODE_21_length_97532_cov_65.343769_69__NAD___r...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,1379,44,LIG,True,True


In [43]:
# =========================================================
# Step D5：提取 batch ligand-centered pocket residues
# =========================================================

from pathlib import Path
import numpy as np
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")
batch_id = "batch_001"

pdb_dir = BASE / f"boltz_outputs/primary_organic/{batch_id}_pdb_models"
out_dir = BASE / f"consensus_holo_pocket/primary_organic/{batch_id}"
out_dir.mkdir(parents=True, exist_ok=True)

def parse_pdb_atoms(pdb_path):
    protein_atoms = []
    ligand_atoms = []

    with open(pdb_path, "r", errors="ignore") as f:
        for line in f:
            if not (line.startswith("ATOM") or line.startswith("HETATM")):
                continue

            try:
                coord = np.array([
                    float(line[30:38]),
                    float(line[38:46]),
                    float(line[46:54])
                ], dtype=float)
            except:
                continue

            atom = {
                "record": line[0:6].strip(),
                "atom_name": line[12:16].strip(),
                "resname": line[17:20].strip(),
                "chain": line[21].strip(),
                "resseq": line[22:26].strip(),
                "icode": line[26].strip(),
                "coord": coord,
            }

            if line.startswith("ATOM"):
                protein_atoms.append(atom)
            elif line.startswith("HETATM"):
                ligand_atoms.append(atom)

    return protein_atoms, ligand_atoms

def residue_key(atom):
    return f"{atom['chain']}:{atom['resname']}{atom['resseq']}{atom['icode']}"

rows = []

for pdb in sorted(pdb_dir.glob("*.pdb")):
    protein_atoms, ligand_atoms = parse_pdb_atoms(pdb)

    if len(ligand_atoms) == 0:
        rows.append({
            "Batch_ID": batch_id,
            "PDB_File": pdb.name,
            "PDB_Path": str(pdb),
            "Has_Ligand": False,
            "Ligand_Atom_Count": 0,
            "Ligand_Resnames": "",
            "N_Residues_5A": 0,
            "N_Residues_8A": 0,
            "Pocket_Residues_5A": "",
            "Pocket_Residues_8A": "",
        })
        continue

    lig_coords = np.array([a["coord"] for a in ligand_atoms], dtype=float)
    lig_center = lig_coords.mean(axis=0)

    res_5 = set()
    res_8 = set()

    for atom in protein_atoms:
        min_dist = np.min(np.linalg.norm(lig_coords - atom["coord"], axis=1))

        if min_dist <= 5.0:
            res_5.add(residue_key(atom))
        if min_dist <= 8.0:
            res_8.add(residue_key(atom))

    rows.append({
        "Batch_ID": batch_id,
        "PDB_File": pdb.name,
        "PDB_Path": str(pdb),
        "Has_Ligand": True,
        "Ligand_Atom_Count": len(ligand_atoms),
        "Ligand_Resnames": ";".join(sorted(set(a["resname"] for a in ligand_atoms))),
        "Ligand_Center_X": lig_center[0],
        "Ligand_Center_Y": lig_center[1],
        "Ligand_Center_Z": lig_center[2],
        "N_Residues_5A": len(res_5),
        "N_Residues_8A": len(res_8),
        "Pocket_Residues_5A": ";".join(sorted(res_5)),
        "Pocket_Residues_8A": ";".join(sorted(res_8)),
    })

df_lig_pocket = pd.DataFrame(rows)

out_xlsx = out_dir / f"{batch_id}_ligand_centered_pocket_residues.xlsx"
df_lig_pocket.to_excel(out_xlsx, index=False)

print("✅ ligand-centered pocket residues 完成")
print(out_xlsx)

display(df_lig_pocket[[
    "PDB_File",
    "Has_Ligand",
    "Ligand_Atom_Count",
    "N_Residues_5A",
    "N_Residues_8A"
]])

✅ ligand-centered pocket residues 完成
/content/drive/MyDrive/Horizyn_Checkpoints/consensus_holo_pocket/primary_organic/batch_001/batch_001_ligand_centered_pocket_residues.xlsx


,PDB_File,Has_Ligand,Ligand_Atom_Count,N_Residues_5A,N_Residues_8A
0,NODE_11_length_154862_cov_66.368492_122__Heme_...,True,43,37,70
1,NODE_12_length_136290_cov_66.430554_166__Heme_...,True,43,29,68
2,NODE_13_length_131311_cov_66.505852_95__FMN__r...,True,31,19,49
3,NODE_1_length_436095_cov_65.793628_206__CoA__r...,True,48,26,47
4,NODE_1_length_436095_cov_65.793628_253__FMN__r...,True,31,23,58
5,NODE_1_length_436095_cov_65.793628_272__Heme__...,True,43,9,29
6,NODE_1_length_436095_cov_65.793628_305__FMN__r...,True,31,18,37
7,NODE_1_length_436095_cov_65.793628_391__FAD__r...,True,53,12,27
8,NODE_1_length_436095_cov_65.793628_391__NADP__...,True,48,10,26
9,NODE_21_length_97532_cov_65.343769_69__NAD___r...,True,44,28,68


In [44]:
# =========================================================
# Step D6：逐个 PDB 运行 P2Rank
# =========================================================

from pathlib import Path
import shutil
import subprocess

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")
batch_id = "batch_001"

pdb_dir = BASE / f"boltz_outputs/primary_organic/{batch_id}_pdb_models"
p2rank_out = BASE / f"p2rank_holo_outputs/primary_organic/{batch_id}_single"

# 找 prank
prank_candidates = list(Path("/content/p2rank").rglob("prank"))
if not prank_candidates:
    raise FileNotFoundError("没有找到 prank，请先安装 P2Rank。")

prank = prank_candidates[0]
prank.chmod(0o755)

if p2rank_out.exists():
    shutil.rmtree(p2rank_out)
p2rank_out.mkdir(parents=True, exist_ok=True)

pdb_files = sorted(pdb_dir.glob("*.pdb"))

print("PDB 数量:", len(pdb_files))
print("PRANK:", prank)

for pdb in pdb_files:
    sample_out = p2rank_out / pdb.stem
    sample_out.mkdir(parents=True, exist_ok=True)

    print("Running P2Rank:", pdb.name)

    cmd = [
        str(prank),
        "predict",
        "-f", str(pdb),
        "-o", str(sample_out),
        "-c", "alphafold",
        "-threads", "2"
    ]

    res = subprocess.run(cmd, capture_output=True, text=True)

    log_file = sample_out / "p2rank_run.log"
    log_file.write_text(res.stdout + "\n\nSTDERR:\n" + res.stderr)

    if res.returncode != 0:
        print("  ⚠️ failed:", pdb.name)
        print(res.stderr[-800:])
    else:
        print("  ✅ done")

print("✅ P2Rank batch 完成")

PDB 数量: 20
PRANK: /content/p2rank/p2rank_2.5/prank
Running P2Rank: NODE_11_length_154862_cov_66.368492_122__Heme__rank1_model_0.pdb
  ✅ done
Running P2Rank: NODE_12_length_136290_cov_66.430554_166__Heme__rank1_model_0.pdb
  ✅ done
Running P2Rank: NODE_13_length_131311_cov_66.505852_95__FMN__rank1_model_0.pdb
  ✅ done
Running P2Rank: NODE_1_length_436095_cov_65.793628_206__CoA__rank1_model_0.pdb
  ✅ done
Running P2Rank: NODE_1_length_436095_cov_65.793628_253__FMN__rank1_model_0.pdb
  ✅ done
Running P2Rank: NODE_1_length_436095_cov_65.793628_272__Heme__rank1_model_0.pdb
  ✅ done
Running P2Rank: NODE_1_length_436095_cov_65.793628_305__FMN__rank1_model_0.pdb
  ✅ done
Running P2Rank: NODE_1_length_436095_cov_65.793628_391__FAD__rank1_model_0.pdb
  ✅ done
Running P2Rank: NODE_1_length_436095_cov_65.793628_391__NADP___rank2_model_0.pdb
  ✅ done
Running P2Rank: NODE_21_length_97532_cov_65.343769_69__NAD___rank1_model_0.pdb
  ✅ done
Running P2Rank: NODE_22_length_94753_cov_66.514481_84__FAD__ra

In [45]:
# =========================================================
# Step D7：计算 ligand 到 P2Rank pocket center 距离
# =========================================================

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")
batch_id = "batch_001"

pdb_dir = BASE / f"boltz_outputs/primary_organic/{batch_id}_pdb_models"
p2rank_out = BASE / f"p2rank_holo_outputs/primary_organic/{batch_id}_single"

def ligand_center_from_pdb(pdb_path):
    coords = []
    with open(pdb_path, "r", errors="ignore") as f:
        for line in f:
            if line.startswith("HETATM"):
                try:
                    coords.append([
                        float(line[30:38]),
                        float(line[38:46]),
                        float(line[46:54])
                    ])
                except:
                    pass
    if not coords:
        return None
    return np.array(coords, dtype=float).mean(axis=0)

def find_center_columns(df):
    cols = list(df.columns)
    low = {c.lower().strip(): c for c in cols}

    for trio in [
        ("center_x", "center_y", "center_z"),
        ("x", "y", "z"),
        ("cen_x", "cen_y", "cen_z")
    ]:
        if all(t in low for t in trio):
            return low[trio[0]], low[trio[1]], low[trio[2]]

    return None, None, None

rows = []

for pdb in sorted(pdb_dir.glob("*.pdb")):
    sample_dir = p2rank_out / pdb.stem
    lig_center = ligand_center_from_pdb(pdb)

    pred_csvs = sorted(sample_dir.rglob("*_predictions.csv"))
    if not pred_csvs:
        pred_csvs = sorted(sample_dir.rglob("*predictions*.csv"))

    if not pred_csvs:
        rows.append({
            "Batch_ID": batch_id,
            "PDB_File": pdb.name,
            "PDB_Path": str(pdb),
            "Has_Ligand": lig_center is not None,
            "Num_P2Rank_Pockets": 0,
            "Nearest_P2Rank_Distance_A": None,
            "P2Rank_Hit_Class": "No_P2Rank_CSV"
        })
        continue

    try:
        df_p = pd.read_csv(pred_csvs[0])
    except:
        df_p = pd.read_csv(pred_csvs[0], sep="\t")

    if df_p.empty:
        rows.append({
            "Batch_ID": batch_id,
            "PDB_File": pdb.name,
            "PDB_Path": str(pdb),
            "Has_Ligand": lig_center is not None,
            "Num_P2Rank_Pockets": 0,
            "Nearest_P2Rank_Distance_A": None,
            "P2Rank_Hit_Class": "No_P2Rank_pocket_detected",
            "P2Rank_CSV": str(pred_csvs[0])
        })
        continue

    xcol, ycol, zcol = find_center_columns(df_p)

    if lig_center is None or xcol is None:
        rows.append({
            "Batch_ID": batch_id,
            "PDB_File": pdb.name,
            "PDB_Path": str(pdb),
            "Has_Ligand": lig_center is not None,
            "Num_P2Rank_Pockets": len(df_p),
            "Nearest_P2Rank_Distance_A": None,
            "P2Rank_Hit_Class": "Cannot_compute_distance",
            "P2Rank_CSV": str(pred_csvs[0])
        })
        continue

    dists = []
    for _, r in df_p.iterrows():
        pc = np.array([float(r[xcol]), float(r[ycol]), float(r[zcol])])
        dists.append(float(np.linalg.norm(pc - lig_center)))

    df_p["Ligand_to_Pocket_Center_A"] = dists
    nearest = df_p.sort_values("Ligand_to_Pocket_Center_A").iloc[0]
    dist = float(nearest["Ligand_to_Pocket_Center_A"])

    if dist <= 6:
        hit = "Strong_<=6A"
    elif dist <= 10:
        hit = "Moderate_6_10A"
    elif dist <= 15:
        hit = "Weak_10_15A"
    else:
        hit = "Miss_>15A"

    score_col = None
    for c in ["score", "probability", "pocket_score", "prob"]:
        if c in df_p.columns:
            score_col = c
            break

    rows.append({
        "Batch_ID": batch_id,
        "PDB_File": pdb.name,
        "PDB_Path": str(pdb),
        "Has_Ligand": True,
        "Num_P2Rank_Pockets": len(df_p),
        "Nearest_P2Rank_Distance_A": round(dist, 3),
        "P2Rank_Hit_Class": hit,
        "Nearest_P2Rank_Score": nearest[score_col] if score_col else None,
        "P2Rank_CSV": str(pred_csvs[0])
    })

df_p2rank_hit = pd.DataFrame(rows)

out_xlsx = BASE / f"p2rank_holo_outputs/primary_organic/{batch_id}_p2rank_ligand_distance_summary.xlsx"
out_xlsx.parent.mkdir(parents=True, exist_ok=True)
df_p2rank_hit.to_excel(out_xlsx, index=False)

print("✅ P2Rank ligand-distance summary 完成")
print(out_xlsx)

print(df_p2rank_hit["P2Rank_Hit_Class"].value_counts(dropna=False))
display(df_p2rank_hit)

✅ P2Rank ligand-distance summary 完成
/content/drive/MyDrive/Horizyn_Checkpoints/p2rank_holo_outputs/primary_organic/batch_001_p2rank_ligand_distance_summary.xlsx
P2Rank_Hit_Class
Strong_<=6A                  10
No_P2Rank_pocket_detected     4
Moderate_6_10A                3
Weak_10_15A                   2
Miss_>15A                     1
Name: count, dtype: int64


,Batch_ID,PDB_File,PDB_Path,Has_Ligand,Num_P2Rank_Pockets,Nearest_P2Rank_Distance_A,P2Rank_Hit_Class,Nearest_P2Rank_Score,P2Rank_CSV
0,batch_001,NODE_11_length_154862_cov_66.368492_122__Heme_...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,True,6,1.446,Strong_<=6A,NaN,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...
1,batch_001,NODE_12_length_136290_cov_66.430554_166__Heme_...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,True,8,0.685,Strong_<=6A,NaN,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...
2,batch_001,NODE_13_length_131311_cov_66.505852_95__FMN__r...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,True,4,1.695,Strong_<=6A,NaN,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...
3,batch_001,NODE_1_length_436095_cov_65.793628_206__CoA__r...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,True,1,14.684,Weak_10_15A,NaN,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...
4,batch_001,NODE_1_length_436095_cov_65.793628_253__FMN__r...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,True,2,5.024,Strong_<=6A,NaN,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...
5,batch_001,NODE_1_length_436095_cov_65.793628_272__Heme__...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,True,3,20.930,Miss_>15A,NaN,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...
6,batch_001,NODE_1_length_436095_cov_65.793628_305__FMN__r...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,True,4,10.449,Weak_10_15A,NaN,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...
7,batch_001,NODE_1_length_436095_cov_65.793628_391__FAD__r...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,True,0,NaN,No_P2Rank_pocket_detected,NaN,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...
8,batch_001,NODE_1_length_436095_cov_65.793628_391__NADP__...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,True,0,NaN,No_P2Rank_pocket_detected,NaN,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...
9,batch_001,NODE_21_length_97532_cov_65.343769_69__NAD___r...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,True,1,2.109,Strong_<=6A,NaN,/content/drive/MyDrive/Horizyn_Checkpoints/p2r...


In [46]:
# =========================================================
# Step D8：protein-only fpocket
# =========================================================

from pathlib import Path
import shutil
import subprocess
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")
batch_id = "batch_001"

holo_pdb_dir = BASE / f"boltz_outputs/primary_organic/{batch_id}_pdb_models"
protein_only_dir = BASE / f"fpocket_holo_inputs/primary_organic/{batch_id}_protein_only_pdb"
fpocket_out_drive = BASE / f"fpocket_holo_outputs/primary_organic/{batch_id}_protein_only"

protein_only_dir.mkdir(parents=True, exist_ok=True)

for f in protein_only_dir.glob("*.pdb"):
    f.unlink()

# 生成 protein-only PDB
for pdb in sorted(holo_pdb_dir.glob("*.pdb")):
    out_pdb = protein_only_dir / pdb.name

    with open(pdb, "r", errors="ignore") as fin, open(out_pdb, "w") as fout:
        for line in fin:
            if line.startswith("ATOM") or line.startswith("TER") or line.startswith("END"):
                fout.write(line)

# 检查 fpocket
FPOCKET_BIN = "/usr/local/bin/fpocket"
if not Path(FPOCKET_BIN).exists():
    raise FileNotFoundError("没有找到 /usr/local/bin/fpocket，请先源码安装 fpocket。")

local_run_dir = Path(f"/content/fpocket_{batch_id}_protein_only")

if local_run_dir.exists():
    shutil.rmtree(local_run_dir)
local_run_dir.mkdir(parents=True, exist_ok=True)

for pdb in sorted(protein_only_dir.glob("*.pdb")):
    shutil.copy2(pdb, local_run_dir / pdb.name)

for pdb in sorted(local_run_dir.glob("*.pdb")):
    print("Running fpocket:", pdb.name)

    cmd = [FPOCKET_BIN, "-f", pdb.name]

    res = subprocess.run(
        cmd,
        cwd=str(local_run_dir),
        capture_output=True,
        text=True
    )

    log_path = local_run_dir / f"{pdb.stem}_fpocket.log"
    log_path.write_text(
        "STDOUT:\n" + res.stdout + "\n\nSTDERR:\n" + res.stderr,
        encoding="utf-8"
    )

    if res.returncode != 0:
        print("  ⚠️ failed")
        print(res.stderr[-800:])
    else:
        print("  ✅ done")

if fpocket_out_drive.exists():
    shutil.rmtree(fpocket_out_drive)

shutil.copytree(local_run_dir, fpocket_out_drive)

print("✅ fpocket batch 完成")
print(fpocket_out_drive)

print("pocket 文件数量:", len(list(fpocket_out_drive.rglob("pocket*_atm.pdb"))))

Running fpocket: NODE_11_length_154862_cov_66.368492_122__Heme__rank1_model_0.pdb
  ✅ done
Running fpocket: NODE_12_length_136290_cov_66.430554_166__Heme__rank1_model_0.pdb
  ✅ done
Running fpocket: NODE_13_length_131311_cov_66.505852_95__FMN__rank1_model_0.pdb
  ✅ done
Running fpocket: NODE_1_length_436095_cov_65.793628_206__CoA__rank1_model_0.pdb
  ✅ done
Running fpocket: NODE_1_length_436095_cov_65.793628_253__FMN__rank1_model_0.pdb
  ✅ done
Running fpocket: NODE_1_length_436095_cov_65.793628_272__Heme__rank1_model_0.pdb
  ✅ done
Running fpocket: NODE_1_length_436095_cov_65.793628_305__FMN__rank1_model_0.pdb
  ✅ done
Running fpocket: NODE_1_length_436095_cov_65.793628_391__FAD__rank1_model_0.pdb
  ✅ done
Running fpocket: NODE_1_length_436095_cov_65.793628_391__NADP___rank2_model_0.pdb
  ✅ done
Running fpocket: NODE_21_length_97532_cov_65.343769_69__NAD___rank1_model_0.pdb
  ✅ done
Running fpocket: NODE_22_length_94753_cov_66.514481_84__FAD__rank1_model_0.pdb
  ✅ done
Running fpocket

In [47]:
# =========================================================
# Step D9：计算 ligand 到 fpocket pocket 距离
# =========================================================

from pathlib import Path
import numpy as np
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")
batch_id = "batch_001"

holo_pdb_dir = BASE / f"boltz_outputs/primary_organic/{batch_id}_pdb_models"
fpocket_base = BASE / f"fpocket_holo_outputs/primary_organic/{batch_id}_protein_only"

def ligand_center_from_holo_pdb(pdb_path):
    coords = []
    with open(pdb_path, "r", errors="ignore") as f:
        for line in f:
            if line.startswith("HETATM"):
                try:
                    coords.append([
                        float(line[30:38]),
                        float(line[38:46]),
                        float(line[46:54])
                    ])
                except:
                    pass
    if not coords:
        return None
    return np.array(coords, dtype=float).mean(axis=0)

def pocket_center_from_pocket_pdb(pocket_pdb):
    coords = []
    with open(pocket_pdb, "r", errors="ignore") as f:
        for line in f:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                try:
                    coords.append([
                        float(line[30:38]),
                        float(line[38:46]),
                        float(line[46:54])
                    ])
                except:
                    pass
    if not coords:
        return None
    return np.array(coords, dtype=float).mean(axis=0)

rows = []

for holo_pdb in sorted(holo_pdb_dir.glob("*.pdb")):
    lig_center = ligand_center_from_holo_pdb(holo_pdb)

    out_folder = fpocket_base / f"{holo_pdb.stem}_out"
    pockets_dir = out_folder / "pockets"

    pocket_files = sorted(pockets_dir.glob("pocket*_atm.pdb")) if pockets_dir.exists() else []

    if lig_center is None:
        rows.append({
            "Batch_ID": batch_id,
            "PDB_File": holo_pdb.name,
            "Num_fpocket_Pockets": len(pocket_files),
            "Nearest_fpocket_Distance_A": None,
            "fpocket_Hit_Class": "No_ligand"
        })
        continue

    if not pocket_files:
        rows.append({
            "Batch_ID": batch_id,
            "PDB_File": holo_pdb.name,
            "Num_fpocket_Pockets": 0,
            "Nearest_fpocket_Distance_A": None,
            "fpocket_Hit_Class": "No_fpocket_pocket"
        })
        continue

    dist_rows = []

    for pocket in pocket_files:
        pc = pocket_center_from_pocket_pdb(pocket)
        if pc is None:
            continue

        dist = float(np.linalg.norm(pc - lig_center))
        dist_rows.append((pocket, dist))

    if not dist_rows:
        rows.append({
            "Batch_ID": batch_id,
            "PDB_File": holo_pdb.name,
            "Num_fpocket_Pockets": len(pocket_files),
            "Nearest_fpocket_Distance_A": None,
            "fpocket_Hit_Class": "Cannot_compute"
        })
        continue

    nearest_pocket, nearest_dist = sorted(dist_rows, key=lambda x: x[1])[0]

    if nearest_dist <= 6:
        hit = "Strong_<=6A"
    elif nearest_dist <= 10:
        hit = "Moderate_6_10A"
    elif nearest_dist <= 15:
        hit = "Weak_10_15A"
    else:
        hit = "Miss_>15A"

    rows.append({
        "Batch_ID": batch_id,
        "PDB_File": holo_pdb.name,
        "Num_fpocket_Pockets": len(pocket_files),
        "Nearest_fpocket_Distance_A": round(nearest_dist, 3),
        "Nearest_fpocket_Pocket": str(nearest_pocket),
        "fpocket_Hit_Class": hit
    })

df_fpocket_hit = pd.DataFrame(rows)

out_xlsx = BASE / f"fpocket_holo_outputs/primary_organic/{batch_id}_fpocket_ligand_distance_summary.xlsx"
out_xlsx.parent.mkdir(parents=True, exist_ok=True)
df_fpocket_hit.to_excel(out_xlsx, index=False)

print("✅ fpocket ligand-distance summary 完成")
print(out_xlsx)

print(df_fpocket_hit["fpocket_Hit_Class"].value_counts(dropna=False))
display(df_fpocket_hit)

✅ fpocket ligand-distance summary 完成
/content/drive/MyDrive/Horizyn_Checkpoints/fpocket_holo_outputs/primary_organic/batch_001_fpocket_ligand_distance_summary.xlsx
fpocket_Hit_Class
Strong_<=6A       8
Moderate_6_10A    7
Weak_10_15A       3
Miss_>15A         2
Name: count, dtype: int64


,Batch_ID,PDB_File,Num_fpocket_Pockets,Nearest_fpocket_Distance_A,Nearest_fpocket_Pocket,fpocket_Hit_Class
0,batch_001,NODE_11_length_154862_cov_66.368492_122__Heme_...,26,1.262,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,Strong_<=6A
1,batch_001,NODE_12_length_136290_cov_66.430554_166__Heme_...,20,0.678,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,Strong_<=6A
2,batch_001,NODE_13_length_131311_cov_66.505852_95__FMN__r...,10,1.052,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,Strong_<=6A
3,batch_001,NODE_1_length_436095_cov_65.793628_206__CoA__r...,9,10.512,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,Weak_10_15A
4,batch_001,NODE_1_length_436095_cov_65.793628_253__FMN__r...,15,5.923,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,Strong_<=6A
5,batch_001,NODE_1_length_436095_cov_65.793628_272__Heme__...,13,9.061,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,Moderate_6_10A
6,batch_001,NODE_1_length_436095_cov_65.793628_305__FMN__r...,14,9.300,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,Moderate_6_10A
7,batch_001,NODE_1_length_436095_cov_65.793628_391__FAD__r...,3,18.122,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,Miss_>15A
8,batch_001,NODE_1_length_436095_cov_65.793628_391__NADP__...,1,21.367,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,Miss_>15A
9,batch_001,NODE_21_length_97532_cov_65.343769_69__NAD___r...,15,6.331,/content/drive/MyDrive/Horizyn_Checkpoints/fpo...,Moderate_6_10A


In [48]:
# =========================================================
# Step D10：合并生成 batch consensus holo-pocket summary
# =========================================================

from pathlib import Path
import pandas as pd
import re

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")
batch_id = "batch_001"

lig_xlsx = BASE / f"consensus_holo_pocket/primary_organic/{batch_id}/{batch_id}_ligand_centered_pocket_residues.xlsx"
p2rank_xlsx = BASE / f"p2rank_holo_outputs/primary_organic/{batch_id}_p2rank_ligand_distance_summary.xlsx"
fpocket_xlsx = BASE / f"fpocket_holo_outputs/primary_organic/{batch_id}_fpocket_ligand_distance_summary.xlsx"
boltz_conf = BASE / f"boltz_outputs/primary_organic/{batch_id}_confidence_summary.xlsx"

for p in [lig_xlsx, p2rank_xlsx, fpocket_xlsx, boltz_conf]:
    print(p, p.exists())

df_lig = pd.read_excel(lig_xlsx)
df_p2 = pd.read_excel(p2rank_xlsx)
df_fp = pd.read_excel(fpocket_xlsx)
df_conf = pd.read_excel(boltz_conf)

def norm_file(x):
    x = str(x).split("/")[-1]
    x = x.replace(".pdb", "")
    x = re.sub(r"[^A-Za-z0-9_.-]+", "_", x)
    return x

df_lig["norm"] = df_lig["PDB_File"].apply(norm_file)
df_p2["norm"] = df_p2["PDB_File"].apply(norm_file)
df_fp["norm"] = df_fp["PDB_File"].apply(norm_file)
df_conf["norm"] = df_conf["Best_Model_PDB"].apply(norm_file)

df = df_lig.merge(df_p2, on="norm", how="left", suffixes=("", "_p2rank"))
df = df.merge(df_fp, on="norm", how="left", suffixes=("", "_fpocket"))
df = df.merge(df_conf, on="norm", how="left", suffixes=("", "_boltz"))

def support_from_hit(hit):
    hit = str(hit)
    if hit.startswith("Strong"):
        return 1.0
    if hit.startswith("Moderate"):
        return 0.75
    if hit.startswith("Weak"):
        return 0.40
    return 0.0

def ligand_support(row):
    if str(row.get("Has_Ligand", False)).lower() not in ["true", "1", "yes"]:
        return 0.0

    n8 = float(row.get("N_Residues_8A", 0) or 0)
    n5 = float(row.get("N_Residues_5A", 0) or 0)

    if n8 >= 10 and n5 >= 5:
        return 1.0
    if n8 >= 10:
        return 0.85
    if n8 >= 5:
        return 0.70
    if n8 > 0:
        return 0.40
    return 0.0

def boltz_support(row):
    c = row.get("confidence_score", 0)

    try:
        c = float(c)
    except:
        c = 0

    if c >= 0.85:
        return 1.0
    if c >= 0.70:
        return 0.80
    if c >= 0.60:
        return 0.60
    if c > 0:
        return 0.30
    return 0.0

df["Ligand_Centered_Support"] = df.apply(ligand_support, axis=1)
df["P2Rank_Support"] = df["P2Rank_Hit_Class"].apply(support_from_hit)
df["fpocket_Support"] = df["fpocket_Hit_Class"].apply(support_from_hit)
df["Boltz_Confidence_Support"] = df.apply(boltz_support, axis=1)

df["Consensus_Holo_Pocket_Score"] = (
    0.35 * df["Ligand_Centered_Support"] +
    0.25 * df["P2Rank_Support"] +
    0.25 * df["fpocket_Support"] +
    0.15 * df["Boltz_Confidence_Support"]
).round(3)

def classify(score):
    if score >= 0.75:
        return "High_confidence_holo_pocket"
    if score >= 0.50:
        return "Probable_holo_pocket"
    if score >= 0.30:
        return "Uncertain_manual_check"
    return "Low_confidence_or_failed"

df["Consensus_Holo_Pocket_Class"] = df["Consensus_Holo_Pocket_Score"].apply(classify)

def next_action(row):
    cls = row["Consensus_Holo_Pocket_Class"]

    if cls == "High_confidence_holo_pocket":
        return "Proceed_to_substrate_docking_and_EZSpecificity"
    if cls == "Probable_holo_pocket":
        return "Proceed_but_keep_manual_inspection"
    if cls == "Uncertain_manual_check":
        return "Manual_inspection_or_rerun_Boltz_multiseed"
    return "Hold_or_alternative_cofactor_pose"

df["Recommended_Next_Action"] = df.apply(next_action, axis=1)

out_dir = BASE / f"consensus_holo_pocket/primary_organic/{batch_id}"
out_dir.mkdir(parents=True, exist_ok=True)

out_xlsx = out_dir / f"{batch_id}_consensus_holo_pocket_summary.xlsx"
df.to_excel(out_xlsx, index=False)

print("✅ batch consensus summary 完成")
print(out_xlsx)

print("\nConsensus class 统计:")
print(df["Consensus_Holo_Pocket_Class"].value_counts(dropna=False))

print("\nRecommended action 统计:")
print(df["Recommended_Next_Action"].value_counts(dropna=False))

display(df[[
    "PDB_File",
    "confidence_score",
    "N_Residues_5A",
    "N_Residues_8A",
    "P2Rank_Hit_Class",
    "Nearest_P2Rank_Distance_A",
    "fpocket_Hit_Class",
    "Nearest_fpocket_Distance_A",
    "Consensus_Holo_Pocket_Score",
    "Consensus_Holo_Pocket_Class",
    "Recommended_Next_Action"
]].sort_values("Consensus_Holo_Pocket_Score", ascending=False))

/content/drive/MyDrive/Horizyn_Checkpoints/consensus_holo_pocket/primary_organic/batch_001/batch_001_ligand_centered_pocket_residues.xlsx True
/content/drive/MyDrive/Horizyn_Checkpoints/p2rank_holo_outputs/primary_organic/batch_001_p2rank_ligand_distance_summary.xlsx True
/content/drive/MyDrive/Horizyn_Checkpoints/fpocket_holo_outputs/primary_organic/batch_001_fpocket_ligand_distance_summary.xlsx True
/content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_001_confidence_summary.xlsx True
✅ batch consensus summary 完成
/content/drive/MyDrive/Horizyn_Checkpoints/consensus_holo_pocket/primary_organic/batch_001/batch_001_consensus_holo_pocket_summary.xlsx

Consensus class 统计:
Consensus_Holo_Pocket_Class
High_confidence_holo_pocket    14
Probable_holo_pocket            4
Uncertain_manual_check          2
Name: count, dtype: int64

Recommended action 统计:
Recommended_Next_Action
Proceed_to_substrate_docking_and_EZSpecificity    14
Proceed_but_keep_manual_inspection      

,PDB_File,confidence_score,N_Residues_5A,N_Residues_8A,P2Rank_Hit_Class,Nearest_P2Rank_Distance_A,fpocket_Hit_Class,Nearest_fpocket_Distance_A,Consensus_Holo_Pocket_Score,Consensus_Holo_Pocket_Class,Recommended_Next_Action
0,NODE_11_length_154862_cov_66.368492_122__Heme_...,0.891467,37,70,Strong_<=6A,1.446,Strong_<=6A,1.262,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
2,NODE_13_length_131311_cov_66.505852_95__FMN__r...,0.878142,19,49,Strong_<=6A,1.695,Strong_<=6A,1.052,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
16,NODE_32_length_49516_cov_65.241935_22__FMN__ra...,0.960423,30,64,Strong_<=6A,1.215,Strong_<=6A,2.041,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
4,NODE_1_length_436095_cov_65.793628_253__FMN__r...,0.948654,23,58,Strong_<=6A,5.024,Strong_<=6A,5.923,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
14,NODE_26_length_70757_cov_67.122722_30__NAD___r...,0.896448,29,77,Strong_<=6A,0.769,Strong_<=6A,0.847,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
13,NODE_25_length_76722_cov_66.604332_89__NADP___...,0.877610,12,36,Strong_<=6A,3.633,Strong_<=6A,5.097,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
1,NODE_12_length_136290_cov_66.430554_166__Heme_...,0.843068,29,68,Strong_<=6A,0.685,Strong_<=6A,0.678,0.970,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
18,NODE_6_length_231008_cov_66.606731_103__FAD__r...,0.622942,28,52,Strong_<=6A,4.937,Strong_<=6A,4.317,0.940,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
11,NODE_25_length_76722_cov_66.604332_6__NAD___ra...,0.893232,19,41,Strong_<=6A,5.364,Moderate_6_10A,6.594,0.938,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
9,NODE_21_length_97532_cov_65.343769_69__NAD___r...,0.886403,28,68,Strong_<=6A,2.109,Moderate_6_10A,6.331,0.938,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity


In [49]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")
batch_id = "batch_001"

in_xlsx = BASE / f"consensus_holo_pocket/primary_organic/{batch_id}/{batch_id}_consensus_holo_pocket_summary.xlsx"
df = pd.read_excel(in_xlsx)

def conservative_action(row):
    score = row["Consensus_Holo_Pocket_Score"]
    p2 = str(row.get("P2Rank_Hit_Class", ""))
    fp = str(row.get("fpocket_Hit_Class", ""))

    p2_bad = (
        p2.startswith("No_") or
        p2.startswith("Miss") or
        p2 == "nan"
    )

    fp_bad = (
        fp.startswith("No_") or
        fp.startswith("Miss") or
        fp == "nan"
    )

    # 双 predictor 都不支持时，强制人工检查
    if p2_bad and fp_bad:
        return "Manual_inspection_or_rerun_Boltz_multiseed"

    if score >= 0.75:
        return "Proceed_to_substrate_docking_and_EZSpecificity"

    if score >= 0.50:
        return "Proceed_but_keep_manual_inspection"

    if score >= 0.30:
        return "Manual_inspection_or_rerun_Boltz_multiseed"

    return "Hold_or_alternative_cofactor_pose"

df["Recommended_Next_Action_Conservative"] = df.apply(conservative_action, axis=1)

out_xlsx = BASE / f"consensus_holo_pocket/primary_organic/{batch_id}/{batch_id}_consensus_holo_pocket_summary_conservative.xlsx"
df.to_excel(out_xlsx, index=False)

print("✅ Conservative summary 已生成")
print(out_xlsx)

print(df["Recommended_Next_Action_Conservative"].value_counts(dropna=False))

display(df[[
    "PDB_File",
    "confidence_score",
    "P2Rank_Hit_Class",
    "Nearest_P2Rank_Distance_A",
    "fpocket_Hit_Class",
    "Nearest_fpocket_Distance_A",
    "Consensus_Holo_Pocket_Score",
    "Consensus_Holo_Pocket_Class",
    "Recommended_Next_Action",
    "Recommended_Next_Action_Conservative"
]].sort_values("Consensus_Holo_Pocket_Score", ascending=False))

✅ Conservative summary 已生成
/content/drive/MyDrive/Horizyn_Checkpoints/consensus_holo_pocket/primary_organic/batch_001/batch_001_consensus_holo_pocket_summary_conservative.xlsx
Recommended_Next_Action_Conservative
Proceed_to_substrate_docking_and_EZSpecificity    14
Proceed_but_keep_manual_inspection                 3
Manual_inspection_or_rerun_Boltz_multiseed         3
Name: count, dtype: int64


,PDB_File,confidence_score,P2Rank_Hit_Class,Nearest_P2Rank_Distance_A,fpocket_Hit_Class,Nearest_fpocket_Distance_A,Consensus_Holo_Pocket_Score,Consensus_Holo_Pocket_Class,Recommended_Next_Action,Recommended_Next_Action_Conservative
0,NODE_11_length_154862_cov_66.368492_122__Heme_...,0.891467,Strong_<=6A,1.446,Strong_<=6A,1.262,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity,Proceed_to_substrate_docking_and_EZSpecificity
2,NODE_13_length_131311_cov_66.505852_95__FMN__r...,0.878142,Strong_<=6A,1.695,Strong_<=6A,1.052,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity,Proceed_to_substrate_docking_and_EZSpecificity
16,NODE_32_length_49516_cov_65.241935_22__FMN__ra...,0.960423,Strong_<=6A,1.215,Strong_<=6A,2.041,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity,Proceed_to_substrate_docking_and_EZSpecificity
4,NODE_1_length_436095_cov_65.793628_253__FMN__r...,0.948654,Strong_<=6A,5.024,Strong_<=6A,5.923,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity,Proceed_to_substrate_docking_and_EZSpecificity
14,NODE_26_length_70757_cov_67.122722_30__NAD___r...,0.896448,Strong_<=6A,0.769,Strong_<=6A,0.847,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity,Proceed_to_substrate_docking_and_EZSpecificity
13,NODE_25_length_76722_cov_66.604332_89__NADP___...,0.877610,Strong_<=6A,3.633,Strong_<=6A,5.097,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity,Proceed_to_substrate_docking_and_EZSpecificity
1,NODE_12_length_136290_cov_66.430554_166__Heme_...,0.843068,Strong_<=6A,0.685,Strong_<=6A,0.678,0.970,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity,Proceed_to_substrate_docking_and_EZSpecificity
18,NODE_6_length_231008_cov_66.606731_103__FAD__r...,0.622942,Strong_<=6A,4.937,Strong_<=6A,4.317,0.940,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity,Proceed_to_substrate_docking_and_EZSpecificity
11,NODE_25_length_76722_cov_66.604332_6__NAD___ra...,0.893232,Strong_<=6A,5.364,Moderate_6_10A,6.594,0.938,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity,Proceed_to_substrate_docking_and_EZSpecificity
9,NODE_21_length_97532_cov_65.343769_69__NAD___r...,0.886403,Strong_<=6A,2.109,Moderate_6_10A,6.331,0.938,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity,Proceed_to_substrate_docking_and_EZSpecificity


In [50]:
# ============================================================
# Master Auto-Runner：Primary organic/heme batch 自动流程
# 从 Boltz → P2Rank → fpocket → consensus 一键运行多个 batch
# ============================================================

from pathlib import Path
import os
import re
import json
import shutil
import subprocess
import pandas as pd
import numpy as np

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

# =========================
# 参数区：按需修改
# =========================
START_BATCH = 2                  # batch_001 已完成，所以从 2 开始
MAX_BATCHES_THIS_SESSION = 2     # 每次建议 1–3 个，稳定后可增加
FORCE_RERUN = False              # True 会强制重跑已有结果
USE_MSA_SERVER = True
USE_NO_KERNELS = True

BATCH_MANIFEST = BASE / "boltz_inputs/primary_organic_batches_manifest.xlsx"
BATCH_ROOT = BASE / "boltz_inputs/primary_organic_batches"

# 工具路径
FPOCKET_BIN = Path("/usr/local/bin/fpocket")

prank_candidates = list(Path("/content/p2rank").rglob("prank"))
if not prank_candidates:
    raise FileNotFoundError("没有找到 P2Rank prank，请先运行 P2Rank 安装 cell。")
PRANK = prank_candidates[0]
PRANK.chmod(0o755)

if not FPOCKET_BIN.exists():
    raise FileNotFoundError("没有找到 /usr/local/bin/fpocket，请先源码安装 fpocket。")

print("PRANK:", PRANK)
print("FPOCKET:", FPOCKET_BIN)
print("Batch manifest exists:", BATCH_MANIFEST.exists())


# ============================================================
# 通用工具函数
# ============================================================

def run_cmd_to_log(cmd, log_path, cwd=None, env=None):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)

    with open(log_path, "w", encoding="utf-8") as log:
        process = subprocess.run(
            cmd,
            cwd=str(cwd) if cwd else None,
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
            env=env
        )

    return process.returncode


def norm_file(x):
    x = str(x).split("/")[-1]
    x = x.replace(".pdb", "")
    x = x.replace(".cif", "")
    x = x.replace(".json", "")
    x = re.sub(r"[^A-Za-z0-9_.-]+", "_", x)
    return x


def ligand_center_from_pdb(pdb_path):
    coords = []
    with open(pdb_path, "r", errors="ignore") as f:
        for line in f:
            if line.startswith("HETATM"):
                try:
                    coords.append([
                        float(line[30:38]),
                        float(line[38:46]),
                        float(line[46:54])
                    ])
                except:
                    pass
    if not coords:
        return None
    return np.array(coords, dtype=float).mean(axis=0)


def pocket_center_from_pocket_pdb(pocket_pdb):
    coords = []
    with open(pocket_pdb, "r", errors="ignore") as f:
        for line in f:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                try:
                    coords.append([
                        float(line[30:38]),
                        float(line[38:46]),
                        float(line[46:54])
                    ])
                except:
                    pass
    if not coords:
        return None
    return np.array(coords, dtype=float).mean(axis=0)


def find_center_columns(df):
    cols = list(df.columns)
    low = {c.lower().strip(): c for c in cols}

    for trio in [
        ("center_x", "center_y", "center_z"),
        ("x", "y", "z"),
        ("cen_x", "cen_y", "cen_z")
    ]:
        if all(t in low for t in trio):
            return low[trio[0]], low[trio[1]], low[trio[2]]

    return None, None, None


def support_from_hit(hit):
    hit = str(hit)
    if hit.startswith("Strong"):
        return 1.0
    if hit.startswith("Moderate"):
        return 0.75
    if hit.startswith("Weak"):
        return 0.40
    return 0.0


def ligand_support(row):
    if str(row.get("Has_Ligand", False)).lower() not in ["true", "1", "yes"]:
        return 0.0

    try:
        n8 = float(row.get("N_Residues_8A", 0) or 0)
    except:
        n8 = 0

    try:
        n5 = float(row.get("N_Residues_5A", 0) or 0)
    except:
        n5 = 0

    if n8 >= 10 and n5 >= 5:
        return 1.0
    if n8 >= 10:
        return 0.85
    if n8 >= 5:
        return 0.70
    if n8 > 0:
        return 0.40
    return 0.0


def boltz_support(row):
    try:
        c = float(row.get("confidence_score", 0))
    except:
        c = 0

    if c >= 0.85:
        return 1.0
    if c >= 0.70:
        return 0.80
    if c >= 0.60:
        return 0.60
    if c > 0:
        return 0.30
    return 0.0


def conservative_action(row):
    score = row["Consensus_Holo_Pocket_Score"]
    p2 = str(row.get("P2Rank_Hit_Class", ""))
    fp = str(row.get("fpocket_Hit_Class", ""))

    p2_bad = p2.startswith("No_") or p2.startswith("Miss") or p2 == "nan"
    fp_bad = fp.startswith("No_") or fp.startswith("Miss") or fp == "nan"

    # 两个独立 pocket predictor 都不支持，则强制人工复核
    if p2_bad and fp_bad:
        return "Manual_inspection_or_rerun_Boltz_multiseed"

    if score >= 0.75:
        return "Proceed_to_substrate_docking_and_EZSpecificity"
    if score >= 0.50:
        return "Proceed_but_keep_manual_inspection"
    if score >= 0.30:
        return "Manual_inspection_or_rerun_Boltz_multiseed"
    return "Hold_or_alternative_cofactor_pose"


# ============================================================
# 单个 batch 的各步骤
# ============================================================

def run_boltz_batch(batch_id):
    drive_batch_dir = BASE / f"boltz_inputs/primary_organic_batches/{batch_id}"
    drive_out_dir = BASE / f"boltz_outputs/primary_organic/{batch_id}"

    local_input = Path(f"/content/boltz_{batch_id}_input")
    local_out = Path(f"/content/boltz_{batch_id}_out")
    local_cache = Path("/content/boltz_cache")

    conf_summary = BASE / f"boltz_outputs/primary_organic/{batch_id}_confidence_summary.xlsx"

    if conf_summary.exists() and not FORCE_RERUN:
        print(f"  ✅ Boltz confidence 已存在，跳过 Boltz: {batch_id}")
        return True

    for d in [local_input, local_out]:
        if d.exists():
            shutil.rmtree(d)
        d.mkdir(parents=True, exist_ok=True)

    local_cache.mkdir(parents=True, exist_ok=True)

    yaml_files = sorted(drive_batch_dir.glob("*.yaml"))
    print(f"  YAML 数量: {len(yaml_files)}")

    if len(yaml_files) == 0:
        print(f"  ❌ 没有 YAML: {drive_batch_dir}")
        return False

    for y in yaml_files:
        shutil.copy2(y, local_input / y.name)

    cmd = [
        "boltz", "predict", str(local_input),
        "--out_dir", str(local_out),
        "--output_format", "pdb",
        "--override"
    ]

    if USE_MSA_SERVER:
        cmd.append("--use_msa_server")

    if USE_NO_KERNELS:
        cmd.append("--no_kernels")

    env = os.environ.copy()
    env["BOLTZ_CACHE"] = str(local_cache)

    log_path = BASE / f"boltz_outputs/primary_organic/{batch_id}_run.log"
    log_path.parent.mkdir(parents=True, exist_ok=True)

    print("  Running Boltz...")
    ret = run_cmd_to_log(cmd, log_path, env=env)

    if ret != 0:
        print(f"  ❌ Boltz 失败: {batch_id}，查看日志: {log_path}")
        return False

    if drive_out_dir.exists():
        shutil.rmtree(drive_out_dir)
    shutil.copytree(local_out, drive_out_dir)

    print(f"  ✅ Boltz 完成: {drive_out_dir}")
    return True


def summarize_boltz_confidence(batch_id):
    out_dir = BASE / f"boltz_outputs/primary_organic/{batch_id}"
    summary_path = BASE / f"boltz_outputs/primary_organic/{batch_id}_confidence_summary.xlsx"

    if summary_path.exists() and not FORCE_RERUN:
        print(f"  ✅ confidence summary 已存在，跳过")
        return summary_path

    rows = []

    all_conf = sorted(out_dir.rglob("confidence*.json"))
    all_pdb = sorted(out_dir.rglob("*.pdb"))

    candidate_dirs = sorted(set([p.parent for p in all_conf + all_pdb]))

    for pred_dir in candidate_dirs:
        conf_files = sorted(pred_dir.glob("confidence*.json"))
        pdb_files = sorted(pred_dir.glob("*.pdb"))

        conf = {}
        if conf_files:
            with open(conf_files[0]) as f:
                conf = json.load(f)

        rows.append({
            "Batch_ID": batch_id,
            "Prediction_Dir": str(pred_dir),
            "Best_Model_PDB": str(pdb_files[0]) if pdb_files else "",
            "Confidence_JSON": str(conf_files[0]) if conf_files else "",
            "confidence_score": conf.get("confidence_score", None),
            "ptm": conf.get("ptm", None),
            "iptm": conf.get("iptm", None),
            "ligand_iptm": conf.get("ligand_iptm", None),
            "complex_plddt": conf.get("complex_plddt", None),
            "complex_iplddt": conf.get("complex_iplddt", None),
        })

    df_conf = pd.DataFrame(rows)
    df_conf.to_excel(summary_path, index=False)

    print(f"  ✅ confidence summary: {summary_path}")
    print(f"  PDB: {len(all_pdb)}, confidence JSON: {len(all_conf)}")
    return summary_path


def collect_pdb_and_ligand_check(batch_id):
    boltz_out = BASE / f"boltz_outputs/primary_organic/{batch_id}"
    pdb_collect_dir = BASE / f"boltz_outputs/primary_organic/{batch_id}_pdb_models"
    pdb_collect_dir.mkdir(parents=True, exist_ok=True)

    if FORCE_RERUN:
        for f in pdb_collect_dir.glob("*.pdb"):
            f.unlink()

    if len(list(pdb_collect_dir.glob("*.pdb"))) == 0:
        pdb_files = sorted(boltz_out.rglob("*.pdb"))
        for pdb in pdb_files:
            shutil.copy2(pdb, pdb_collect_dir / pdb.name)

    rows = []

    for pdb in sorted(pdb_collect_dir.glob("*.pdb")):
        atom_count = 0
        hetatm_count = 0
        ligand_resnames = set()

        with open(pdb, "r", errors="ignore") as f:
            for line in f:
                if line.startswith("ATOM"):
                    atom_count += 1
                elif line.startswith("HETATM"):
                    hetatm_count += 1
                    ligand_resnames.add(line[17:20].strip())

        rows.append({
            "Batch_ID": batch_id,
            "PDB_File": pdb.name,
            "PDB_Path": str(pdb),
            "ATOM_count": atom_count,
            "HETATM_count": hetatm_count,
            "Ligand_Resnames": ";".join(sorted(ligand_resnames)),
            "Has_Ligand": hetatm_count > 0,
            "Has_Protein": atom_count > 0,
        })

    df = pd.DataFrame(rows)
    out_xlsx = BASE / f"boltz_outputs/primary_organic/{batch_id}_ligand_check.xlsx"
    df.to_excel(out_xlsx, index=False)

    print(f"  ✅ ligand check: {out_xlsx}")
    print(df[["Has_Protein", "Has_Ligand"]].value_counts(dropna=False))

    return pdb_collect_dir, out_xlsx


def extract_ligand_centered_pockets(batch_id, pdb_dir):
    out_dir = BASE / f"consensus_holo_pocket/primary_organic/{batch_id}"
    out_dir.mkdir(parents=True, exist_ok=True)

    out_xlsx = out_dir / f"{batch_id}_ligand_centered_pocket_residues.xlsx"

    if out_xlsx.exists() and not FORCE_RERUN:
        print("  ✅ ligand-centered pocket 已存在，跳过")
        return out_xlsx

    def parse_pdb_atoms(pdb_path):
        protein_atoms = []
        ligand_atoms = []

        with open(pdb_path, "r", errors="ignore") as f:
            for line in f:
                if not (line.startswith("ATOM") or line.startswith("HETATM")):
                    continue

                try:
                    coord = np.array([
                        float(line[30:38]),
                        float(line[38:46]),
                        float(line[46:54])
                    ], dtype=float)
                except:
                    continue

                atom = {
                    "record": line[0:6].strip(),
                    "atom_name": line[12:16].strip(),
                    "resname": line[17:20].strip(),
                    "chain": line[21].strip(),
                    "resseq": line[22:26].strip(),
                    "icode": line[26].strip(),
                    "coord": coord,
                }

                if line.startswith("ATOM"):
                    protein_atoms.append(atom)
                elif line.startswith("HETATM"):
                    ligand_atoms.append(atom)

        return protein_atoms, ligand_atoms

    def residue_key(atom):
        return f"{atom['chain']}:{atom['resname']}{atom['resseq']}{atom['icode']}"

    rows = []

    for pdb in sorted(pdb_dir.glob("*.pdb")):
        protein_atoms, ligand_atoms = parse_pdb_atoms(pdb)

        if len(ligand_atoms) == 0:
            rows.append({
                "Batch_ID": batch_id,
                "PDB_File": pdb.name,
                "PDB_Path": str(pdb),
                "Has_Ligand": False,
                "Ligand_Atom_Count": 0,
                "Ligand_Resnames": "",
                "N_Residues_5A": 0,
                "N_Residues_8A": 0,
                "Pocket_Residues_5A": "",
                "Pocket_Residues_8A": "",
            })
            continue

        lig_coords = np.array([a["coord"] for a in ligand_atoms], dtype=float)
        lig_center = lig_coords.mean(axis=0)

        res_5 = set()
        res_8 = set()

        for atom in protein_atoms:
            min_dist = np.min(np.linalg.norm(lig_coords - atom["coord"], axis=1))
            if min_dist <= 5.0:
                res_5.add(residue_key(atom))
            if min_dist <= 8.0:
                res_8.add(residue_key(atom))

        rows.append({
            "Batch_ID": batch_id,
            "PDB_File": pdb.name,
            "PDB_Path": str(pdb),
            "Has_Ligand": True,
            "Ligand_Atom_Count": len(ligand_atoms),
            "Ligand_Resnames": ";".join(sorted(set(a["resname"] for a in ligand_atoms))),
            "Ligand_Center_X": lig_center[0],
            "Ligand_Center_Y": lig_center[1],
            "Ligand_Center_Z": lig_center[2],
            "N_Residues_5A": len(res_5),
            "N_Residues_8A": len(res_8),
            "Pocket_Residues_5A": ";".join(sorted(res_5)),
            "Pocket_Residues_8A": ";".join(sorted(res_8)),
        })

    df = pd.DataFrame(rows)
    df.to_excel(out_xlsx, index=False)

    print(f"  ✅ ligand-centered pocket: {out_xlsx}")
    return out_xlsx


def run_p2rank_and_distance(batch_id, pdb_dir):
    p2rank_out = BASE / f"p2rank_holo_outputs/primary_organic/{batch_id}_single"
    summary_xlsx = BASE / f"p2rank_holo_outputs/primary_organic/{batch_id}_p2rank_ligand_distance_summary.xlsx"
    summary_xlsx.parent.mkdir(parents=True, exist_ok=True)

    if summary_xlsx.exists() and not FORCE_RERUN:
        print("  ✅ P2Rank summary 已存在，跳过")
        return summary_xlsx

    if p2rank_out.exists():
        shutil.rmtree(p2rank_out)
    p2rank_out.mkdir(parents=True, exist_ok=True)

    for pdb in sorted(pdb_dir.glob("*.pdb")):
        sample_out = p2rank_out / pdb.stem
        sample_out.mkdir(parents=True, exist_ok=True)

        cmd = [
            str(PRANK),
            "predict",
            "-f", str(pdb),
            "-o", str(sample_out),
            "-c", "alphafold",
            "-threads", "2"
        ]

        log_file = sample_out / "p2rank_run.log"
        run_cmd_to_log(cmd, log_file)

    rows = []

    for pdb in sorted(pdb_dir.glob("*.pdb")):
        sample_dir = p2rank_out / pdb.stem
        lig_center = ligand_center_from_pdb(pdb)

        pred_csvs = sorted(sample_dir.rglob("*_predictions.csv"))
        if not pred_csvs:
            pred_csvs = sorted(sample_dir.rglob("*predictions*.csv"))

        if not pred_csvs:
            rows.append({
                "Batch_ID": batch_id,
                "PDB_File": pdb.name,
                "PDB_Path": str(pdb),
                "Has_Ligand": lig_center is not None,
                "Num_P2Rank_Pockets": 0,
                "Nearest_P2Rank_Distance_A": None,
                "P2Rank_Hit_Class": "No_P2Rank_CSV"
            })
            continue

        try:
            df_p = pd.read_csv(pred_csvs[0])
        except:
            df_p = pd.read_csv(pred_csvs[0], sep="\t")

        if df_p.empty:
            rows.append({
                "Batch_ID": batch_id,
                "PDB_File": pdb.name,
                "PDB_Path": str(pdb),
                "Has_Ligand": lig_center is not None,
                "Num_P2Rank_Pockets": 0,
                "Nearest_P2Rank_Distance_A": None,
                "P2Rank_Hit_Class": "No_P2Rank_pocket_detected",
                "P2Rank_CSV": str(pred_csvs[0])
            })
            continue

        xcol, ycol, zcol = find_center_columns(df_p)

        if lig_center is None or xcol is None:
            rows.append({
                "Batch_ID": batch_id,
                "PDB_File": pdb.name,
                "PDB_Path": str(pdb),
                "Has_Ligand": lig_center is not None,
                "Num_P2Rank_Pockets": len(df_p),
                "Nearest_P2Rank_Distance_A": None,
                "P2Rank_Hit_Class": "Cannot_compute_distance",
                "P2Rank_CSV": str(pred_csvs[0])
            })
            continue

        dists = []
        for _, r in df_p.iterrows():
            pc = np.array([float(r[xcol]), float(r[ycol]), float(r[zcol])])
            dists.append(float(np.linalg.norm(pc - lig_center)))

        df_p["Ligand_to_Pocket_Center_A"] = dists
        nearest = df_p.sort_values("Ligand_to_Pocket_Center_A").iloc[0]
        dist = float(nearest["Ligand_to_Pocket_Center_A"])

        if dist <= 6:
            hit = "Strong_<=6A"
        elif dist <= 10:
            hit = "Moderate_6_10A"
        elif dist <= 15:
            hit = "Weak_10_15A"
        else:
            hit = "Miss_>15A"

        score_col = None
        for c in ["score", "probability", "pocket_score", "prob"]:
            if c in df_p.columns:
                score_col = c
                break

        rows.append({
            "Batch_ID": batch_id,
            "PDB_File": pdb.name,
            "PDB_Path": str(pdb),
            "Has_Ligand": True,
            "Num_P2Rank_Pockets": len(df_p),
            "Nearest_P2Rank_Distance_A": round(dist, 3),
            "P2Rank_Hit_Class": hit,
            "Nearest_P2Rank_Score": nearest[score_col] if score_col else None,
            "P2Rank_CSV": str(pred_csvs[0])
        })

    df = pd.DataFrame(rows)
    df.to_excel(summary_xlsx, index=False)

    print(f"  ✅ P2Rank summary: {summary_xlsx}")
    print(df["P2Rank_Hit_Class"].value_counts(dropna=False))

    return summary_xlsx


def run_fpocket_and_distance(batch_id, pdb_dir):
    protein_only_dir = BASE / f"fpocket_holo_inputs/primary_organic/{batch_id}_protein_only_pdb"
    fpocket_out_drive = BASE / f"fpocket_holo_outputs/primary_organic/{batch_id}_protein_only"
    summary_xlsx = BASE / f"fpocket_holo_outputs/primary_organic/{batch_id}_fpocket_ligand_distance_summary.xlsx"
    summary_xlsx.parent.mkdir(parents=True, exist_ok=True)

    if summary_xlsx.exists() and not FORCE_RERUN:
        print("  ✅ fpocket summary 已存在，跳过")
        return summary_xlsx

    protein_only_dir.mkdir(parents=True, exist_ok=True)
    for f in protein_only_dir.glob("*.pdb"):
        f.unlink()

    # protein-only PDB
    for pdb in sorted(pdb_dir.glob("*.pdb")):
        out_pdb = protein_only_dir / pdb.name
        with open(pdb, "r", errors="ignore") as fin, open(out_pdb, "w") as fout:
            for line in fin:
                if line.startswith("ATOM") or line.startswith("TER") or line.startswith("END"):
                    fout.write(line)

    local_run_dir = Path(f"/content/fpocket_{batch_id}_protein_only")
    if local_run_dir.exists():
        shutil.rmtree(local_run_dir)
    local_run_dir.mkdir(parents=True, exist_ok=True)

    for pdb in sorted(protein_only_dir.glob("*.pdb")):
        shutil.copy2(pdb, local_run_dir / pdb.name)

    for pdb in sorted(local_run_dir.glob("*.pdb")):
        cmd = [str(FPOCKET_BIN), "-f", pdb.name]
        log_path = local_run_dir / f"{pdb.stem}_fpocket.log"
        run_cmd_to_log(cmd, log_path, cwd=local_run_dir)

    if fpocket_out_drive.exists():
        shutil.rmtree(fpocket_out_drive)
    shutil.copytree(local_run_dir, fpocket_out_drive)

    rows = []

    for holo_pdb in sorted(pdb_dir.glob("*.pdb")):
        lig_center = ligand_center_from_pdb(holo_pdb)

        out_folder = fpocket_out_drive / f"{holo_pdb.stem}_out"
        pockets_dir = out_folder / "pockets"

        pocket_files = sorted(pockets_dir.glob("pocket*_atm.pdb")) if pockets_dir.exists() else []

        if lig_center is None:
            rows.append({
                "Batch_ID": batch_id,
                "PDB_File": holo_pdb.name,
                "Num_fpocket_Pockets": len(pocket_files),
                "Nearest_fpocket_Distance_A": None,
                "fpocket_Hit_Class": "No_ligand"
            })
            continue

        if not pocket_files:
            rows.append({
                "Batch_ID": batch_id,
                "PDB_File": holo_pdb.name,
                "Num_fpocket_Pockets": 0,
                "Nearest_fpocket_Distance_A": None,
                "fpocket_Hit_Class": "No_fpocket_pocket"
            })
            continue

        dist_rows = []

        for pocket in pocket_files:
            pc = pocket_center_from_pocket_pdb(pocket)
            if pc is None:
                continue
            dist = float(np.linalg.norm(pc - lig_center))
            dist_rows.append((pocket, dist))

        if not dist_rows:
            rows.append({
                "Batch_ID": batch_id,
                "PDB_File": holo_pdb.name,
                "Num_fpocket_Pockets": len(pocket_files),
                "Nearest_fpocket_Distance_A": None,
                "fpocket_Hit_Class": "Cannot_compute"
            })
            continue

        nearest_pocket, nearest_dist = sorted(dist_rows, key=lambda x: x[1])[0]

        if nearest_dist <= 6:
            hit = "Strong_<=6A"
        elif nearest_dist <= 10:
            hit = "Moderate_6_10A"
        elif nearest_dist <= 15:
            hit = "Weak_10_15A"
        else:
            hit = "Miss_>15A"

        rows.append({
            "Batch_ID": batch_id,
            "PDB_File": holo_pdb.name,
            "Num_fpocket_Pockets": len(pocket_files),
            "Nearest_fpocket_Distance_A": round(nearest_dist, 3),
            "Nearest_fpocket_Pocket": str(nearest_pocket),
            "fpocket_Hit_Class": hit
        })

    df = pd.DataFrame(rows)
    df.to_excel(summary_xlsx, index=False)

    print(f"  ✅ fpocket summary: {summary_xlsx}")
    print(df["fpocket_Hit_Class"].value_counts(dropna=False))

    return summary_xlsx


def make_consensus(batch_id):
    lig_xlsx = BASE / f"consensus_holo_pocket/primary_organic/{batch_id}/{batch_id}_ligand_centered_pocket_residues.xlsx"
    p2rank_xlsx = BASE / f"p2rank_holo_outputs/primary_organic/{batch_id}_p2rank_ligand_distance_summary.xlsx"
    fpocket_xlsx = BASE / f"fpocket_holo_outputs/primary_organic/{batch_id}_fpocket_ligand_distance_summary.xlsx"
    boltz_conf = BASE / f"boltz_outputs/primary_organic/{batch_id}_confidence_summary.xlsx"

    out_dir = BASE / f"consensus_holo_pocket/primary_organic/{batch_id}"
    out_dir.mkdir(parents=True, exist_ok=True)

    out_xlsx = out_dir / f"{batch_id}_consensus_holo_pocket_summary_conservative.xlsx"

    if out_xlsx.exists() and not FORCE_RERUN:
        print("  ✅ consensus summary 已存在，跳过")
        return out_xlsx

    df_lig = pd.read_excel(lig_xlsx)
    df_p2 = pd.read_excel(p2rank_xlsx)
    df_fp = pd.read_excel(fpocket_xlsx)
    df_conf = pd.read_excel(boltz_conf)

    df_lig["norm"] = df_lig["PDB_File"].apply(norm_file)
    df_p2["norm"] = df_p2["PDB_File"].apply(norm_file)
    df_fp["norm"] = df_fp["PDB_File"].apply(norm_file)
    df_conf["norm"] = df_conf["Best_Model_PDB"].apply(norm_file)

    df = df_lig.merge(df_p2, on="norm", how="left", suffixes=("", "_p2rank"))
    df = df.merge(df_fp, on="norm", how="left", suffixes=("", "_fpocket"))
    df = df.merge(df_conf, on="norm", how="left", suffixes=("", "_boltz"))

    df["Ligand_Centered_Support"] = df.apply(ligand_support, axis=1)
    df["P2Rank_Support"] = df["P2Rank_Hit_Class"].apply(support_from_hit)
    df["fpocket_Support"] = df["fpocket_Hit_Class"].apply(support_from_hit)
    df["Boltz_Confidence_Support"] = df.apply(boltz_support, axis=1)

    df["Consensus_Holo_Pocket_Score"] = (
        0.35 * df["Ligand_Centered_Support"] +
        0.25 * df["P2Rank_Support"] +
        0.25 * df["fpocket_Support"] +
        0.15 * df["Boltz_Confidence_Support"]
    ).round(3)

    def classify(score):
        if score >= 0.75:
            return "High_confidence_holo_pocket"
        if score >= 0.50:
            return "Probable_holo_pocket"
        if score >= 0.30:
            return "Uncertain_manual_check"
        return "Low_confidence_or_failed"

    df["Consensus_Holo_Pocket_Class"] = df["Consensus_Holo_Pocket_Score"].apply(classify)
    df["Recommended_Next_Action_Conservative"] = df.apply(conservative_action, axis=1)

    df.to_excel(out_xlsx, index=False)

    print(f"  ✅ consensus: {out_xlsx}")
    print(df["Consensus_Holo_Pocket_Class"].value_counts(dropna=False))
    print(df["Recommended_Next_Action_Conservative"].value_counts(dropna=False))

    return out_xlsx


def run_one_batch(batch_id):
    print("\n" + "=" * 80)
    print(f"🚀 Running {batch_id}")
    print("=" * 80)

    final_xlsx = BASE / f"consensus_holo_pocket/primary_organic/{batch_id}/{batch_id}_consensus_holo_pocket_summary_conservative.xlsx"

    if final_xlsx.exists() and not FORCE_RERUN:
        print(f"✅ {batch_id} 已完成，跳过")
        return {"Batch_ID": batch_id, "Status": "skipped_existing", "Consensus": str(final_xlsx)}

    ok = run_boltz_batch(batch_id)
    if not ok:
        return {"Batch_ID": batch_id, "Status": "boltz_failed", "Consensus": ""}

    summarize_boltz_confidence(batch_id)
    pdb_dir, _ = collect_pdb_and_ligand_check(batch_id)

    extract_ligand_centered_pockets(batch_id, pdb_dir)
    run_p2rank_and_distance(batch_id, pdb_dir)
    run_fpocket_and_distance(batch_id, pdb_dir)
    consensus = make_consensus(batch_id)

    return {"Batch_ID": batch_id, "Status": "completed", "Consensus": str(consensus)}


# ============================================================
# 选择本次要跑的 batch
# ============================================================

df_batches = pd.read_excel(BATCH_MANIFEST)

all_batch_ids = sorted(df_batches["Batch_ID"].dropna().unique().tolist())
all_batch_nums = [(bid, int(str(bid).split("_")[1])) for bid in all_batch_ids]

selected = [
    bid for bid, n in all_batch_nums
    if n >= START_BATCH
][:MAX_BATCHES_THIS_SESSION]

print("全部 batch 数:", len(all_batch_ids))
print("本次将运行:", selected)

if not selected:
    raise ValueError("没有可运行的 batch。请检查 START_BATCH 或 manifest。")


# ============================================================
# 开始运行
# ============================================================

run_records = []

for batch_id in selected:
    record = run_one_batch(batch_id)
    run_records.append(record)

df_run = pd.DataFrame(run_records)

run_summary = BASE / f"consensus_holo_pocket/primary_organic/autorun_summary_start{START_BATCH}_n{MAX_BATCHES_THIS_SESSION}.xlsx"
run_summary.parent.mkdir(parents=True, exist_ok=True)
df_run.to_excel(run_summary, index=False)

print("\n" + "=" * 80)
print("✅ 本次自动运行完成")
print(run_summary)
display(df_run)

PRANK: /content/p2rank/p2rank_2.5/prank
FPOCKET: /usr/local/bin/fpocket
Batch manifest exists: True
全部 batch 数: 19
本次将运行: ['batch_002', 'batch_003']

🚀 Running batch_002
  YAML 数量: 20
  Running Boltz...
  ✅ Boltz 完成: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_002
  ✅ confidence summary: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_002_confidence_summary.xlsx
  PDB: 20, confidence JSON: 20
  ✅ ligand check: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_002_ligand_check.xlsx
Has_Protein  Has_Ligand
True         True          20
Name: count, dtype: int64
  ✅ ligand-centered pocket: /content/drive/MyDrive/Horizyn_Checkpoints/consensus_holo_pocket/primary_organic/batch_002/batch_002_ligand_centered_pocket_residues.xlsx
  ✅ P2Rank summary: /content/drive/MyDrive/Horizyn_Checkpoints/p2rank_holo_outputs/primary_organic/batch_002_p2rank_ligand_distance_summary.xlsx
P2Rank_Hit_Class
Strong_<=6A

,Batch_ID,Status,Consensus
0,batch_002,completed,/content/drive/MyDrive/Horizyn_Checkpoints/con...
1,batch_003,completed,/content/drive/MyDrive/Horizyn_Checkpoints/con...


In [51]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

summary_files = sorted(
    (BASE / "consensus_holo_pocket/primary_organic").rglob(
        "batch_*_consensus_holo_pocket_summary_conservative.xlsx"
    )
)

print("已完成 batch summary 数量:", len(summary_files))

dfs = []

for f in summary_files:
    df = pd.read_excel(f)
    df["Summary_File"] = str(f)
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)

out_xlsx = BASE / "consensus_holo_pocket/primary_organic/primary_organic_all_consensus_holo_pocket_summary_current.xlsx"
df_all.to_excel(out_xlsx, index=False)

print("✅ 当前已完成 batch 合并表:")
print(out_xlsx)

print("\nConsensus class 总统计:")
print(df_all["Consensus_Holo_Pocket_Class"].value_counts(dropna=False))

print("\nConservative action 总统计:")
print(df_all["Recommended_Next_Action_Conservative"].value_counts(dropna=False))

display(df_all[[
    "Batch_ID",
    "PDB_File",
    "confidence_score",
    "P2Rank_Hit_Class",
    "fpocket_Hit_Class",
    "Consensus_Holo_Pocket_Score",
    "Consensus_Holo_Pocket_Class",
    "Recommended_Next_Action_Conservative"
]].sort_values("Consensus_Holo_Pocket_Score", ascending=False).head(30))

已完成 batch summary 数量: 3
✅ 当前已完成 batch 合并表:
/content/drive/MyDrive/Horizyn_Checkpoints/consensus_holo_pocket/primary_organic/primary_organic_all_consensus_holo_pocket_summary_current.xlsx

Consensus class 总统计:
Consensus_Holo_Pocket_Class
High_confidence_holo_pocket    45
Probable_holo_pocket           10
Uncertain_manual_check          4
Low_confidence_or_failed        1
Name: count, dtype: int64

Conservative action 总统计:
Recommended_Next_Action_Conservative
Proceed_to_substrate_docking_and_EZSpecificity    45
Proceed_but_keep_manual_inspection                 9
Manual_inspection_or_rerun_Boltz_multiseed         6
Name: count, dtype: int64


,Batch_ID,PDB_File,confidence_score,P2Rank_Hit_Class,fpocket_Hit_Class,Consensus_Holo_Pocket_Score,Consensus_Holo_Pocket_Class,Recommended_Next_Action_Conservative
0,batch_001,NODE_11_length_154862_cov_66.368492_122__Heme_...,0.891467,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
2,batch_001,NODE_13_length_131311_cov_66.505852_95__FMN__r...,0.878142,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
4,batch_001,NODE_1_length_436095_cov_65.793628_253__FMN__r...,0.948654,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
13,batch_001,NODE_25_length_76722_cov_66.604332_89__NADP___...,0.877610,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
20,batch_002,NODE_13_length_131311_cov_66.505852_95__PLP__r...,0.942248,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
23,batch_002,NODE_27_length_69195_cov_65.603663_62__NADP___...,0.940614,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
27,batch_002,NODE_3_length_362446_cov_66.707646_252__FMN__r...,0.912947,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
24,batch_002,NODE_27_length_69195_cov_65.603663_62__NAD___r...,0.936008,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
14,batch_001,NODE_26_length_70757_cov_67.122722_30__NAD___r...,0.896448,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
16,batch_001,NODE_32_length_49516_cov_65.241935_22__FMN__ra...,0.960423,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity


In [52]:
from pathlib import Path
import pandas as pd
import re

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

batch_manifest = BASE / "boltz_inputs/primary_organic_batches_manifest.xlsx"
consensus_root = BASE / "consensus_holo_pocket/primary_organic"

df_batches = pd.read_excel(batch_manifest)

all_batches = sorted(df_batches["Batch_ID"].dropna().unique().tolist())
all_batch_nums = {b: int(b.split("_")[1]) for b in all_batches}

finished = []
for b in all_batches:
    f = consensus_root / b / f"{b}_consensus_holo_pocket_summary_conservative.xlsx"
    if f.exists():
        finished.append(b)

unfinished = [b for b in all_batches if b not in finished]

print("总 batch 数:", len(all_batches))
print("已完成:", len(finished), finished)
print("未完成:", len(unfinished), unfinished[:10])

if unfinished:
    next_batch = unfinished[0]
    next_num = all_batch_nums[next_batch]
    print("\n下一轮建议从这里开始：")
    print("START_BATCH =", next_num)
else:
    print("全部 batch 已完成")

总 batch 数: 19
已完成: 3 ['batch_001', 'batch_002', 'batch_003']
未完成: 16 ['batch_004', 'batch_005', 'batch_006', 'batch_007', 'batch_008', 'batch_009', 'batch_010', 'batch_011', 'batch_012', 'batch_013']

下一轮建议从这里开始：
START_BATCH = 4


In [ ]:
# ============================================================
# 续跑 Primary organic/heme batches
# 当前已完成 batch_001–batch_003
# 本轮运行 batch_004–batch_005
# ============================================================

from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

START_BATCH = 4
MAX_BATCHES_THIS_SESSION = 2
FORCE_RERUN = False

BATCH_MANIFEST = BASE / "boltz_inputs/primary_organic_batches_manifest.xlsx"
CONSENSUS_ROOT = BASE / "consensus_holo_pocket/primary_organic"

# 检查 run_one_batch 是否已经定义
if "run_one_batch" not in globals():
    raise RuntimeError(
        "当前 runtime 里没有 run_one_batch()。请先运行前面那个完整 Master Auto-Runner 定义 Cell，"
        "然后再运行这个续跑 Cell。"
    )

df_batches = pd.read_excel(BATCH_MANIFEST)

all_batch_ids = sorted(df_batches["Batch_ID"].dropna().unique().tolist())
all_batch_nums = [(bid, int(str(bid).split("_")[1])) for bid in all_batch_ids]

# 找到未完成 batch
finished = []
for bid in all_batch_ids:
    final_xlsx = CONSENSUS_ROOT / bid / f"{bid}_consensus_holo_pocket_summary_conservative.xlsx"
    if final_xlsx.exists():
        finished.append(bid)

unfinished = [bid for bid in all_batch_ids if bid not in finished]

selected = [
    bid for bid, n in all_batch_nums
    if n >= START_BATCH and bid in unfinished
][:MAX_BATCHES_THIS_SESSION]

print("总 batch 数:", len(all_batch_ids))
print("已完成:", len(finished), finished)
print("未完成:", len(unfinished), unfinished[:20])
print("本轮将运行:", selected)

if not selected:
    raise ValueError("没有需要运行的 batch。可能全部完成，或 START_BATCH 设置过大。")

run_records = []

for batch_id in selected:
    record = run_one_batch(batch_id)
    run_records.append(record)

df_run = pd.DataFrame(run_records)

run_summary = BASE / f"consensus_holo_pocket/primary_organic/autorun_summary_start{START_BATCH}_n{MAX_BATCHES_THIS_SESSION}.xlsx"
run_summary.parent.mkdir(parents=True, exist_ok=True)
df_run.to_excel(run_summary, index=False)

print("\n✅ 本轮 batch 自动运行完成")
print(run_summary)

display(df_run)

In [53]:
# ============================================================
# 续跑 Primary organic/heme batches
# 当前已完成 batch_001–batch_003
# 本轮运行 batch_004–batch_005
# ============================================================

from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

START_BATCH = 4
MAX_BATCHES_THIS_SESSION = 2
FORCE_RERUN = False

BATCH_MANIFEST = BASE / "boltz_inputs/primary_organic_batches_manifest.xlsx"
CONSENSUS_ROOT = BASE / "consensus_holo_pocket/primary_organic"

# 检查 run_one_batch 是否已经定义
if "run_one_batch" not in globals():
    raise RuntimeError(
        "当前 runtime 里没有 run_one_batch()。请先运行前面那个完整 Master Auto-Runner 定义 Cell，"
        "然后再运行这个续跑 Cell。"
    )

df_batches = pd.read_excel(BATCH_MANIFEST)

all_batch_ids = sorted(df_batches["Batch_ID"].dropna().unique().tolist())
all_batch_nums = [(bid, int(str(bid).split("_")[1])) for bid in all_batch_ids]

# 找到未完成 batch
finished = []
for bid in all_batch_ids:
    final_xlsx = CONSENSUS_ROOT / bid / f"{bid}_consensus_holo_pocket_summary_conservative.xlsx"
    if final_xlsx.exists():
        finished.append(bid)

unfinished = [bid for bid in all_batch_ids if bid not in finished]

selected = [
    bid for bid, n in all_batch_nums
    if n >= START_BATCH and bid in unfinished
][:MAX_BATCHES_THIS_SESSION]

print("总 batch 数:", len(all_batch_ids))
print("已完成:", len(finished), finished)
print("未完成:", len(unfinished), unfinished[:20])
print("本轮将运行:", selected)

if not selected:
    raise ValueError("没有需要运行的 batch。可能全部完成，或 START_BATCH 设置过大。")

run_records = []

for batch_id in selected:
    record = run_one_batch(batch_id)
    run_records.append(record)

df_run = pd.DataFrame(run_records)

run_summary = BASE / f"consensus_holo_pocket/primary_organic/autorun_summary_start{START_BATCH}_n{MAX_BATCHES_THIS_SESSION}.xlsx"
run_summary.parent.mkdir(parents=True, exist_ok=True)
df_run.to_excel(run_summary, index=False)

print("\n✅ 本轮 batch 自动运行完成")
print(run_summary)

display(df_run)

总 batch 数: 19
已完成: 3 ['batch_001', 'batch_002', 'batch_003']
未完成: 16 ['batch_004', 'batch_005', 'batch_006', 'batch_007', 'batch_008', 'batch_009', 'batch_010', 'batch_011', 'batch_012', 'batch_013', 'batch_014', 'batch_015', 'batch_016', 'batch_017', 'batch_018', 'batch_019']
本轮将运行: ['batch_004', 'batch_005']

🚀 Running batch_004
  YAML 数量: 20
  Running Boltz...
  ✅ Boltz 完成: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_004
  ✅ confidence summary: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_004_confidence_summary.xlsx
  PDB: 20, confidence JSON: 20
  ✅ ligand check: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_004_ligand_check.xlsx
Has_Protein  Has_Ligand
True         True          20
Name: count, dtype: int64
  ✅ ligand-centered pocket: /content/drive/MyDrive/Horizyn_Checkpoints/consensus_holo_pocket/primary_organic/batch_004/batch_004_ligand_centered_pocket_residues.xlsx
  ✅ P2Rank

,Batch_ID,Status,Consensus
0,batch_004,completed,/content/drive/MyDrive/Horizyn_Checkpoints/con...
1,batch_005,completed,/content/drive/MyDrive/Horizyn_Checkpoints/con...


In [54]:
# ============================================================
# 合并当前已完成 batch 的 consensus summary 并做 QC
# ============================================================

from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

summary_files = sorted(
    (BASE / "consensus_holo_pocket/primary_organic").rglob(
        "batch_*_consensus_holo_pocket_summary_conservative.xlsx"
    )
)

print("已完成 batch summary 数量:", len(summary_files))

dfs = []

for f in summary_files:
    df = pd.read_excel(f)
    df["Summary_File"] = str(f)
    dfs.append(df)

if len(dfs) == 0:
    raise FileNotFoundError("没有找到任何 batch consensus summary。")

df_all = pd.concat(dfs, ignore_index=True)

out_xlsx = BASE / "consensus_holo_pocket/primary_organic/primary_organic_all_consensus_holo_pocket_summary_current.xlsx"
df_all.to_excel(out_xlsx, index=False)

print("✅ 当前已完成 batch 合并表:")
print(out_xlsx)

print("\nConsensus class 总统计:")
print(df_all["Consensus_Holo_Pocket_Class"].value_counts(dropna=False))

print("\nConservative action 总统计:")
print(df_all["Recommended_Next_Action_Conservative"].value_counts(dropna=False))

print("\n按 batch 统计:")
display(
    df_all.groupby(["Batch_ID", "Consensus_Holo_Pocket_Class"])
    .size()
    .reset_index(name="count")
    .pivot(index="Batch_ID", columns="Consensus_Holo_Pocket_Class", values="count")
    .fillna(0)
)

display(
    df_all[[
        "Batch_ID",
        "PDB_File",
        "confidence_score",
        "P2Rank_Hit_Class",
        "fpocket_Hit_Class",
        "Consensus_Holo_Pocket_Score",
        "Consensus_Holo_Pocket_Class",
        "Recommended_Next_Action_Conservative"
    ]]
    .sort_values(["Batch_ID", "Consensus_Holo_Pocket_Score"], ascending=[True, False])
    .head(40)
)

已完成 batch summary 数量: 5
✅ 当前已完成 batch 合并表:
/content/drive/MyDrive/Horizyn_Checkpoints/consensus_holo_pocket/primary_organic/primary_organic_all_consensus_holo_pocket_summary_current.xlsx

Consensus class 总统计:
Consensus_Holo_Pocket_Class
High_confidence_holo_pocket    79
Probable_holo_pocket           14
Uncertain_manual_check          6
Low_confidence_or_failed        1
Name: count, dtype: int64

Conservative action 总统计:
Recommended_Next_Action_Conservative
Proceed_to_substrate_docking_and_EZSpecificity    79
Proceed_but_keep_manual_inspection                13
Manual_inspection_or_rerun_Boltz_multiseed         8
Name: count, dtype: int64

按 batch 统计:


Consensus_Holo_Pocket_Class,High_confidence_holo_pocket,Low_confidence_or_failed,Probable_holo_pocket,Uncertain_manual_check
Batch_ID,,,,
batch_001,14.0,0.0,4.0,2.0
batch_002,16.0,1.0,3.0,0.0
batch_003,15.0,0.0,3.0,2.0
batch_004,16.0,0.0,2.0,2.0
batch_005,18.0,0.0,2.0,0.0


,Batch_ID,PDB_File,confidence_score,P2Rank_Hit_Class,fpocket_Hit_Class,Consensus_Holo_Pocket_Score,Consensus_Holo_Pocket_Class,Recommended_Next_Action_Conservative
0,batch_001,NODE_11_length_154862_cov_66.368492_122__Heme_...,0.891467,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
2,batch_001,NODE_13_length_131311_cov_66.505852_95__FMN__r...,0.878142,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
4,batch_001,NODE_1_length_436095_cov_65.793628_253__FMN__r...,0.948654,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
13,batch_001,NODE_25_length_76722_cov_66.604332_89__NADP___...,0.877610,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
14,batch_001,NODE_26_length_70757_cov_67.122722_30__NAD___r...,0.896448,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
16,batch_001,NODE_32_length_49516_cov_65.241935_22__FMN__ra...,0.960423,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
1,batch_001,NODE_12_length_136290_cov_66.430554_166__Heme_...,0.843068,Strong_<=6A,Strong_<=6A,0.970,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
18,batch_001,NODE_6_length_231008_cov_66.606731_103__FAD__r...,0.622942,Strong_<=6A,Strong_<=6A,0.940,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
9,batch_001,NODE_21_length_97532_cov_65.343769_69__NAD___r...,0.886403,Strong_<=6A,Moderate_6_10A,0.938,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
11,batch_001,NODE_25_length_76722_cov_66.604332_6__NAD___ra...,0.893232,Strong_<=6A,Moderate_6_10A,0.938,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity


In [55]:
# ============================================================
# 续跑 Primary organic/heme batches
# 当前已完成 batch_001–batch_005
# 本轮运行 batch_006–batch_007
# ============================================================

from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

START_BATCH = 6
MAX_BATCHES_THIS_SESSION = 2
FORCE_RERUN = False

BATCH_MANIFEST = BASE / "boltz_inputs/primary_organic_batches_manifest.xlsx"
CONSENSUS_ROOT = BASE / "consensus_holo_pocket/primary_organic"

# 检查 run_one_batch 是否已经定义
if "run_one_batch" not in globals():
    raise RuntimeError(
        "当前 runtime 里没有 run_one_batch()。请先运行前面那个完整 Master Auto-Runner 定义 Cell，"
        "然后再运行这个续跑 Cell。"
    )

df_batches = pd.read_excel(BATCH_MANIFEST)

all_batch_ids = sorted(df_batches["Batch_ID"].dropna().unique().tolist())
all_batch_nums = [(bid, int(str(bid).split("_")[1])) for bid in all_batch_ids]

# 找到未完成 batch
finished = []
for bid in all_batch_ids:
    final_xlsx = CONSENSUS_ROOT / bid / f"{bid}_consensus_holo_pocket_summary_conservative.xlsx"
    if final_xlsx.exists():
        finished.append(bid)

unfinished = [bid for bid in all_batch_ids if bid not in finished]

selected = [
    bid for bid, n in all_batch_nums
    if n >= START_BATCH and bid in unfinished
][:MAX_BATCHES_THIS_SESSION]

print("总 batch 数:", len(all_batch_ids))
print("已完成:", len(finished), finished)
print("未完成:", len(unfinished), unfinished[:20])
print("本轮将运行:", selected)

if not selected:
    raise ValueError("没有需要运行的 batch。可能全部完成，或 START_BATCH 设置过大。")

run_records = []

for batch_id in selected:
    record = run_one_batch(batch_id)
    run_records.append(record)

df_run = pd.DataFrame(run_records)

run_summary = BASE / f"consensus_holo_pocket/primary_organic/autorun_summary_start{START_BATCH}_n{MAX_BATCHES_THIS_SESSION}.xlsx"
run_summary.parent.mkdir(parents=True, exist_ok=True)
df_run.to_excel(run_summary, index=False)

print("\n✅ 本轮 batch 自动运行完成")
print(run_summary)

display(df_run)

总 batch 数: 19
已完成: 5 ['batch_001', 'batch_002', 'batch_003', 'batch_004', 'batch_005']
未完成: 14 ['batch_006', 'batch_007', 'batch_008', 'batch_009', 'batch_010', 'batch_011', 'batch_012', 'batch_013', 'batch_014', 'batch_015', 'batch_016', 'batch_017', 'batch_018', 'batch_019']
本轮将运行: ['batch_006', 'batch_007']

🚀 Running batch_006
  YAML 数量: 20
  Running Boltz...
  ✅ Boltz 完成: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_006
  ✅ confidence summary: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_006_confidence_summary.xlsx
  PDB: 20, confidence JSON: 20
  ✅ ligand check: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_006_ligand_check.xlsx
Has_Protein  Has_Ligand
True         True          20
Name: count, dtype: int64
  ✅ ligand-centered pocket: /content/drive/MyDrive/Horizyn_Checkpoints/consensus_holo_pocket/primary_organic/batch_006/batch_006_ligand_centered_pocket_residues.xlsx
  ✅ P2Rank

,Batch_ID,Status,Consensus
0,batch_006,completed,/content/drive/MyDrive/Horizyn_Checkpoints/con...
1,batch_007,completed,/content/drive/MyDrive/Horizyn_Checkpoints/con...


In [56]:
# ============================================================
# 合并当前已完成 batch 的 consensus summary 并做 QC
# ============================================================

from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

summary_files = sorted(
    (BASE / "consensus_holo_pocket/primary_organic").rglob(
        "batch_*_consensus_holo_pocket_summary_conservative.xlsx"
    )
)

print("已完成 batch summary 数量:", len(summary_files))

dfs = []

for f in summary_files:
    df = pd.read_excel(f)
    df["Summary_File"] = str(f)
    dfs.append(df)

if len(dfs) == 0:
    raise FileNotFoundError("没有找到任何 batch consensus summary。")

df_all = pd.concat(dfs, ignore_index=True)

out_xlsx = BASE / "consensus_holo_pocket/primary_organic/primary_organic_all_consensus_holo_pocket_summary_current.xlsx"
df_all.to_excel(out_xlsx, index=False)

print("✅ 当前已完成 batch 合并表:")
print(out_xlsx)

print("\nConsensus class 总统计:")
print(df_all["Consensus_Holo_Pocket_Class"].value_counts(dropna=False))

print("\nConservative action 总统计:")
print(df_all["Recommended_Next_Action_Conservative"].value_counts(dropna=False))

print("\n按 batch 统计:")
display(
    df_all.groupby(["Batch_ID", "Consensus_Holo_Pocket_Class"])
    .size()
    .reset_index(name="count")
    .pivot(index="Batch_ID", columns="Consensus_Holo_Pocket_Class", values="count")
    .fillna(0)
)

display(
    df_all[[
        "Batch_ID",
        "PDB_File",
        "confidence_score",
        "P2Rank_Hit_Class",
        "fpocket_Hit_Class",
        "Consensus_Holo_Pocket_Score",
        "Consensus_Holo_Pocket_Class",
        "Recommended_Next_Action_Conservative"
    ]]
    .sort_values(["Batch_ID", "Consensus_Holo_Pocket_Score"], ascending=[True, False])
    .head(40)
)

已完成 batch summary 数量: 7
✅ 当前已完成 batch 合并表:
/content/drive/MyDrive/Horizyn_Checkpoints/consensus_holo_pocket/primary_organic/primary_organic_all_consensus_holo_pocket_summary_current.xlsx

Consensus class 总统计:
Consensus_Holo_Pocket_Class
High_confidence_holo_pocket    116
Probable_holo_pocket            17
Uncertain_manual_check           6
Low_confidence_or_failed         1
Name: count, dtype: int64

Conservative action 总统计:
Recommended_Next_Action_Conservative
Proceed_to_substrate_docking_and_EZSpecificity    116
Proceed_but_keep_manual_inspection                 16
Manual_inspection_or_rerun_Boltz_multiseed          8
Name: count, dtype: int64

按 batch 统计:


Consensus_Holo_Pocket_Class,High_confidence_holo_pocket,Low_confidence_or_failed,Probable_holo_pocket,Uncertain_manual_check
Batch_ID,,,,
batch_001,14.0,0.0,4.0,2.0
batch_002,16.0,1.0,3.0,0.0
batch_003,15.0,0.0,3.0,2.0
batch_004,16.0,0.0,2.0,2.0
batch_005,18.0,0.0,2.0,0.0
batch_006,19.0,0.0,1.0,0.0
batch_007,18.0,0.0,2.0,0.0


,Batch_ID,PDB_File,confidence_score,P2Rank_Hit_Class,fpocket_Hit_Class,Consensus_Holo_Pocket_Score,Consensus_Holo_Pocket_Class,Recommended_Next_Action_Conservative
0,batch_001,NODE_11_length_154862_cov_66.368492_122__Heme_...,0.891467,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
2,batch_001,NODE_13_length_131311_cov_66.505852_95__FMN__r...,0.878142,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
4,batch_001,NODE_1_length_436095_cov_65.793628_253__FMN__r...,0.948654,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
13,batch_001,NODE_25_length_76722_cov_66.604332_89__NADP___...,0.877610,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
14,batch_001,NODE_26_length_70757_cov_67.122722_30__NAD___r...,0.896448,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
16,batch_001,NODE_32_length_49516_cov_65.241935_22__FMN__ra...,0.960423,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
1,batch_001,NODE_12_length_136290_cov_66.430554_166__Heme_...,0.843068,Strong_<=6A,Strong_<=6A,0.970,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
18,batch_001,NODE_6_length_231008_cov_66.606731_103__FAD__r...,0.622942,Strong_<=6A,Strong_<=6A,0.940,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
9,batch_001,NODE_21_length_97532_cov_65.343769_69__NAD___r...,0.886403,Strong_<=6A,Moderate_6_10A,0.938,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
11,batch_001,NODE_25_length_76722_cov_66.604332_6__NAD___ra...,0.893232,Strong_<=6A,Moderate_6_10A,0.938,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity


In [57]:
# ============================================================
# 续跑 Primary organic/heme batches
# 当前已完成 batch_001–batch_007
# 本轮运行 batch_008–batch_009
# ============================================================

from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

START_BATCH = 8
MAX_BATCHES_THIS_SESSION = 2
FORCE_RERUN = False

BATCH_MANIFEST = BASE / "boltz_inputs/primary_organic_batches_manifest.xlsx"
CONSENSUS_ROOT = BASE / "consensus_holo_pocket/primary_organic"

# 检查 run_one_batch 是否已经定义
if "run_one_batch" not in globals():
    raise RuntimeError(
        "当前 runtime 里没有 run_one_batch()。请先运行前面那个完整 Master Auto-Runner 定义 Cell，"
        "然后再运行这个续跑 Cell。"
    )

df_batches = pd.read_excel(BATCH_MANIFEST)

all_batch_ids = sorted(df_batches["Batch_ID"].dropna().unique().tolist())
all_batch_nums = [(bid, int(str(bid).split("_")[1])) for bid in all_batch_ids]

# 找到未完成 batch
finished = []
for bid in all_batch_ids:
    final_xlsx = CONSENSUS_ROOT / bid / f"{bid}_consensus_holo_pocket_summary_conservative.xlsx"
    if final_xlsx.exists():
        finished.append(bid)

unfinished = [bid for bid in all_batch_ids if bid not in finished]

selected = [
    bid for bid, n in all_batch_nums
    if n >= START_BATCH and bid in unfinished
][:MAX_BATCHES_THIS_SESSION]

print("总 batch 数:", len(all_batch_ids))
print("已完成:", len(finished), finished)
print("未完成:", len(unfinished), unfinished[:20])
print("本轮将运行:", selected)

if not selected:
    raise ValueError("没有需要运行的 batch。可能全部完成，或 START_BATCH 设置过大。")

run_records = []

for batch_id in selected:
    record = run_one_batch(batch_id)
    run_records.append(record)

df_run = pd.DataFrame(run_records)

run_summary = BASE / f"consensus_holo_pocket/primary_organic/autorun_summary_start{START_BATCH}_n{MAX_BATCHES_THIS_SESSION}.xlsx"
run_summary.parent.mkdir(parents=True, exist_ok=True)
df_run.to_excel(run_summary, index=False)

print("\n✅ 本轮 batch 自动运行完成")
print(run_summary)

display(df_run)

总 batch 数: 19
已完成: 7 ['batch_001', 'batch_002', 'batch_003', 'batch_004', 'batch_005', 'batch_006', 'batch_007']
未完成: 12 ['batch_008', 'batch_009', 'batch_010', 'batch_011', 'batch_012', 'batch_013', 'batch_014', 'batch_015', 'batch_016', 'batch_017', 'batch_018', 'batch_019']
本轮将运行: ['batch_008', 'batch_009']

🚀 Running batch_008
  YAML 数量: 20
  Running Boltz...
  ✅ Boltz 完成: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_008
  ✅ confidence summary: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_008_confidence_summary.xlsx
  PDB: 20, confidence JSON: 20
  ✅ ligand check: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_008_ligand_check.xlsx
Has_Protein  Has_Ligand
True         True          20
Name: count, dtype: int64
  ✅ ligand-centered pocket: /content/drive/MyDrive/Horizyn_Checkpoints/consensus_holo_pocket/primary_organic/batch_008/batch_008_ligand_centered_pocket_residues.xlsx
  ✅ P2Rank

,Batch_ID,Status,Consensus
0,batch_008,completed,/content/drive/MyDrive/Horizyn_Checkpoints/con...
1,batch_009,completed,/content/drive/MyDrive/Horizyn_Checkpoints/con...


In [58]:
# ============================================================
# 合并当前已完成 batch 的 consensus summary 并做 QC
# ============================================================

from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

summary_files = sorted(
    (BASE / "consensus_holo_pocket/primary_organic").rglob(
        "batch_*_consensus_holo_pocket_summary_conservative.xlsx"
    )
)

print("已完成 batch summary 数量:", len(summary_files))

dfs = []

for f in summary_files:
    df = pd.read_excel(f)
    df["Summary_File"] = str(f)
    dfs.append(df)

if len(dfs) == 0:
    raise FileNotFoundError("没有找到任何 batch consensus summary。")

df_all = pd.concat(dfs, ignore_index=True)

out_xlsx = BASE / "consensus_holo_pocket/primary_organic/primary_organic_all_consensus_holo_pocket_summary_current.xlsx"
df_all.to_excel(out_xlsx, index=False)

print("✅ 当前已完成 batch 合并表:")
print(out_xlsx)

print("\nConsensus class 总统计:")
print(df_all["Consensus_Holo_Pocket_Class"].value_counts(dropna=False))

print("\nConservative action 总统计:")
print(df_all["Recommended_Next_Action_Conservative"].value_counts(dropna=False))

print("\n按 batch 统计:")
display(
    df_all.groupby(["Batch_ID", "Consensus_Holo_Pocket_Class"])
    .size()
    .reset_index(name="count")
    .pivot(index="Batch_ID", columns="Consensus_Holo_Pocket_Class", values="count")
    .fillna(0)
)

display(
    df_all[[
        "Batch_ID",
        "PDB_File",
        "confidence_score",
        "P2Rank_Hit_Class",
        "fpocket_Hit_Class",
        "Consensus_Holo_Pocket_Score",
        "Consensus_Holo_Pocket_Class",
        "Recommended_Next_Action_Conservative"
    ]]
    .sort_values(["Batch_ID", "Consensus_Holo_Pocket_Score"], ascending=[True, False])
    .head(40)
)

已完成 batch summary 数量: 9
✅ 当前已完成 batch 合并表:
/content/drive/MyDrive/Horizyn_Checkpoints/consensus_holo_pocket/primary_organic/primary_organic_all_consensus_holo_pocket_summary_current.xlsx

Consensus class 总统计:
Consensus_Holo_Pocket_Class
High_confidence_holo_pocket    151
Probable_holo_pocket            22
Uncertain_manual_check           6
Low_confidence_or_failed         1
Name: count, dtype: int64

Conservative action 总统计:
Recommended_Next_Action_Conservative
Proceed_to_substrate_docking_and_EZSpecificity    151
Proceed_but_keep_manual_inspection                 21
Manual_inspection_or_rerun_Boltz_multiseed          8
Name: count, dtype: int64

按 batch 统计:


Consensus_Holo_Pocket_Class,High_confidence_holo_pocket,Low_confidence_or_failed,Probable_holo_pocket,Uncertain_manual_check
Batch_ID,,,,
batch_001,14.0,0.0,4.0,2.0
batch_002,16.0,1.0,3.0,0.0
batch_003,15.0,0.0,3.0,2.0
batch_004,16.0,0.0,2.0,2.0
batch_005,18.0,0.0,2.0,0.0
batch_006,19.0,0.0,1.0,0.0
batch_007,18.0,0.0,2.0,0.0
batch_008,17.0,0.0,3.0,0.0
batch_009,18.0,0.0,2.0,0.0


,Batch_ID,PDB_File,confidence_score,P2Rank_Hit_Class,fpocket_Hit_Class,Consensus_Holo_Pocket_Score,Consensus_Holo_Pocket_Class,Recommended_Next_Action_Conservative
0,batch_001,NODE_11_length_154862_cov_66.368492_122__Heme_...,0.891467,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
2,batch_001,NODE_13_length_131311_cov_66.505852_95__FMN__r...,0.878142,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
4,batch_001,NODE_1_length_436095_cov_65.793628_253__FMN__r...,0.948654,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
13,batch_001,NODE_25_length_76722_cov_66.604332_89__NADP___...,0.877610,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
14,batch_001,NODE_26_length_70757_cov_67.122722_30__NAD___r...,0.896448,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
16,batch_001,NODE_32_length_49516_cov_65.241935_22__FMN__ra...,0.960423,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
1,batch_001,NODE_12_length_136290_cov_66.430554_166__Heme_...,0.843068,Strong_<=6A,Strong_<=6A,0.970,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
18,batch_001,NODE_6_length_231008_cov_66.606731_103__FAD__r...,0.622942,Strong_<=6A,Strong_<=6A,0.940,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
9,batch_001,NODE_21_length_97532_cov_65.343769_69__NAD___r...,0.886403,Strong_<=6A,Moderate_6_10A,0.938,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
11,batch_001,NODE_25_length_76722_cov_66.604332_6__NAD___ra...,0.893232,Strong_<=6A,Moderate_6_10A,0.938,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity


In [59]:
# ============================================================
# 续跑 Primary organic/heme batches
# 当前已完成 batch_001–batch_009
# 本轮运行 batch_010–batch_011
# ============================================================

from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

START_BATCH = 10
MAX_BATCHES_THIS_SESSION = 2
FORCE_RERUN = False

BATCH_MANIFEST = BASE / "boltz_inputs/primary_organic_batches_manifest.xlsx"
CONSENSUS_ROOT = BASE / "consensus_holo_pocket/primary_organic"

# 检查 run_one_batch 是否已经定义
if "run_one_batch" not in globals():
    raise RuntimeError(
        "当前 runtime 里没有 run_one_batch()。请先运行前面那个完整 Master Auto-Runner 定义 Cell，"
        "然后再运行这个续跑 Cell。"
    )

df_batches = pd.read_excel(BATCH_MANIFEST)

all_batch_ids = sorted(df_batches["Batch_ID"].dropna().unique().tolist())
all_batch_nums = [(bid, int(str(bid).split("_")[1])) for bid in all_batch_ids]

# 找到未完成 batch
finished = []
for bid in all_batch_ids:
    final_xlsx = CONSENSUS_ROOT / bid / f"{bid}_consensus_holo_pocket_summary_conservative.xlsx"
    if final_xlsx.exists():
        finished.append(bid)

unfinished = [bid for bid in all_batch_ids if bid not in finished]

selected = [
    bid for bid, n in all_batch_nums
    if n >= START_BATCH and bid in unfinished
][:MAX_BATCHES_THIS_SESSION]

print("总 batch 数:", len(all_batch_ids))
print("已完成:", len(finished), finished)
print("未完成:", len(unfinished), unfinished[:20])
print("本轮将运行:", selected)

if not selected:
    raise ValueError("没有需要运行的 batch。可能全部完成，或 START_BATCH 设置过大。")

run_records = []

for batch_id in selected:
    record = run_one_batch(batch_id)
    run_records.append(record)

df_run = pd.DataFrame(run_records)

run_summary = BASE / f"consensus_holo_pocket/primary_organic/autorun_summary_start{START_BATCH}_n{MAX_BATCHES_THIS_SESSION}.xlsx"
run_summary.parent.mkdir(parents=True, exist_ok=True)
df_run.to_excel(run_summary, index=False)

print("\n✅ 本轮 batch 自动运行完成")
print(run_summary)

display(df_run)

总 batch 数: 19
已完成: 9 ['batch_001', 'batch_002', 'batch_003', 'batch_004', 'batch_005', 'batch_006', 'batch_007', 'batch_008', 'batch_009']
未完成: 10 ['batch_010', 'batch_011', 'batch_012', 'batch_013', 'batch_014', 'batch_015', 'batch_016', 'batch_017', 'batch_018', 'batch_019']
本轮将运行: ['batch_010', 'batch_011']

🚀 Running batch_010
  YAML 数量: 20
  Running Boltz...
  ✅ Boltz 完成: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_010
  ✅ confidence summary: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_010_confidence_summary.xlsx
  PDB: 20, confidence JSON: 20
  ✅ ligand check: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_010_ligand_check.xlsx
Has_Protein  Has_Ligand
True         True          20
Name: count, dtype: int64
  ✅ ligand-centered pocket: /content/drive/MyDrive/Horizyn_Checkpoints/consensus_holo_pocket/primary_organic/batch_010/batch_010_ligand_centered_pocket_residues.xlsx
  ✅ P2Rank

,Batch_ID,Status,Consensus
0,batch_010,completed,/content/drive/MyDrive/Horizyn_Checkpoints/con...
1,batch_011,completed,/content/drive/MyDrive/Horizyn_Checkpoints/con...


In [60]:
# ============================================================
# 合并当前已完成 batch 的 consensus summary 并做 QC
# ============================================================

from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

summary_files = sorted(
    (BASE / "consensus_holo_pocket/primary_organic").rglob(
        "batch_*_consensus_holo_pocket_summary_conservative.xlsx"
    )
)

print("已完成 batch summary 数量:", len(summary_files))

dfs = []

for f in summary_files:
    df = pd.read_excel(f)
    df["Summary_File"] = str(f)
    dfs.append(df)

if len(dfs) == 0:
    raise FileNotFoundError("没有找到任何 batch consensus summary。")

df_all = pd.concat(dfs, ignore_index=True)

out_xlsx = BASE / "consensus_holo_pocket/primary_organic/primary_organic_all_consensus_holo_pocket_summary_current.xlsx"
df_all.to_excel(out_xlsx, index=False)

print("✅ 当前已完成 batch 合并表:")
print(out_xlsx)

print("\nConsensus class 总统计:")
print(df_all["Consensus_Holo_Pocket_Class"].value_counts(dropna=False))

print("\nConservative action 总统计:")
print(df_all["Recommended_Next_Action_Conservative"].value_counts(dropna=False))

print("\n按 batch 统计:")
display(
    df_all.groupby(["Batch_ID", "Consensus_Holo_Pocket_Class"])
    .size()
    .reset_index(name="count")
    .pivot(index="Batch_ID", columns="Consensus_Holo_Pocket_Class", values="count")
    .fillna(0)
)

display(
    df_all[[
        "Batch_ID",
        "PDB_File",
        "confidence_score",
        "P2Rank_Hit_Class",
        "fpocket_Hit_Class",
        "Consensus_Holo_Pocket_Score",
        "Consensus_Holo_Pocket_Class",
        "Recommended_Next_Action_Conservative"
    ]]
    .sort_values(["Batch_ID", "Consensus_Holo_Pocket_Score"], ascending=[True, False])
    .head(40)
)

已完成 batch summary 数量: 11
✅ 当前已完成 batch 合并表:
/content/drive/MyDrive/Horizyn_Checkpoints/consensus_holo_pocket/primary_organic/primary_organic_all_consensus_holo_pocket_summary_current.xlsx

Consensus class 总统计:
Consensus_Holo_Pocket_Class
High_confidence_holo_pocket    187
Probable_holo_pocket            26
Uncertain_manual_check           6
Low_confidence_or_failed         1
Name: count, dtype: int64

Conservative action 总统计:
Recommended_Next_Action_Conservative
Proceed_to_substrate_docking_and_EZSpecificity    187
Proceed_but_keep_manual_inspection                 25
Manual_inspection_or_rerun_Boltz_multiseed          8
Name: count, dtype: int64

按 batch 统计:


Consensus_Holo_Pocket_Class,High_confidence_holo_pocket,Low_confidence_or_failed,Probable_holo_pocket,Uncertain_manual_check
Batch_ID,,,,
batch_001,14.0,0.0,4.0,2.0
batch_002,16.0,1.0,3.0,0.0
batch_003,15.0,0.0,3.0,2.0
batch_004,16.0,0.0,2.0,2.0
batch_005,18.0,0.0,2.0,0.0
batch_006,19.0,0.0,1.0,0.0
batch_007,18.0,0.0,2.0,0.0
batch_008,17.0,0.0,3.0,0.0
batch_009,18.0,0.0,2.0,0.0


,Batch_ID,PDB_File,confidence_score,P2Rank_Hit_Class,fpocket_Hit_Class,Consensus_Holo_Pocket_Score,Consensus_Holo_Pocket_Class,Recommended_Next_Action_Conservative
0,batch_001,NODE_11_length_154862_cov_66.368492_122__Heme_...,0.891467,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
2,batch_001,NODE_13_length_131311_cov_66.505852_95__FMN__r...,0.878142,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
4,batch_001,NODE_1_length_436095_cov_65.793628_253__FMN__r...,0.948654,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
13,batch_001,NODE_25_length_76722_cov_66.604332_89__NADP___...,0.877610,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
14,batch_001,NODE_26_length_70757_cov_67.122722_30__NAD___r...,0.896448,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
16,batch_001,NODE_32_length_49516_cov_65.241935_22__FMN__ra...,0.960423,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
1,batch_001,NODE_12_length_136290_cov_66.430554_166__Heme_...,0.843068,Strong_<=6A,Strong_<=6A,0.970,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
18,batch_001,NODE_6_length_231008_cov_66.606731_103__FAD__r...,0.622942,Strong_<=6A,Strong_<=6A,0.940,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
9,batch_001,NODE_21_length_97532_cov_65.343769_69__NAD___r...,0.886403,Strong_<=6A,Moderate_6_10A,0.938,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
11,batch_001,NODE_25_length_76722_cov_66.604332_6__NAD___ra...,0.893232,Strong_<=6A,Moderate_6_10A,0.938,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity


In [61]:
# ============================================================
# 续跑 Primary organic/heme batches
# 当前已完成 batch_001–batch_011
# 本轮运行 batch_012–batch_0013
# ============================================================

from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

START_BATCH = 12
MAX_BATCHES_THIS_SESSION = 2
FORCE_RERUN = False

BATCH_MANIFEST = BASE / "boltz_inputs/primary_organic_batches_manifest.xlsx"
CONSENSUS_ROOT = BASE / "consensus_holo_pocket/primary_organic"

# 检查 run_one_batch 是否已经定义
if "run_one_batch" not in globals():
    raise RuntimeError(
        "当前 runtime 里没有 run_one_batch()。请先运行前面那个完整 Master Auto-Runner 定义 Cell，"
        "然后再运行这个续跑 Cell。"
    )

df_batches = pd.read_excel(BATCH_MANIFEST)

all_batch_ids = sorted(df_batches["Batch_ID"].dropna().unique().tolist())
all_batch_nums = [(bid, int(str(bid).split("_")[1])) for bid in all_batch_ids]

# 找到未完成 batch
finished = []
for bid in all_batch_ids:
    final_xlsx = CONSENSUS_ROOT / bid / f"{bid}_consensus_holo_pocket_summary_conservative.xlsx"
    if final_xlsx.exists():
        finished.append(bid)

unfinished = [bid for bid in all_batch_ids if bid not in finished]

selected = [
    bid for bid, n in all_batch_nums
    if n >= START_BATCH and bid in unfinished
][:MAX_BATCHES_THIS_SESSION]

print("总 batch 数:", len(all_batch_ids))
print("已完成:", len(finished), finished)
print("未完成:", len(unfinished), unfinished[:20])
print("本轮将运行:", selected)

if not selected:
    raise ValueError("没有需要运行的 batch。可能全部完成，或 START_BATCH 设置过大。")

run_records = []

for batch_id in selected:
    record = run_one_batch(batch_id)
    run_records.append(record)

df_run = pd.DataFrame(run_records)

run_summary = BASE / f"consensus_holo_pocket/primary_organic/autorun_summary_start{START_BATCH}_n{MAX_BATCHES_THIS_SESSION}.xlsx"
run_summary.parent.mkdir(parents=True, exist_ok=True)
df_run.to_excel(run_summary, index=False)

print("\n✅ 本轮 batch 自动运行完成")
print(run_summary)

display(df_run)

总 batch 数: 19
已完成: 11 ['batch_001', 'batch_002', 'batch_003', 'batch_004', 'batch_005', 'batch_006', 'batch_007', 'batch_008', 'batch_009', 'batch_010', 'batch_011']
未完成: 8 ['batch_012', 'batch_013', 'batch_014', 'batch_015', 'batch_016', 'batch_017', 'batch_018', 'batch_019']
本轮将运行: ['batch_012', 'batch_013']

🚀 Running batch_012
  YAML 数量: 20
  Running Boltz...
  ✅ Boltz 完成: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_012
  ✅ confidence summary: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_012_confidence_summary.xlsx
  PDB: 20, confidence JSON: 20
  ✅ ligand check: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_012_ligand_check.xlsx
Has_Protein  Has_Ligand
True         True          20
Name: count, dtype: int64
  ✅ ligand-centered pocket: /content/drive/MyDrive/Horizyn_Checkpoints/consensus_holo_pocket/primary_organic/batch_012/batch_012_ligand_centered_pocket_residues.xlsx
  ✅ P2Rank

,Batch_ID,Status,Consensus
0,batch_012,completed,/content/drive/MyDrive/Horizyn_Checkpoints/con...
1,batch_013,completed,/content/drive/MyDrive/Horizyn_Checkpoints/con...


In [62]:
# ============================================================
# 合并当前已完成 batch 的 consensus summary 并做 QC
# ============================================================

from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

summary_files = sorted(
    (BASE / "consensus_holo_pocket/primary_organic").rglob(
        "batch_*_consensus_holo_pocket_summary_conservative.xlsx"
    )
)

print("已完成 batch summary 数量:", len(summary_files))

dfs = []

for f in summary_files:
    df = pd.read_excel(f)
    df["Summary_File"] = str(f)
    dfs.append(df)

if len(dfs) == 0:
    raise FileNotFoundError("没有找到任何 batch consensus summary。")

df_all = pd.concat(dfs, ignore_index=True)

out_xlsx = BASE / "consensus_holo_pocket/primary_organic/primary_organic_all_consensus_holo_pocket_summary_current.xlsx"
df_all.to_excel(out_xlsx, index=False)

print("✅ 当前已完成 batch 合并表:")
print(out_xlsx)

print("\nConsensus class 总统计:")
print(df_all["Consensus_Holo_Pocket_Class"].value_counts(dropna=False))

print("\nConservative action 总统计:")
print(df_all["Recommended_Next_Action_Conservative"].value_counts(dropna=False))

print("\n按 batch 统计:")
display(
    df_all.groupby(["Batch_ID", "Consensus_Holo_Pocket_Class"])
    .size()
    .reset_index(name="count")
    .pivot(index="Batch_ID", columns="Consensus_Holo_Pocket_Class", values="count")
    .fillna(0)
)

display(
    df_all[[
        "Batch_ID",
        "PDB_File",
        "confidence_score",
        "P2Rank_Hit_Class",
        "fpocket_Hit_Class",
        "Consensus_Holo_Pocket_Score",
        "Consensus_Holo_Pocket_Class",
        "Recommended_Next_Action_Conservative"
    ]]
    .sort_values(["Batch_ID", "Consensus_Holo_Pocket_Score"], ascending=[True, False])
    .head(40)
)

已完成 batch summary 数量: 13
✅ 当前已完成 batch 合并表:
/content/drive/MyDrive/Horizyn_Checkpoints/consensus_holo_pocket/primary_organic/primary_organic_all_consensus_holo_pocket_summary_current.xlsx

Consensus class 总统计:
Consensus_Holo_Pocket_Class
High_confidence_holo_pocket    224
Probable_holo_pocket            29
Uncertain_manual_check           6
Low_confidence_or_failed         1
Name: count, dtype: int64

Conservative action 总统计:
Recommended_Next_Action_Conservative
Proceed_to_substrate_docking_and_EZSpecificity    224
Proceed_but_keep_manual_inspection                 28
Manual_inspection_or_rerun_Boltz_multiseed          8
Name: count, dtype: int64

按 batch 统计:


Consensus_Holo_Pocket_Class,High_confidence_holo_pocket,Low_confidence_or_failed,Probable_holo_pocket,Uncertain_manual_check
Batch_ID,,,,
batch_001,14.0,0.0,4.0,2.0
batch_002,16.0,1.0,3.0,0.0
batch_003,15.0,0.0,3.0,2.0
batch_004,16.0,0.0,2.0,2.0
batch_005,18.0,0.0,2.0,0.0
batch_006,19.0,0.0,1.0,0.0
batch_007,18.0,0.0,2.0,0.0
batch_008,17.0,0.0,3.0,0.0
batch_009,18.0,0.0,2.0,0.0


,Batch_ID,PDB_File,confidence_score,P2Rank_Hit_Class,fpocket_Hit_Class,Consensus_Holo_Pocket_Score,Consensus_Holo_Pocket_Class,Recommended_Next_Action_Conservative
0,batch_001,NODE_11_length_154862_cov_66.368492_122__Heme_...,0.891467,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
2,batch_001,NODE_13_length_131311_cov_66.505852_95__FMN__r...,0.878142,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
4,batch_001,NODE_1_length_436095_cov_65.793628_253__FMN__r...,0.948654,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
13,batch_001,NODE_25_length_76722_cov_66.604332_89__NADP___...,0.877610,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
14,batch_001,NODE_26_length_70757_cov_67.122722_30__NAD___r...,0.896448,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
16,batch_001,NODE_32_length_49516_cov_65.241935_22__FMN__ra...,0.960423,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
1,batch_001,NODE_12_length_136290_cov_66.430554_166__Heme_...,0.843068,Strong_<=6A,Strong_<=6A,0.970,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
18,batch_001,NODE_6_length_231008_cov_66.606731_103__FAD__r...,0.622942,Strong_<=6A,Strong_<=6A,0.940,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
9,batch_001,NODE_21_length_97532_cov_65.343769_69__NAD___r...,0.886403,Strong_<=6A,Moderate_6_10A,0.938,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
11,batch_001,NODE_25_length_76722_cov_66.604332_6__NAD___ra...,0.893232,Strong_<=6A,Moderate_6_10A,0.938,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity


In [63]:
# ============================================================
# 续跑 Primary organic/heme batches
# 当前已完成 batch_001–batch_013
# 本轮运行 batch_014–batch_015
# ============================================================

from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

START_BATCH = 14
MAX_BATCHES_THIS_SESSION = 2
FORCE_RERUN = False

BATCH_MANIFEST = BASE / "boltz_inputs/primary_organic_batches_manifest.xlsx"
CONSENSUS_ROOT = BASE / "consensus_holo_pocket/primary_organic"

# 检查 run_one_batch 是否已经定义
if "run_one_batch" not in globals():
    raise RuntimeError(
        "当前 runtime 里没有 run_one_batch()。请先运行前面那个完整 Master Auto-Runner 定义 Cell，"
        "然后再运行这个续跑 Cell。"
    )

df_batches = pd.read_excel(BATCH_MANIFEST)

all_batch_ids = sorted(df_batches["Batch_ID"].dropna().unique().tolist())
all_batch_nums = [(bid, int(str(bid).split("_")[1])) for bid in all_batch_ids]

# 找到未完成 batch
finished = []
for bid in all_batch_ids:
    final_xlsx = CONSENSUS_ROOT / bid / f"{bid}_consensus_holo_pocket_summary_conservative.xlsx"
    if final_xlsx.exists():
        finished.append(bid)

unfinished = [bid for bid in all_batch_ids if bid not in finished]

selected = [
    bid for bid, n in all_batch_nums
    if n >= START_BATCH and bid in unfinished
][:MAX_BATCHES_THIS_SESSION]

print("总 batch 数:", len(all_batch_ids))
print("已完成:", len(finished), finished)
print("未完成:", len(unfinished), unfinished[:20])
print("本轮将运行:", selected)

if not selected:
    raise ValueError("没有需要运行的 batch。可能全部完成，或 START_BATCH 设置过大。")

run_records = []

for batch_id in selected:
    record = run_one_batch(batch_id)
    run_records.append(record)

df_run = pd.DataFrame(run_records)

run_summary = BASE / f"consensus_holo_pocket/primary_organic/autorun_summary_start{START_BATCH}_n{MAX_BATCHES_THIS_SESSION}.xlsx"
run_summary.parent.mkdir(parents=True, exist_ok=True)
df_run.to_excel(run_summary, index=False)

print("\n✅ 本轮 batch 自动运行完成")
print(run_summary)

display(df_run)

总 batch 数: 19
已完成: 13 ['batch_001', 'batch_002', 'batch_003', 'batch_004', 'batch_005', 'batch_006', 'batch_007', 'batch_008', 'batch_009', 'batch_010', 'batch_011', 'batch_012', 'batch_013']
未完成: 6 ['batch_014', 'batch_015', 'batch_016', 'batch_017', 'batch_018', 'batch_019']
本轮将运行: ['batch_014', 'batch_015']

🚀 Running batch_014
  YAML 数量: 20
  Running Boltz...
  ✅ Boltz 完成: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_014
  ✅ confidence summary: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_014_confidence_summary.xlsx
  PDB: 20, confidence JSON: 20
  ✅ ligand check: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_014_ligand_check.xlsx
Has_Protein  Has_Ligand
True         True          20
Name: count, dtype: int64
  ✅ ligand-centered pocket: /content/drive/MyDrive/Horizyn_Checkpoints/consensus_holo_pocket/primary_organic/batch_014/batch_014_ligand_centered_pocket_residues.xlsx
  ✅ P2Rank

,Batch_ID,Status,Consensus
0,batch_014,completed,/content/drive/MyDrive/Horizyn_Checkpoints/con...
1,batch_015,completed,/content/drive/MyDrive/Horizyn_Checkpoints/con...


In [65]:
# ============================================================
# 合并当前已完成 batch 的 consensus summary 并做 QC
# ============================================================

from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

summary_files = sorted(
    (BASE / "consensus_holo_pocket/primary_organic").rglob(
        "batch_*_consensus_holo_pocket_summary_conservative.xlsx"
    )
)

print("已完成 batch summary 数量:", len(summary_files))

dfs = []

for f in summary_files:
    df = pd.read_excel(f)
    df["Summary_File"] = str(f)
    dfs.append(df)

if len(dfs) == 0:
    raise FileNotFoundError("没有找到任何 batch consensus summary。")

df_all = pd.concat(dfs, ignore_index=True)

out_xlsx = BASE / "consensus_holo_pocket/primary_organic/primary_organic_all_consensus_holo_pocket_summary_current.xlsx"
df_all.to_excel(out_xlsx, index=False)

print("✅ 当前已完成 batch 合并表:")
print(out_xlsx)

print("\nConsensus class 总统计:")
print(df_all["Consensus_Holo_Pocket_Class"].value_counts(dropna=False))

print("\nConservative action 总统计:")
print(df_all["Recommended_Next_Action_Conservative"].value_counts(dropna=False))

print("\n按 batch 统计:")
display(
    df_all.groupby(["Batch_ID", "Consensus_Holo_Pocket_Class"])
    .size()
    .reset_index(name="count")
    .pivot(index="Batch_ID", columns="Consensus_Holo_Pocket_Class", values="count")
    .fillna(0)
)

display(
    df_all[[
        "Batch_ID",
        "PDB_File",
        "confidence_score",
        "P2Rank_Hit_Class",
        "fpocket_Hit_Class",
        "Consensus_Holo_Pocket_Score",
        "Consensus_Holo_Pocket_Class",
        "Recommended_Next_Action_Conservative"
    ]]
    .sort_values(["Batch_ID", "Consensus_Holo_Pocket_Score"], ascending=[True, False])
    .head(40)
)

已完成 batch summary 数量: 15
✅ 当前已完成 batch 合并表:
/content/drive/MyDrive/Horizyn_Checkpoints/consensus_holo_pocket/primary_organic/primary_organic_all_consensus_holo_pocket_summary_current.xlsx

Consensus class 总统计:
Consensus_Holo_Pocket_Class
High_confidence_holo_pocket    255
Probable_holo_pocket            38
Uncertain_manual_check           6
Low_confidence_or_failed         1
Name: count, dtype: int64

Conservative action 总统计:
Recommended_Next_Action_Conservative
Proceed_to_substrate_docking_and_EZSpecificity    255
Proceed_but_keep_manual_inspection                 37
Manual_inspection_or_rerun_Boltz_multiseed          8
Name: count, dtype: int64

按 batch 统计:


Consensus_Holo_Pocket_Class,High_confidence_holo_pocket,Low_confidence_or_failed,Probable_holo_pocket,Uncertain_manual_check
Batch_ID,,,,
batch_001,14.0,0.0,4.0,2.0
batch_002,16.0,1.0,3.0,0.0
batch_003,15.0,0.0,3.0,2.0
batch_004,16.0,0.0,2.0,2.0
batch_005,18.0,0.0,2.0,0.0
batch_006,19.0,0.0,1.0,0.0
batch_007,18.0,0.0,2.0,0.0
batch_008,17.0,0.0,3.0,0.0
batch_009,18.0,0.0,2.0,0.0


,Batch_ID,PDB_File,confidence_score,P2Rank_Hit_Class,fpocket_Hit_Class,Consensus_Holo_Pocket_Score,Consensus_Holo_Pocket_Class,Recommended_Next_Action_Conservative
0,batch_001,NODE_11_length_154862_cov_66.368492_122__Heme_...,0.891467,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
2,batch_001,NODE_13_length_131311_cov_66.505852_95__FMN__r...,0.878142,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
4,batch_001,NODE_1_length_436095_cov_65.793628_253__FMN__r...,0.948654,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
13,batch_001,NODE_25_length_76722_cov_66.604332_89__NADP___...,0.877610,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
14,batch_001,NODE_26_length_70757_cov_67.122722_30__NAD___r...,0.896448,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
16,batch_001,NODE_32_length_49516_cov_65.241935_22__FMN__ra...,0.960423,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
1,batch_001,NODE_12_length_136290_cov_66.430554_166__Heme_...,0.843068,Strong_<=6A,Strong_<=6A,0.970,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
18,batch_001,NODE_6_length_231008_cov_66.606731_103__FAD__r...,0.622942,Strong_<=6A,Strong_<=6A,0.940,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
9,batch_001,NODE_21_length_97532_cov_65.343769_69__NAD___r...,0.886403,Strong_<=6A,Moderate_6_10A,0.938,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
11,batch_001,NODE_25_length_76722_cov_66.604332_6__NAD___ra...,0.893232,Strong_<=6A,Moderate_6_10A,0.938,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity


In [66]:
# ============================================================
# 续跑 Primary organic/heme batches
# 当前已完成 batch_001–batch_015
# 本轮运行 batch_016–batch_017
# ============================================================

from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

START_BATCH = 16
MAX_BATCHES_THIS_SESSION = 2
FORCE_RERUN = False

BATCH_MANIFEST = BASE / "boltz_inputs/primary_organic_batches_manifest.xlsx"
CONSENSUS_ROOT = BASE / "consensus_holo_pocket/primary_organic"

# 检查 run_one_batch 是否已经定义
if "run_one_batch" not in globals():
    raise RuntimeError(
        "当前 runtime 里没有 run_one_batch()。请先运行前面那个完整 Master Auto-Runner 定义 Cell，"
        "然后再运行这个续跑 Cell。"
    )

df_batches = pd.read_excel(BATCH_MANIFEST)

all_batch_ids = sorted(df_batches["Batch_ID"].dropna().unique().tolist())
all_batch_nums = [(bid, int(str(bid).split("_")[1])) for bid in all_batch_ids]

# 找到未完成 batch
finished = []
for bid in all_batch_ids:
    final_xlsx = CONSENSUS_ROOT / bid / f"{bid}_consensus_holo_pocket_summary_conservative.xlsx"
    if final_xlsx.exists():
        finished.append(bid)

unfinished = [bid for bid in all_batch_ids if bid not in finished]

selected = [
    bid for bid, n in all_batch_nums
    if n >= START_BATCH and bid in unfinished
][:MAX_BATCHES_THIS_SESSION]

print("总 batch 数:", len(all_batch_ids))
print("已完成:", len(finished), finished)
print("未完成:", len(unfinished), unfinished[:20])
print("本轮将运行:", selected)

if not selected:
    raise ValueError("没有需要运行的 batch。可能全部完成，或 START_BATCH 设置过大。")

run_records = []

for batch_id in selected:
    record = run_one_batch(batch_id)
    run_records.append(record)

df_run = pd.DataFrame(run_records)

run_summary = BASE / f"consensus_holo_pocket/primary_organic/autorun_summary_start{START_BATCH}_n{MAX_BATCHES_THIS_SESSION}.xlsx"
run_summary.parent.mkdir(parents=True, exist_ok=True)
df_run.to_excel(run_summary, index=False)

print("\n✅ 本轮 batch 自动运行完成")
print(run_summary)

display(df_run)

总 batch 数: 19
已完成: 15 ['batch_001', 'batch_002', 'batch_003', 'batch_004', 'batch_005', 'batch_006', 'batch_007', 'batch_008', 'batch_009', 'batch_010', 'batch_011', 'batch_012', 'batch_013', 'batch_014', 'batch_015']
未完成: 4 ['batch_016', 'batch_017', 'batch_018', 'batch_019']
本轮将运行: ['batch_016', 'batch_017']

🚀 Running batch_016
  YAML 数量: 20
  Running Boltz...
  ✅ Boltz 完成: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_016
  ✅ confidence summary: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_016_confidence_summary.xlsx
  PDB: 20, confidence JSON: 20
  ✅ ligand check: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_016_ligand_check.xlsx
Has_Protein  Has_Ligand
True         True          20
Name: count, dtype: int64
  ✅ ligand-centered pocket: /content/drive/MyDrive/Horizyn_Checkpoints/consensus_holo_pocket/primary_organic/batch_016/batch_016_ligand_centered_pocket_residues.xlsx
  ✅ P2Rank

,Batch_ID,Status,Consensus
0,batch_016,completed,/content/drive/MyDrive/Horizyn_Checkpoints/con...
1,batch_017,completed,/content/drive/MyDrive/Horizyn_Checkpoints/con...


In [67]:
# ============================================================
# 合并当前已完成 batch 的 consensus summary 并做 QC
# ============================================================

from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

summary_files = sorted(
    (BASE / "consensus_holo_pocket/primary_organic").rglob(
        "batch_*_consensus_holo_pocket_summary_conservative.xlsx"
    )
)

print("已完成 batch summary 数量:", len(summary_files))

dfs = []

for f in summary_files:
    df = pd.read_excel(f)
    df["Summary_File"] = str(f)
    dfs.append(df)

if len(dfs) == 0:
    raise FileNotFoundError("没有找到任何 batch consensus summary。")

df_all = pd.concat(dfs, ignore_index=True)

out_xlsx = BASE / "consensus_holo_pocket/primary_organic/primary_organic_all_consensus_holo_pocket_summary_current.xlsx"
df_all.to_excel(out_xlsx, index=False)

print("✅ 当前已完成 batch 合并表:")
print(out_xlsx)

print("\nConsensus class 总统计:")
print(df_all["Consensus_Holo_Pocket_Class"].value_counts(dropna=False))

print("\nConservative action 总统计:")
print(df_all["Recommended_Next_Action_Conservative"].value_counts(dropna=False))

print("\n按 batch 统计:")
display(
    df_all.groupby(["Batch_ID", "Consensus_Holo_Pocket_Class"])
    .size()
    .reset_index(name="count")
    .pivot(index="Batch_ID", columns="Consensus_Holo_Pocket_Class", values="count")
    .fillna(0)
)

display(
    df_all[[
        "Batch_ID",
        "PDB_File",
        "confidence_score",
        "P2Rank_Hit_Class",
        "fpocket_Hit_Class",
        "Consensus_Holo_Pocket_Score",
        "Consensus_Holo_Pocket_Class",
        "Recommended_Next_Action_Conservative"
    ]]
    .sort_values(["Batch_ID", "Consensus_Holo_Pocket_Score"], ascending=[True, False])
    .head(40)
)

已完成 batch summary 数量: 17
✅ 当前已完成 batch 合并表:
/content/drive/MyDrive/Horizyn_Checkpoints/consensus_holo_pocket/primary_organic/primary_organic_all_consensus_holo_pocket_summary_current.xlsx

Consensus class 总统计:
Consensus_Holo_Pocket_Class
High_confidence_holo_pocket    289
Probable_holo_pocket            43
Uncertain_manual_check           7
Low_confidence_or_failed         1
Name: count, dtype: int64

Conservative action 总统计:
Recommended_Next_Action_Conservative
Proceed_to_substrate_docking_and_EZSpecificity    289
Proceed_but_keep_manual_inspection                 42
Manual_inspection_or_rerun_Boltz_multiseed          9
Name: count, dtype: int64

按 batch 统计:


Consensus_Holo_Pocket_Class,High_confidence_holo_pocket,Low_confidence_or_failed,Probable_holo_pocket,Uncertain_manual_check
Batch_ID,,,,
batch_001,14.0,0.0,4.0,2.0
batch_002,16.0,1.0,3.0,0.0
batch_003,15.0,0.0,3.0,2.0
batch_004,16.0,0.0,2.0,2.0
batch_005,18.0,0.0,2.0,0.0
batch_006,19.0,0.0,1.0,0.0
batch_007,18.0,0.0,2.0,0.0
batch_008,17.0,0.0,3.0,0.0
batch_009,18.0,0.0,2.0,0.0


,Batch_ID,PDB_File,confidence_score,P2Rank_Hit_Class,fpocket_Hit_Class,Consensus_Holo_Pocket_Score,Consensus_Holo_Pocket_Class,Recommended_Next_Action_Conservative
0,batch_001,NODE_11_length_154862_cov_66.368492_122__Heme_...,0.891467,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
2,batch_001,NODE_13_length_131311_cov_66.505852_95__FMN__r...,0.878142,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
4,batch_001,NODE_1_length_436095_cov_65.793628_253__FMN__r...,0.948654,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
13,batch_001,NODE_25_length_76722_cov_66.604332_89__NADP___...,0.877610,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
14,batch_001,NODE_26_length_70757_cov_67.122722_30__NAD___r...,0.896448,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
16,batch_001,NODE_32_length_49516_cov_65.241935_22__FMN__ra...,0.960423,Strong_<=6A,Strong_<=6A,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
1,batch_001,NODE_12_length_136290_cov_66.430554_166__Heme_...,0.843068,Strong_<=6A,Strong_<=6A,0.970,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
18,batch_001,NODE_6_length_231008_cov_66.606731_103__FAD__r...,0.622942,Strong_<=6A,Strong_<=6A,0.940,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
9,batch_001,NODE_21_length_97532_cov_65.343769_69__NAD___r...,0.886403,Strong_<=6A,Moderate_6_10A,0.938,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
11,batch_001,NODE_25_length_76722_cov_66.604332_6__NAD___ra...,0.893232,Strong_<=6A,Moderate_6_10A,0.938,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity


In [68]:
# ============================================================
# 续跑 Primary organic/heme batches
# 当前已完成 batch_001–batch_017
# 本轮运行 batch_018–batch_019
# ============================================================

from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

START_BATCH = 18
MAX_BATCHES_THIS_SESSION = 2
FORCE_RERUN = False

BATCH_MANIFEST = BASE / "boltz_inputs/primary_organic_batches_manifest.xlsx"
CONSENSUS_ROOT = BASE / "consensus_holo_pocket/primary_organic"

# 检查 run_one_batch 是否已经定义
if "run_one_batch" not in globals():
    raise RuntimeError(
        "当前 runtime 里没有 run_one_batch()。请先运行前面那个完整 Master Auto-Runner 定义 Cell，"
        "然后再运行这个续跑 Cell。"
    )

df_batches = pd.read_excel(BATCH_MANIFEST)

all_batch_ids = sorted(df_batches["Batch_ID"].dropna().unique().tolist())
all_batch_nums = [(bid, int(str(bid).split("_")[1])) for bid in all_batch_ids]

# 找到未完成 batch
finished = []
for bid in all_batch_ids:
    final_xlsx = CONSENSUS_ROOT / bid / f"{bid}_consensus_holo_pocket_summary_conservative.xlsx"
    if final_xlsx.exists():
        finished.append(bid)

unfinished = [bid for bid in all_batch_ids if bid not in finished]

selected = [
    bid for bid, n in all_batch_nums
    if n >= START_BATCH and bid in unfinished
][:MAX_BATCHES_THIS_SESSION]

print("总 batch 数:", len(all_batch_ids))
print("已完成:", len(finished), finished)
print("未完成:", len(unfinished), unfinished[:20])
print("本轮将运行:", selected)

if not selected:
    raise ValueError("没有需要运行的 batch。可能全部完成，或 START_BATCH 设置过大。")

run_records = []

for batch_id in selected:
    record = run_one_batch(batch_id)
    run_records.append(record)

df_run = pd.DataFrame(run_records)

run_summary = BASE / f"consensus_holo_pocket/primary_organic/autorun_summary_start{START_BATCH}_n{MAX_BATCHES_THIS_SESSION}.xlsx"
run_summary.parent.mkdir(parents=True, exist_ok=True)
df_run.to_excel(run_summary, index=False)

print("\n✅ 本轮 batch 自动运行完成")
print(run_summary)

display(df_run)

总 batch 数: 19
已完成: 17 ['batch_001', 'batch_002', 'batch_003', 'batch_004', 'batch_005', 'batch_006', 'batch_007', 'batch_008', 'batch_009', 'batch_010', 'batch_011', 'batch_012', 'batch_013', 'batch_014', 'batch_015', 'batch_016', 'batch_017']
未完成: 2 ['batch_018', 'batch_019']
本轮将运行: ['batch_018', 'batch_019']

🚀 Running batch_018
  YAML 数量: 20
  Running Boltz...
  ✅ Boltz 完成: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_018
  ✅ confidence summary: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_018_confidence_summary.xlsx
  PDB: 20, confidence JSON: 20
  ✅ ligand check: /content/drive/MyDrive/Horizyn_Checkpoints/boltz_outputs/primary_organic/batch_018_ligand_check.xlsx
Has_Protein  Has_Ligand
True         True          20
Name: count, dtype: int64
  ✅ ligand-centered pocket: /content/drive/MyDrive/Horizyn_Checkpoints/consensus_holo_pocket/primary_organic/batch_018/batch_018_ligand_centered_pocket_residues.xlsx
  ✅ P2Rank

,Batch_ID,Status,Consensus
0,batch_018,completed,/content/drive/MyDrive/Horizyn_Checkpoints/con...
1,batch_019,completed,/content/drive/MyDrive/Horizyn_Checkpoints/con...


In [69]:
# ============================================================
# Final Merge：合并 19 个 batch 的 consensus holo-pocket 结果
# 生成全量主表 + best model per enzyme + docking/EZSpecificity 输入表
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re
import shutil

BASE = Path("/content/drive/MyDrive/Horizyn_Checkpoints")

consensus_root = BASE / "consensus_holo_pocket/primary_organic"
batch_manifest_xlsx = BASE / "boltz_inputs/primary_organic_batches_manifest.xlsx"

out_dir = BASE / "final_primary_organic_holo_results"
out_dir.mkdir(parents=True, exist_ok=True)

# -----------------------------
# 1. 收集所有 batch consensus summary
# -----------------------------
summary_files = sorted(
    consensus_root.rglob("batch_*_consensus_holo_pocket_summary_conservative.xlsx")
)

print("发现 batch consensus 文件数:", len(summary_files))

if len(summary_files) == 0:
    raise FileNotFoundError("没有找到 batch consensus summary 文件。")

dfs = []

for f in summary_files:
    df = pd.read_excel(f)

    # 从路径中提取 batch_id
    m = re.search(r"(batch_\d+)", str(f))
    batch_id = m.group(1) if m else "unknown"

    df["Batch_ID_from_file"] = batch_id
    df["Summary_File"] = str(f)

    if "Batch_ID" not in df.columns:
        df["Batch_ID"] = batch_id

    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)

print("合并后总任务数:", len(df_all))
print("Batch 数:", df_all["Batch_ID"].nunique())

# -----------------------------
# 2. 读取 batch manifest，补充 Enzyme_ID / Cofactor / EC / CCD
# -----------------------------
df_manifest = pd.read_excel(batch_manifest_xlsx)

def norm_file(x):
    x = str(x).split("/")[-1]
    x = x.replace(".pdb", "")
    x = x.replace(".cif", "")
    x = x.replace(".json", "")
    x = x.replace("_model_0", "")
    x = re.sub(r"[^A-Za-z0-9_.-]+", "_", x)
    return x

def yaml_stem(x):
    try:
        return Path(str(x)).stem
    except:
        return ""

df_all["PDB_Stem_Norm"] = df_all["PDB_File"].apply(norm_file)

if "YAML_Path" in df_manifest.columns:
    df_manifest["YAML_Stem_Norm"] = df_manifest["YAML_Path"].apply(yaml_stem).apply(norm_file)
else:
    raise ValueError("batch manifest 中没有 YAML_Path 列，无法准确匹配任务。")

# 用 Batch_ID + YAML stem 匹配
df_merged = df_all.merge(
    df_manifest,
    left_on=["Batch_ID", "PDB_Stem_Norm"],
    right_on=["Batch_ID", "YAML_Stem_Norm"],
    how="left",
    suffixes=("", "_manifest")
)

# -----------------------------
# 3. 检查未匹配项
# -----------------------------
if "Task_ID" in df_merged.columns:
    unmatched = df_merged[df_merged["Task_ID"].isna()].copy()
else:
    unmatched = df_merged[df_merged["YAML_Stem_Norm"].isna()].copy()

print("\nManifest 匹配失败数量:", len(unmatched))

# 如果有少量未匹配，做一个兜底 contains 匹配
if len(unmatched) > 0:
    print("尝试兜底 fuzzy matching...")

    # 建立 manifest 索引
    manifest_records = df_manifest.to_dict("records")

    fill_cols = [c for c in df_manifest.columns if c not in df_merged.columns or c.endswith("_manifest") == False]

    for idx, row in df_merged[df_merged.get("Task_ID").isna() if "Task_ID" in df_merged.columns else df_merged["YAML_Stem_Norm"].isna()].iterrows():
        pdb_norm = row["PDB_Stem_Norm"]
        batch_id = row["Batch_ID"]

        candidates = []
        for r in manifest_records:
            if str(r.get("Batch_ID", "")) != str(batch_id):
                continue

            ystem = norm_file(yaml_stem(r.get("YAML_Path", "")))

            if pdb_norm in ystem or ystem in pdb_norm:
                candidates.append(r)

        if len(candidates) == 1:
            r = candidates[0]
            for c, v in r.items():
                if c not in df_merged.columns or pd.isna(df_merged.at[idx, c]):
                    df_merged.at[idx, c] = v

# 重新检查
if "Task_ID" in df_merged.columns:
    unmatched = df_merged[df_merged["Task_ID"].isna()].copy()
else:
    unmatched = pd.DataFrame()

print("最终 Manifest 匹配失败数量:", len(unmatched))

# -----------------------------
# 4. 统一 action 列
# -----------------------------
if "Recommended_Next_Action_Conservative" not in df_merged.columns:
    if "Recommended_Next_Action" in df_merged.columns:
        df_merged["Recommended_Next_Action_Conservative"] = df_merged["Recommended_Next_Action"]
    else:
        df_merged["Recommended_Next_Action_Conservative"] = "Unknown"

# -----------------------------
# 5. 生成 QC 表
# -----------------------------
batch_qc = (
    df_merged
    .groupby("Batch_ID")
    .agg(
        n_tasks=("PDB_File", "count"),
        n_high=("Consensus_Holo_Pocket_Class", lambda x: (x == "High_confidence_holo_pocket").sum()),
        n_probable=("Consensus_Holo_Pocket_Class", lambda x: (x == "Probable_holo_pocket").sum()),
        n_uncertain=("Consensus_Holo_Pocket_Class", lambda x: (x == "Uncertain_manual_check").sum()),
        n_low=("Consensus_Holo_Pocket_Class", lambda x: (x == "Low_confidence_or_failed").sum()),
        mean_consensus=("Consensus_Holo_Pocket_Score", "mean"),
        median_consensus=("Consensus_Holo_Pocket_Score", "median"),
        mean_boltz_conf=("confidence_score", "mean"),
    )
    .reset_index()
)

batch_qc["pass_rate_high_probable"] = (
    (batch_qc["n_high"] + batch_qc["n_probable"]) / batch_qc["n_tasks"]
).round(3)

# 按辅因子统计
if "Cofactor" in df_merged.columns:
    cofactor_qc = (
        df_merged
        .groupby("Cofactor")
        .agg(
            n_tasks=("PDB_File", "count"),
            n_high=("Consensus_Holo_Pocket_Class", lambda x: (x == "High_confidence_holo_pocket").sum()),
            n_probable=("Consensus_Holo_Pocket_Class", lambda x: (x == "Probable_holo_pocket").sum()),
            n_uncertain=("Consensus_Holo_Pocket_Class", lambda x: (x == "Uncertain_manual_check").sum()),
            mean_consensus=("Consensus_Holo_Pocket_Score", "mean"),
            median_consensus=("Consensus_Holo_Pocket_Score", "median"),
        )
        .reset_index()
        .sort_values("n_tasks", ascending=False)
    )
else:
    cofactor_qc = pd.DataFrame()

# -----------------------------
# 6. 生成后续分析队列表
# -----------------------------
df_ready_primary = df_merged[
    df_merged["Recommended_Next_Action_Conservative"].eq(
        "Proceed_to_substrate_docking_and_EZSpecificity"
    )
].copy()

df_ready_secondary = df_merged[
    df_merged["Recommended_Next_Action_Conservative"].eq(
        "Proceed_but_keep_manual_inspection"
    )
].copy()

df_manual = df_merged[
    df_merged["Recommended_Next_Action_Conservative"].str.contains(
        "Manual_inspection", na=False
    )
].copy()

df_hold = df_merged[
    df_merged["Recommended_Next_Action_Conservative"].str.contains(
        "Hold|alternative", na=False
    )
].copy()

print("\n后续分析队列:")
print("Primary docking/EZSpecificity:", len(df_ready_primary))
print("Secondary with manual inspection:", len(df_ready_secondary))
print("Manual / multiseed rerun:", len(df_manual))
print("Hold:", len(df_hold))

# -----------------------------
# 7. 每个酶选择最佳 holo model
# -----------------------------
if "Enzyme_ID" in df_merged.columns:
    sort_cols = [
        "Consensus_Holo_Pocket_Score",
        "confidence_score",
    ]

    df_best = (
        df_merged
        .sort_values(sort_cols, ascending=[False, False])
        .groupby("Enzyme_ID", as_index=False)
        .head(1)
        .copy()
    )

    # 每个酶保留所有可选 cofactor 的记录，便于 NAD/NADP 或 FAD/FMN 对比
    df_all_options = df_merged.sort_values(
        ["Enzyme_ID", "Consensus_Holo_Pocket_Score"],
        ascending=[True, False]
    ).copy()

else:
    df_best = pd.DataFrame()
    df_all_options = df_merged.copy()

# -----------------------------
# 8. 生成 docking/EZSpecificity 输入表
# -----------------------------
# 主队列：High-confidence
dock_cols = [
    "Batch_ID",
    "Task_ID",
    "Enzyme_ID",
    "Original_ID",
    "CleanContact_Entry",
    "Predicted_EC_4digit",
    "Cofactor",
    "Cofactor_Score",
    "Cofactor_Type",
    "Model_Token_or_CCD",
    "PDB_File",
    "PDB_Path",
    "Best_Model_PDB",
    "confidence_score",
    "ligand_iptm",
    "complex_plddt",
    "N_Residues_5A",
    "N_Residues_8A",
    "Pocket_Residues_5A",
    "Pocket_Residues_8A",
    "P2Rank_Hit_Class",
    "Nearest_P2Rank_Distance_A",
    "fpocket_Hit_Class",
    "Nearest_fpocket_Distance_A",
    "Consensus_Holo_Pocket_Score",
    "Consensus_Holo_Pocket_Class",
    "Recommended_Next_Action_Conservative",
]

dock_cols = [c for c in dock_cols if c in df_merged.columns]

df_docking_input_primary = df_ready_primary[dock_cols].copy()
df_docking_input_secondary = df_ready_secondary[dock_cols].copy()

# -----------------------------
# 9. 输出 Excel
# -----------------------------
final_xlsx = out_dir / "primary_organic_all_consensus_holo_pocket_FINAL.xlsx"
final_csv = out_dir / "primary_organic_all_consensus_holo_pocket_FINAL.csv"

with pd.ExcelWriter(final_xlsx, engine="openpyxl") as writer:
    df_merged.to_excel(writer, index=False, sheet_name="all_holo_tasks")
    batch_qc.to_excel(writer, index=False, sheet_name="batch_qc")
    cofactor_qc.to_excel(writer, index=False, sheet_name="cofactor_qc")
    df_ready_primary.to_excel(writer, index=False, sheet_name="primary_ready")
    df_ready_secondary.to_excel(writer, index=False, sheet_name="secondary_review")
    df_manual.to_excel(writer, index=False, sheet_name="manual_or_multiseed")
    df_hold.to_excel(writer, index=False, sheet_name="hold")
    df_best.to_excel(writer, index=False, sheet_name="best_model_per_enzyme")
    df_all_options.to_excel(writer, index=False, sheet_name="all_options_by_enzyme")
    df_docking_input_primary.to_excel(writer, index=False, sheet_name="docking_input_primary")
    df_docking_input_secondary.to_excel(writer, index=False, sheet_name="docking_input_secondary")
    unmatched.to_excel(writer, index=False, sheet_name="unmatched_manifest")

df_merged.to_csv(final_csv, index=False)

print("\n✅ 最终全量结果已生成:")
print(final_xlsx)
print(final_csv)

print("\nConsensus class 总统计:")
print(df_merged["Consensus_Holo_Pocket_Class"].value_counts(dropna=False))

print("\nConservative action 总统计:")
print(df_merged["Recommended_Next_Action_Conservative"].value_counts(dropna=False))

print("\nBatch QC:")
display(batch_qc)

print("\nPrimary docking/EZSpecificity 输入预览:")
display(df_docking_input_primary.head(20))

发现 batch consensus 文件数: 19
合并后总任务数: 371
Batch 数: 19

Manifest 匹配失败数量: 0
最终 Manifest 匹配失败数量: 0

后续分析队列:
Primary docking/EZSpecificity: 313
Secondary with manual inspection: 46
Manual / multiseed rerun: 12
Hold: 0

✅ 最终全量结果已生成:
/content/drive/MyDrive/Horizyn_Checkpoints/final_primary_organic_holo_results/primary_organic_all_consensus_holo_pocket_FINAL.xlsx
/content/drive/MyDrive/Horizyn_Checkpoints/final_primary_organic_holo_results/primary_organic_all_consensus_holo_pocket_FINAL.csv

Consensus class 总统计:
Consensus_Holo_Pocket_Class
High_confidence_holo_pocket    313
Probable_holo_pocket            48
Uncertain_manual_check           9
Low_confidence_or_failed         1
Name: count, dtype: int64

Conservative action 总统计:
Recommended_Next_Action_Conservative
Proceed_to_substrate_docking_and_EZSpecificity    313
Proceed_but_keep_manual_inspection                 46
Manual_inspection_or_rerun_Boltz_multiseed         12
Name: count, dtype: int64

Batch QC:


,Batch_ID,n_tasks,n_high,n_probable,n_uncertain,n_low,mean_consensus,median_consensus,mean_boltz_conf,pass_rate_high_probable
0,batch_001,20,14,4,2,0,0.822850,0.9065,0.829002,0.900
1,batch_002,20,16,3,0,1,0.856350,0.9380,0.862221,0.950
2,batch_003,20,15,3,2,0,0.830350,0.8750,0.833909,0.900
3,batch_004,20,16,2,2,0,0.849150,0.9065,0.880062,0.900
4,batch_005,20,18,2,0,0,0.903550,0.9380,0.881471,1.000
5,batch_006,20,19,1,0,0,0.934900,0.9700,0.878633,1.000
6,batch_007,20,18,2,0,0,0.913400,0.9380,0.930596,1.000
7,batch_008,20,17,3,0,0,0.889400,0.9380,0.879535,1.000
8,batch_009,20,18,2,0,0,0.913450,0.9380,0.886357,1.000
9,batch_010,20,20,0,0,0,0.932450,0.9380,0.901602,1.000



Primary docking/EZSpecificity 输入预览:


,Batch_ID,Task_ID,Enzyme_ID,Predicted_EC_4digit,Cofactor,Cofactor_Score,Cofactor_Type,PDB_File,PDB_Path,Best_Model_PDB,...,N_Residues_8A,Pocket_Residues_5A,Pocket_Residues_8A,P2Rank_Hit_Class,Nearest_P2Rank_Distance_A,fpocket_Hit_Class,Nearest_fpocket_Distance_A,Consensus_Holo_Pocket_Score,Consensus_Holo_Pocket_Class,Recommended_Next_Action_Conservative
0,batch_001,NODE_11_length_154862_cov_66.368492_122__Heme_...,NODE_11_length_154862_cov_66.368492_122,1.14.15.35,Heme,0.90,heme,NODE_11_length_154862_cov_66.368492_122__Heme_...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,...,70,A:ALA234;A:ALA350;A:ARG286;A:ARG94;A:CYS344;A:...,A:ALA234;A:ALA347;A:ALA350;A:ARG286;A:ARG339;A...,Strong_<=6A,1.446,Strong_<=6A,1.262,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
1,batch_001,NODE_12_length_136290_cov_66.430554_166__Heme_...,NODE_12_length_136290_cov_66.430554_166,1.14.14.18,Heme,0.90,heme,NODE_12_length_136290_cov_66.430554_166__Heme_...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,...,68,A:ARG69;A:ASP72;A:GLN61;A:GLU110;A:GLY277;A:HI...,A:ALA103;A:ALA285;A:ALA74;A:ALA93;A:ARG112;A:A...,Strong_<=6A,0.685,Strong_<=6A,0.678,0.970,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
2,batch_001,NODE_13_length_131311_cov_66.505852_95__FMN__r...,NODE_13_length_131311_cov_66.505852_95,1.4.3.5;1.5.1.38,FMN,0.90,organic_cofactor,NODE_13_length_131311_cov_66.505852_95__FMN__r...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,...,49,A:ALA135;A:ALA84;A:ARG148;A:ARG52;A:ARG95;A:GL...,A:ALA135;A:ALA78;A:ALA84;A:ALA91;A:ALA99;A:ARG...,Strong_<=6A,1.695,Strong_<=6A,1.052,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
4,batch_001,NODE_1_length_436095_cov_65.793628_253__FMN__r...,NODE_1_length_436095_cov_65.793628_253,1.14.14.3,FMN,0.90,organic_cofactor,NODE_1_length_436095_cov_65.793628_253__FMN__r...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,...,58,A:ALA170;A:ALA80;A:ARG173;A:GLN182;A:GLN184;A:...,A:ALA169;A:ALA170;A:ALA171;A:ALA211;A:ALA250;A...,Strong_<=6A,5.024,Strong_<=6A,5.923,1.000,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
6,batch_001,NODE_1_length_436095_cov_65.793628_305__FMN__r...,NODE_1_length_436095_cov_65.793628_305,1.4.3.5,FMN,0.90,organic_cofactor,NODE_1_length_436095_cov_65.793628_305__FMN__r...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,...,37,A:ALA134;A:ALA248;A:ALA250;A:ARG169;A:GLY123;A...,A:ALA132;A:ALA134;A:ALA174;A:ALA248;A:ALA250;A...,Weak_10_15A,10.449,Moderate_6_10A,9.300,0.788,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
9,batch_001,NODE_21_length_97532_cov_65.343769_69__NAD+__r...,NODE_21_length_97532_cov_65.343769_69,1.5.1.34,NAD+,0.90,organic_cofactor,NODE_21_length_97532_cov_65.343769_69__NAD___r...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,...,68,A:ALA103;A:ALA150;A:ALA159;A:ALA183;A:ALA49;A:...,A:ALA103;A:ALA113;A:ALA117;A:ALA150;A:ALA159;A...,Strong_<=6A,2.109,Moderate_6_10A,6.331,0.938,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
11,batch_001,NODE_25_length_76722_cov_66.604332_6__NAD+__rank1,NODE_25_length_76722_cov_66.604332_6,1.5.1.34,NAD+,0.90,organic_cofactor,NODE_25_length_76722_cov_66.604332_6__NAD___ra...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,/content/drive/MyDrive/Horizyn_Checkpoints/bol...,...,41,A:ARG156;A:ARG162;A:ASN61;A:ASP34;A:CYS32;A:GL...,A:ALA225;A:ALA51;A:ARG113;A:ARG156;A:ARG162;A:...,Strong_<=6A,5.364,Moderate_6_10A,6.594,0.938,High_confidence_holo_pocket,Proceed_to_substrate_docking_and_EZSpecificity
12,batch_001,NODE_25_length_76722_cov_66.604332_89__FAD__rank1,NODE_25_l